# Fast-screening low-quality STL generator

This version is optimized for quick candidate detection before final high-quality STL export.

Key settings:
- 1000 candidates retained.
- Low voxel/marching-cubes resolution for fast screening.
- CPU-only serial execution by default to avoid Windows/Jupyter hanging.
- Progress is printed after every candidate and saved frequently.
- All user-editable structural parameters are centralized in Cell 1.
- Lattice and Voxel are forced to satisfy the requested face-contact rule and become one connected solid component; TPMS is intentionally excluded from Isotropic/Orthotropic face constraints.

After screening works, increase `tpms_grid_n`, `voxel_grid_n`, `lattice_grid_n`, and reduce `marching_cubes_step_size` to produce final-quality STL files.

Integrated update: full structural descriptor extraction has been added in Cell 8, including point/surface/slice/lattice descriptors with CPU/GPU-aware execution.


In [ ]:
# ============================================================
# Cell 1. Imports + ONE-CELL global configuration (harmonized for v4)
# ============================================================

import os
import json
import math
import struct
import random
import warnings
from pathlib import Path
from datetime import datetime
from itertools import combinations

from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/CMSL_results')  # 이 폴더는 미리 Drive에 만들어두세요

import numpy as np
import pandas as pd
import networkx as nx

try:
    import trimesh
    TRIMESH_GEN_AVAILABLE = True
except Exception:
    trimesh = None
    TRIMESH_GEN_AVAILABLE = False

from scipy.spatial.distance import pdist, squareform
from scipy.ndimage import (
    gaussian_filter,
    binary_closing,
    binary_opening,
    binary_dilation,
    label,
    distance_transform_edt,
)
from skimage import measure
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ============================================================
# User-editable structure / batch parameters
# ============================================================
# Change values here first. Later cells read from this single dictionary.
#
# Main role of this block:
# - control candidate generation count and mode
# - control CPU/GPU / worker settings
# - control STL quality / voxel resolution
# - control structure-family-specific ranges
#
# Candidate counts are kept as originally defined in each family/mode.
# Preview mode is effectively disabled by setting run_generation_mode="all".
# preview_n is kept only for compatibility with downstream cells.

USER_STRUCTURE_CONFIG = {
    # ========================================================
    # FAST SCREENING MODE
    # ========================================================
    # True  = prioritize fast preview / debugging / connectivity checking
    # False = prioritize higher-quality geometry and closer-to-final STL
    "fast_screening_mode": False,

    # --------------------------------------------------------
    # Output and run mode
    # --------------------------------------------------------
    "output_prefix": "Result_Batch_FastScreening_LowQualitySTL",  # Prefix of output folder name
    "run_generation_mode": "all",    # "all" = generate all candidates
                                      # "preview" = generate only first preview_n candidates
                                      # "none" = build candidate table only, no geometry generation
    "preview_n": 20,                   # Compatibility placeholder; ignored when run_generation_mode="all"
    "auto_run_generation": True,      # If True, generation starts automatically in later cells
    "random_seed": 42,                # Global random seed for reproducible candidate generation
    "save_every_n": 5,               # Save progress/log backup every N generated candidates
    "resume_mode": False,              # If True, try to continue from previous progress/log files
    "overwrite_existing": True,      # If False, skip already-generated files when possible

    # --------------------------------------------------------
    # CPU/GPU acceleration / stability
    # --------------------------------------------------------
    # Current notebook generation logic supports selecting one GPU device.
    # Descriptor extraction is still mainly CPU-safe unless later cells explicitly enable GPU.
    "compute_backend": "cpu",         # "cpu", "gpu", or "auto"
    "gpu_device_id": 0,               # Fallback GPU index used when compute_backend uses GPU
    "gpu_device_ids": [0],          # GPU device list used for TPMS multi-GPU dispatch when available
    "cpu_workers": "auto",                 # Number of CPU workers; 1 is safest for Jupyter/Windows
    "max_cpu_workers": 4,             # Upper bound when cpu_workers="auto"
    "parallel_backend": "thread",     # "serial" or "thread"; serial is safest, thread is faster
    "progress_print_every_n": 5,      # Print progress every N candidates
    "gpu_workers": 2,                 # Number of concurrent GPU worker lanes across available GPU devices
    "prefer_gpu_for_tpms_voxel_fields": True,  # If True, prefer GPU for TPMS voxel-field generation when available

    # --------------------------------------------------------
    # Common geometric targets
    # --------------------------------------------------------
    "size_mm": 30.0,                   # Outer specimen size in mm
    "target_vf": 0.30,                # Target volume fraction (solid fraction)
    "vf_tolerance": 0.050,            # Allowed deviation from target_vf after symmetry/connectivity repair
    "stl_mesh_size_mm": 0.08,         # STL mesh-size hint in mm; actual quality also depends on voxel grid
    "export_stl": True,               # Save STL files for generated models
    "export_stp": False,              # Save STP/STEP files if supported
    "export_stp_for_selected_only": True,  # Save STP only for selected candidates if STP export is enabled

    # --------------------------------------------------------
    # STL extraction / marching cubes
    # --------------------------------------------------------
    # Smaller step = better STL quality but slower
    # Larger step  = rougher STL but faster
    "marching_cubes_step_size": 1,    # 1 = finest, 2 = faster/rougher, 3 = very rough

    # --------------------------------------------------------
    # Boundary/contact-face symmetry control
    # --------------------------------------------------------
    "enforce_contact_face_symmetry": True,   # Enforce symmetric contact pattern on the six outer faces
    "strict_global_symmetry": True,          # Enforce full 3D symmetry, not only surface-contact symmetry
    "strict_symmetry_method": "score_threshold",  # Method for deciding whether symmetry condition is satisfied
    "contact_surface_depth_vox": 3,          # Number of voxels inward from each boundary face used for symmetry

    # --------------------------------------------------------
    # Lattice/Voxel connectivity repair control
    # --------------------------------------------------------
    # TPMS is excluded from this repair logic.
    # This is used to avoid disconnected floating solid islands.
    "force_connected_lattice_voxel": True,   # If True, force connected solid for lattice/voxel structures
    "connectivity_repair_mode": "bridge",    # "bridge" = connect components with bridges
                                             # "largest" = keep only largest component
    "connectivity_bridge_radius_vox": 2,     # Radius of voxel bridge used when repair_mode="bridge"
    "connectivity_min_component_voxels": 1,  # Components smaller than this size are removed as tiny fragments
    "connectivity_max_bridges": 200,         # Safety cap on the number of added bridges
    "connectivity_retry_after_contact_symmetry": True,  # Re-check connectivity after symmetry enforcement

    # --------------------------------------------------------
    # FAST voxel / marching-cubes resolutions
    # --------------------------------------------------------
    # These are generation-side voxel/grid resolutions, not descriptor-side resolutions.
    "tpms_grid_n": 150,                # Grid resolution per axis for TPMS generation
    "voxel_grid_n": 280,               # Grid resolution per axis for voxel-structure generation
    "lattice_grid_n": 280,             # Grid resolution per axis for lattice rasterization / voxelization

    # --------------------------------------------------------
    # Lattice structure parameters
    # --------------------------------------------------------
    "lattice": {
        "node_blend_factor": 1.00,    # Controls node blending / thickening around lattice joints
        "binary_closing_iters": 0,    # Morphological closing iterations on lattice voxel mask
        "radius_search_iters": 6,     # Number of search/refinement iterations for radius-based fitting
        "cylinder_segments": 20,       # Number of angular segments for cylindrical strut approximation
        "enforce_contact_face_symmetry": True,  # Lattice-specific face symmetry enforcement
        "contact_surface_depth_vox": 3,         # Lattice-specific contact depth in voxels
        "modes": {
            "periodic_isotropic": {
                "n_candidates": 0,              # Number of lattice candidates in this mode
                "node_count_range": [10, 24],    # Min/max lattice node count
                "strut_count_range": [18, 56],   # Min/max strut count
                "min_thickness_mm_range": [0.25, 0.55],  # Min/max strut thickness range in mm
                "connectivity_k_range": [3, 6],  # Neighbor count / graph connectivity control
                "vertical_bias_range": [0.25, 0.75],     # Preference for vertical struts
                "diagonal_bias_range": [0.20, 0.80],     # Preference for diagonal struts
                "node_perturbation_range": [0.00, 0.08], # Random node perturbation magnitude
            },
            "periodic_orthotropic": {
                "n_candidates": 0,              # Number of lattice candidates in this mode
                "node_count_range": [10, 24],    # Min/max lattice node count
                "strut_count_range": [18, 56],   # Min/max strut count
                "min_thickness_mm_range": [0.25, 0.55],  # Min/max strut thickness range in mm
                "connectivity_k_range": [3, 6],  # Neighbor count / graph connectivity control
                "vertical_bias_range": [0.45, 0.90],     # Stronger vertical preference for orthotropic structure
                "diagonal_bias_range": [0.10, 0.60],     # Diagonal-strut preference range
                "node_perturbation_range": [0.00, 0.06], # Random node perturbation magnitude
                "orthotropic_z_scale_range": [0.8, 1.5], # Z-direction scaling for orthotropic behavior
            },
            "stochastic": {
                "n_candidates": 0,              # Number of lattice candidates in this mode
                "node_count_range": [12, 30],    # Min/max lattice node count
                "strut_count_range": [20, 70],   # Min/max strut count
                "min_thickness_mm_range": [0.22, 0.55],  # Min/max strut thickness range in mm
                "connectivity_k_range": [3, 7],  # Neighbor count / graph connectivity control
                "vertical_bias_range": [0.10, 0.90],     # Vertical-strut preference range
                "diagonal_bias_range": [0.10, 0.90],     # Diagonal-strut preference range
                "node_perturbation_range": [0.00, 0.20], # Random node perturbation magnitude
            },
        },
    },

    # --------------------------------------------------------
    # TPMS structure parameters
    # --------------------------------------------------------
    "tpms": {
        "library": [
            "gyroid", "primitive", "diamond", "iwp", "neovius", "lidinoid",
            "split_p", "double_gyroid", "frd", "fischer_koch_s",
            "fischer_koch_c", "pw_hybrid", "srs_like", "karcher_k_like",
        ],  # Available TPMS function families
        "basic_modes": ["wall", "solid_A", "solid_B"],  # Basic TPMS topology modes / level-set interpretations
        "basic": {
            "n_candidates_per_tpms_mode": 0,    # Number of candidates per TPMS family × basic mode
            "offset_range": [0.04, 0.20],       # Level-set offset range controlling solid/void split
            "tpms_thickness_mm_range": [0.8, 2.5],  # Shell / wall thickness range in mm
            "min_hole_size_vox_range": [1, 2],  # Minimum pore/hole size target in voxels
            "frequency_range": [5.0, 5.0],      # TPMS spatial frequency range
            "anisotropy_xyz_range": [0.90, 1.20],  # XYZ anisotropy scaling range
            "noise_amp_range": [0.0, 0.020],    # Random perturbation amplitude added to TPMS field
        },
        "adv2": {
            "n_candidates": 22,                 # Number of advanced 2-component TPMS candidates
            "n_components": 2,                  # Number of mixed TPMS components
            "offset_range": [0.03, 0.18],       # Level-set offset range
            "tpms_thickness_mm_range": [0.8, 2.5],  # Shell / wall thickness range in mm
            "min_hole_size_vox_range": [1, 2],  # Minimum pore/hole size target in voxels
            "component_gap_mm_range": [0.05, 0.35],   # Gap between mixed TPMS components in mm
            "frequency_range": [5.0, 5.0],      # TPMS spatial frequency range
            "anisotropy_xyz_range": [0.90, 1.20],  # XYZ anisotropy scaling range
            "noise_amp_range": [0.0, 0.015],    # Random perturbation amplitude added to TPMS field
        },
        "adv3": {
            "n_candidates": 21,                 # Number of advanced 3-component TPMS candidates
            "n_components": 3,                  # Number of mixed TPMS components
            "offset_range": [0.02, 0.16],       # Level-set offset range
            "tpms_thickness_mm_range": [0.8, 2.5],  # Shell / wall thickness range in mm
            "min_hole_size_vox_range": [1, 2],  # Minimum pore/hole size target in voxels
            "component_gap_mm_range": [0.05, 0.30],   # Gap between mixed TPMS components in mm
            "frequency_range": [5.0, 5.0],      # TPMS spatial frequency range
            "anisotropy_xyz_range": [0.90, 1.20],  # XYZ anisotropy scaling range
            "noise_amp_range": [0.0, 0.010],    # Random perturbation amplitude added to TPMS field
        },
        "adv4": {
            "n_candidates": 21,                 # Number of advanced 4-component TPMS candidates
            "n_components": 4,                  # Number of mixed TPMS components
            "offset_range": [0.02, 0.14],       # Level-set offset range
            "tpms_thickness_mm_range": [0.8, 2.5],  # Shell / wall thickness range in mm
            "min_hole_size_vox_range": [1, 2],  # Minimum pore/hole size target in voxels
            "component_gap_mm_range": [0.03, 0.25],   # Gap between mixed TPMS components in mm
            "frequency_range": [5.0, 5.0],      # TPMS spatial frequency range
            "anisotropy_xyz_range": [0.90, 1.20],  # XYZ anisotropy scaling range
            "noise_amp_range": [0.0, 0.010],    # Random perturbation amplitude added to TPMS field
        },
    },

    # --------------------------------------------------------
    # Voxel morphogenesis structure parameters
    # --------------------------------------------------------
    "voxel": {
        "modes": {
            "periodic_isotropic": {
                "n_candidates": 0,                # Number of voxel candidates in this mode
                "min_thickness_vox_range": [1, 2], # Minimum solid thickness range in voxels
                "min_hole_size_vox_range": [1, 2], # Minimum pore/hole size range in voxels
                "max_thickness_vox_range": [5, 12],# Maximum solid thickness range in voxels
                "sigma_xyz_range": [1.5, 5.0],     # Smoothing / field sigma range
                "num_fourier_terms_range": [4, 14],# Number of Fourier terms used for field generation
                "closing_iter_range": [0, 1],      # Morphological closing iteration range
                "opening_iter_range": [0, 1],      # Morphological opening iteration range
                "anisotropy_z_range": [0.9, 1.1],  # Z-direction anisotropy range
            },
            "periodic_orthotropic": {
                "n_candidates": 0,                # Number of voxel candidates in this mode
                "min_thickness_vox_range": [1, 2], # Minimum solid thickness range in voxels
                "min_hole_size_vox_range": [1, 2], # Minimum pore/hole size range in voxels
                "max_thickness_vox_range": [5, 12],# Maximum solid thickness range in voxels
                "sigma_xyz_range": [1.5, 5.0],     # Smoothing / field sigma range
                "num_fourier_terms_range": [4, 14],# Number of Fourier terms used for field generation
                "closing_iter_range": [0, 1],      # Morphological closing iteration range
                "opening_iter_range": [0, 1],      # Morphological opening iteration range
                "anisotropy_z_range": [1.2, 2.0],  # Stronger Z-direction anisotropy range
            },
            "stochastic": {
                "n_candidates": 0,                # Number of voxel candidates in this mode
                "min_thickness_vox_range": [1, 2], # Minimum solid thickness range in voxels
                "min_hole_size_vox_range": [1, 2], # Minimum pore/hole size range in voxels
                "max_thickness_vox_range": [5, 14],# Maximum solid thickness range in voxels
                "sigma_xyz_range": [1.5, 5.5],     # Smoothing / field sigma range
                "closing_iter_range": [0, 1],      # Morphological closing iteration range
                "opening_iter_range": [0, 1],      # Morphological opening iteration range
                "anisotropy_z_range": [0.7, 1.8],  # Broad Z-direction anisotropy range
            },
        },
    },
}

# ============================================================
# User-editable structural descriptor extraction parameters
# ============================================================
# This block controls descriptor extraction only.
# Later descriptor cells should read from here and merge with internal defaults.
#
# Descriptor categories:
# - point-based descriptors
# - surface-based descriptors
# - slice-image-based descriptors
# - lattice-only descriptors
# - voxel topology / run-length descriptors
# - graph / skeleton descriptors
#
# Important interpretation:
# - Larger sampling counts / grid sizes = more accurate but slower and heavier
# - Smaller values = faster preview / debugging mode

USER_DESCRIPTOR_CONFIG = {
    # --------------------------------------------------------
    # General execution / output
    # --------------------------------------------------------
    "descriptor_backend": "cpu",   # Backend used during descriptor extraction; current notebook is safest on CPU
    "cpu_workers": max(1, min(16, os.cpu_count() or 1)),  # Number of CPU workers for descriptor extraction
    "parallel_backend": "thread",  # Descriptor parallel mode: "serial" or "thread"
    "candidate_table_name": "candidate_table.csv",   # File name of candidate parameter table used as descriptor input
    "progress_table_name": "progress_log.csv",       # File name of generation/progress log used as descriptor input
    "descriptor_output_folder": "04_structural_descriptors_split",  # Folder where descriptor tables are saved
    "write_csv": True,              # Save descriptor result tables in CSV format
    "write_excel": True,            # Save descriptor result tables in Excel format
    "write_merged_table": True,     # Save merged table combining metadata + descriptor results
    "save_point_cloud_csv": False,  # Save sampled point clouds used for point descriptors as CSV
    "save_curvature_raw_csv": False,# Save raw per-vertex curvature values before statistical aggregation
    "save_slice_merge_images": False,   # Save merged N/N+1 grayscale slice images used in slice descriptors
    "slice_image_max_save_per_axis": 12,# Maximum number of merged slice images saved per axis when saving is enabled

    # --------------------------------------------------------
    # Point-based structure descriptors
    # --------------------------------------------------------
    "enable_point_descriptors": True,    # Enable point-distribution / mass-distribution style descriptors
    "surface_point_count": 3000,         # Number of points sampled on the surface mesh; larger = denser surface statistics
    "surface_point_oversample_factor": 4,# Oversampling factor before filtering/rebalancing surface points; helps coverage uniformity
    "interior_grid_n": 32,               # Coarse grid resolution per axis used to find/sample interior points
    "interior_point_max": 8000,          # Maximum number of interior points retained for descriptor calculation
    "interior_sampling_method": "auto",  # Interior sampling strategy: "auto", "contains", "voxel", or "inward_proxy"
    "interior_inward_offset_fraction": 0.018,  # Inward offset distance as fraction of bbox size for proxy interior points
    "interior_inward_offset_steps": 3,   # Number of inward offset layers used when creating proxy interior points
    "interior_require_exact": False,     # If True, use only exact interior points; if False, allow proxy interior points
    "local_range_abs_norm": 0.10,        # Normalized local-region range used for local point-distribution statistics

    # --------------------------------------------------------
    # Surface-based structure descriptors
    # --------------------------------------------------------
    "enable_surface_descriptors": True,  # Enable surface-area / surface-shape descriptor extraction
    "enable_surface_curvature": True,    # Compute curvature-based descriptors from the mesh surface
    "max_vertices_for_curvature": 80000, # Maximum number of vertices used in curvature computation; limits memory/time
    "curvature_clip_percentile": 99.0,   # Clip extreme curvature outliers above this percentile for robust statistics

    # --------------------------------------------------------
    # Slice-image-based structure descriptors
    # --------------------------------------------------------
    "enable_slice_descriptors": True,    # Enable descriptors from 2D slice-pair images
    "slice_count": 160,                  # Number of slices extracted through each selected axis/direction
    "slice_image_size": 200,             # Pixel resolution of each slice image (width = height)
    "slice_pair_step": 1,                # Slice pairing interval; 1 = N and N+1, 2 = N and N+2
    "slice_axes": ["z", "x", "y"],       # Slice directions used for descriptor extraction
    "min_component_pixels": 2,           # Connected components smaller than this pixel count are treated as noise
    "layer_height_mm": None,             # Physical spacing between adjacent slices; None = auto from size_mm and slice_count

    # --------------------------------------------------------
    # Mesh voxelization used during descriptor extraction
    # --------------------------------------------------------
    "voxel_grid_n": 96,                  # Voxel grid resolution per axis used for descriptor-side voxelization
    "voxelize_pitch_factor": 1.0,        # Multiplier on voxel pitch; >1 coarser voxelization, <1 finer voxelization

    # --------------------------------------------------------
    # Lattice-only structure descriptors
    # --------------------------------------------------------
    "enable_lattice_descriptors": True,  # Enable lattice node/strut/graph descriptors for lattice candidates
    "lattice_coord_key_decimals": 6,     # Decimal rounding used when grouping nearly identical lattice node coordinates
    "lattice_l_over_d_from_radius": True,# If True, compute L/D using radius-derived diameter when radius exists

    # --------------------------------------------------------
    # Additional topology / voxel / graph descriptors
    # --------------------------------------------------------
    "enable_additional_descriptors": True,      # Master switch for extra descriptor families beyond core descriptors
    "enable_voxel_topology_descriptors": True,  # Enable voxel-topology / run-length / connectivity descriptors
    "voxel_topology_grid_n": 96,                # Voxel resolution per axis for topology descriptor extraction
    "run_length_sample_max": 250000,            # Maximum number of sampled run-length / chord-length observations
    "overhang_angle_threshold_deg": 45.0,       # Threshold angle for overhang-related descriptor classification

    # --------------------------------------------------------
    # Advanced graph / topology descriptors
    # --------------------------------------------------------
    "enable_advanced_graph_descriptors": True,  # Enable skeleton/graph-based advanced descriptors
    "advanced_graph_grid_n": 96,                # Voxel resolution per axis for skeleton / graph extraction
    "graph_boundary_margin_vox": 1,             # Boundary margin in voxels when checking spanning paths / boundary contact
    "graph_algebraic_connectivity_max_nodes": 400,  # Max node count allowed for expensive algebraic-connectivity calculation
    "graph_path_axes": ["x", "y", "z"],         # Axes along which path continuity / tortuosity are evaluated
    "local_density_block_sizes": [4, 8],        # Block sizes used for local density heterogeneity descriptors
    "save_skeleton_graph_csv": False,           # Save node/edge tables of skeleton graph if supported

    # --------------------------------------------------------
    # Optional canonical descriptor / physics-proxy settings
    # --------------------------------------------------------
    "modal_proxy_band_count": 8,              # Number of modal bands used for graph/skeleton modal-count proxy
    "modal_proxy_max_modes": 24,              # Maximum number of low-order proxy modes to compute
    "stiffness_proxy_axis_weights": {"x": 1.0, "y": 1.0, "z": 1.0},  # Axis weights used in stiffness/load-path proxy
    "enable_modal_proxy_descriptors": True    # Enable low-order modal-count / modal-density proxy descriptors
}

# ------------------------------------------------------------
# Optional convenience aliases
# ------------------------------------------------------------
TPMS_VOXEL_SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"]) / max(1, (int(USER_STRUCTURE_CONFIG["tpms_grid_n"]) - 1))
VOXEL_STRUCTURE_VOXEL_SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"]) / max(1, (int(USER_STRUCTURE_CONFIG["voxel_grid_n"]) - 1))
LATTICE_VOXEL_SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"]) / max(1, (int(USER_STRUCTURE_CONFIG["lattice_grid_n"]) - 1))
DESCRIPTOR_VOXEL_SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"]) / max(1, (int(USER_DESCRIPTOR_CONFIG["voxel_grid_n"]) - 1))
DESCRIPTOR_TOPOLOGY_VOXEL_SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"]) / max(1, (int(USER_DESCRIPTOR_CONFIG["voxel_topology_grid_n"]) - 1))
DESCRIPTOR_GRAPH_VOXEL_SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"]) / max(1, (int(USER_DESCRIPTOR_CONFIG["advanced_graph_grid_n"]) - 1))

# ============================================================
# Derived variables from USER_STRUCTURE_CONFIG
# ============================================================
RUN_TAG = datetime.now().strftime("%y%m%d_%H%M%S")
OUTPUT_ROOT = Path.cwd() / f"{USER_STRUCTURE_CONFIG['output_prefix']}_{RUN_TAG}"

SIZE_MM = float(USER_STRUCTURE_CONFIG["size_mm"])
TARGET_VF = float(USER_STRUCTURE_CONFIG["target_vf"])
VF_TOLERANCE = float(USER_STRUCTURE_CONFIG["vf_tolerance"])
STL_MESH_SIZE_MM = float(USER_STRUCTURE_CONFIG["stl_mesh_size_mm"])
MARCHING_CUBES_STEP_SIZE = max(1, int(USER_STRUCTURE_CONFIG.get("marching_cubes_step_size", 1)))
FAST_SCREENING_MODE = bool(USER_STRUCTURE_CONFIG.get("fast_screening_mode", False))
TPMS_GRID_N = int(USER_STRUCTURE_CONFIG["tpms_grid_n"])
VOXEL_GRID_N = int(USER_STRUCTURE_CONFIG["voxel_grid_n"])
LATTICE_GRID_N = int(USER_STRUCTURE_CONFIG["lattice_grid_n"])

LATTICE_NODE_BLEND_FACTOR = float(USER_STRUCTURE_CONFIG["lattice"]["node_blend_factor"])
LATTICE_BINARY_CLOSING_ITERS = int(USER_STRUCTURE_CONFIG["lattice"]["binary_closing_iters"])
LATTICE_RADIUS_SEARCH_ITERS = int(USER_STRUCTURE_CONFIG["lattice"]["radius_search_iters"])
CYLINDER_SEGMENTS = int(USER_STRUCTURE_CONFIG["lattice"]["cylinder_segments"])

BATCH_MODE = True
RUN_GENERATION_MODE = str(USER_STRUCTURE_CONFIG.get("run_generation_mode", "all")).lower()
PREVIEW_N = int(USER_STRUCTURE_CONFIG.get("preview_n", 0))   # Compatibility-safe placeholder
AUTO_RUN_GENERATION = bool(USER_STRUCTURE_CONFIG["auto_run_generation"])
SAVE_EVERY_N = int(USER_STRUCTURE_CONFIG["save_every_n"])
RESUME_MODE = bool(USER_STRUCTURE_CONFIG["resume_mode"])
OVERWRITE_EXISTING = bool(USER_STRUCTURE_CONFIG["overwrite_existing"])
EXPORT_STL = bool(USER_STRUCTURE_CONFIG["export_stl"])
EXPORT_STP = bool(USER_STRUCTURE_CONFIG["export_stp"])
EXPORT_STP_FOR_SELECTED_ONLY = bool(USER_STRUCTURE_CONFIG["export_stp_for_selected_only"])
ENFORCE_CONTACT_FACE_SYMMETRY = bool(USER_STRUCTURE_CONFIG.get("enforce_contact_face_symmetry", True))
CONTACT_SURFACE_DEPTH_VOX = int(USER_STRUCTURE_CONFIG.get("contact_surface_depth_vox", 1))
STRICT_GLOBAL_SYMMETRY = bool(USER_STRUCTURE_CONFIG.get("strict_global_symmetry", True))
STRICT_SYMMETRY_METHOD = str(USER_STRUCTURE_CONFIG.get("strict_symmetry_method", "score_threshold"))
FORCE_CONNECTED_LATTICE_VOXEL = bool(USER_STRUCTURE_CONFIG.get("force_connected_lattice_voxel", True))
CONNECTIVITY_REPAIR_MODE = str(USER_STRUCTURE_CONFIG.get("connectivity_repair_mode", "bridge")).lower()
CONNECTIVITY_BRIDGE_RADIUS_VOX = int(USER_STRUCTURE_CONFIG.get("connectivity_bridge_radius_vox", 1))
CONNECTIVITY_MIN_COMPONENT_VOXELS = int(USER_STRUCTURE_CONFIG.get("connectivity_min_component_voxels", 1))
CONNECTIVITY_MAX_BRIDGES = int(USER_STRUCTURE_CONFIG.get("connectivity_max_bridges", 200))
CONNECTIVITY_RETRY_AFTER_CONTACT_SYMMETRY = bool(USER_STRUCTURE_CONFIG.get("connectivity_retry_after_contact_symmetry", True))
RANDOM_SEED = int(USER_STRUCTURE_CONFIG["random_seed"])
rng_global = np.random.default_rng(RANDOM_SEED)

# CPU/GPU setup
try:
    import cupy as cp
    CUPY_AVAILABLE = True
except Exception:
    cp = None
    CUPY_AVAILABLE = False

COMPUTE_BACKEND = str(USER_STRUCTURE_CONFIG["compute_backend"]).lower()
GPU_DEVICE_ID = int(USER_STRUCTURE_CONFIG["gpu_device_id"])
GPU_DEVICE_IDS = [int(x) for x in USER_STRUCTURE_CONFIG.get("gpu_device_ids", [GPU_DEVICE_ID])]
PREFER_GPU_FOR_TPMS_VOXEL_FIELDS = bool(USER_STRUCTURE_CONFIG["prefer_gpu_for_tpms_voxel_fields"])

if COMPUTE_BACKEND == "gpu":
    USE_GPU_FOR_GENERATION = CUPY_AVAILABLE
elif COMPUTE_BACKEND == "auto":
    USE_GPU_FOR_GENERATION = CUPY_AVAILABLE and PREFER_GPU_FOR_TPMS_VOXEL_FIELDS
else:
    USE_GPU_FOR_GENERATION = False

DEVICE = f"cuda:{GPU_DEVICE_ID}" if USE_GPU_FOR_GENERATION else "cpu"
USE_GPU_FOR_DESCRIPTOR = False  # Descriptor extraction is kept CPU-safe in current notebook

cpu_workers_cfg = USER_STRUCTURE_CONFIG["cpu_workers"]
if isinstance(cpu_workers_cfg, str) and cpu_workers_cfg.lower() == "auto":
    N_WORKERS_CPU = max(1, min(int(USER_STRUCTURE_CONFIG["max_cpu_workers"]), (os.cpu_count() or 4) - 2))
else:
    N_WORKERS_CPU = max(1, int(cpu_workers_cfg))

N_WORKERS_GPU = max(1, int(USER_STRUCTURE_CONFIG["gpu_workers"]))
N_GPU_LANES = max(1, min(len(GPU_DEVICE_IDS), N_WORKERS_GPU)) if USE_GPU_FOR_GENERATION else 0
N_WORKERS = N_WORKERS_GPU if USE_GPU_FOR_GENERATION else N_WORKERS_CPU
PARALLEL_BACKEND = str(USER_STRUCTURE_CONFIG.get("parallel_backend", "thread")).lower()
PROGRESS_PRINT_EVERY_N = int(USER_STRUCTURE_CONFIG.get("progress_print_every_n", 5))

LATTICE_GENERATION_CONFIG = USER_STRUCTURE_CONFIG["lattice"]["modes"]
TPMS_GENERATION_CONFIG = {k: v for k, v in USER_STRUCTURE_CONFIG["tpms"].items() if k not in ["library", "basic_modes"]}
TPMS_BASIC_MODES = USER_STRUCTURE_CONFIG["tpms"]["basic_modes"]
TPMS_LIBRARY = USER_STRUCTURE_CONFIG["tpms"]["library"]
VOXEL_GENERATION_CONFIG = USER_STRUCTURE_CONFIG["voxel"]["modes"]

EXPECTED_N_CANDIDATES = (
    sum(int(v["n_candidates"]) for v in LATTICE_GENERATION_CONFIG.values())
    + len(TPMS_LIBRARY) * len(TPMS_BASIC_MODES) * int(TPMS_GENERATION_CONFIG["basic"]["n_candidates_per_tpms_mode"])
    + sum(int(TPMS_GENERATION_CONFIG[k]["n_candidates"]) for k in ["adv2", "adv3", "adv4"])
    + sum(int(v["n_candidates"]) for v in VOXEL_GENERATION_CONFIG.values())
)

# ============================================================
# Output folders
# ============================================================
DIR_CONFIG = OUTPUT_ROOT / "00_config"
DIR_TABLE = OUTPUT_ROOT / "01_candidate_table"
DIR_GEOM = OUTPUT_ROOT / "02_geometry"
DIR_LOG = OUTPUT_ROOT / "03_generation_logs"
DIR_DESC = OUTPUT_ROOT / "04_descriptors_placeholder"
DIR_SELECTED = OUTPUT_ROOT / "06_selected_candidates"

for d in [DIR_CONFIG, DIR_TABLE, DIR_GEOM, DIR_LOG, DIR_DESC, DIR_SELECTED]:
    d.mkdir(parents=True, exist_ok=True)

for gen in ["lattice", "tpms", "voxel"]:
    (DIR_GEOM / gen / "STL").mkdir(parents=True, exist_ok=True)
    (DIR_GEOM / gen / "STP_selected_only").mkdir(parents=True, exist_ok=True)

CONFIG_SNAPSHOT = {
    "RUN_TAG": RUN_TAG,
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "EXPECTED_N_CANDIDATES": EXPECTED_N_CANDIDATES,
    "FAST_SCREENING_MODE": FAST_SCREENING_MODE,
    "MARCHING_CUBES_STEP_SIZE": MARCHING_CUBES_STEP_SIZE,
    "DEVICE": DEVICE,
    "CUPY_AVAILABLE": CUPY_AVAILABLE,
    "USE_GPU_FOR_GENERATION": USE_GPU_FOR_GENERATION,
    "N_WORKERS": N_WORKERS,
    "N_WORKERS_CPU": N_WORKERS_CPU,
    "N_WORKERS_GPU": N_WORKERS_GPU,
    "GPU_DEVICE_IDS": GPU_DEVICE_IDS,
    "N_GPU_LANES": N_GPU_LANES,
    "USER_STRUCTURE_CONFIG": USER_STRUCTURE_CONFIG,
    "USER_DESCRIPTOR_CONFIG": USER_DESCRIPTOR_CONFIG,
}

with open(DIR_CONFIG / "generation_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG_SNAPSHOT, f, indent=2, ensure_ascii=False)

print("Configuration initialized")
print("Expected candidates:", EXPECTED_N_CANDIDATES)
print("Compute backend:", DEVICE, "| CuPy available:", CUPY_AVAILABLE, "| trimesh:", TRIMESH_GEN_AVAILABLE, "| workers:", N_WORKERS)
print("Run generation mode:", RUN_GENERATION_MODE, "| Preview_n (compatibility only):", PREVIEW_N)
print("Output root:", OUTPUT_ROOT.resolve())
print("Descriptor voxel size (mm):", round(DESCRIPTOR_VOXEL_SIZE_MM, 6))
print("Descriptor topology voxel size (mm):", round(DESCRIPTOR_TOPOLOGY_VOXEL_SIZE_MM, 6))
print("Descriptor graph voxel size (mm):", round(DESCRIPTOR_GRAPH_VOXEL_SIZE_MM, 6))

In [ ]:

# ============================================================
# Cell 2. Utility functions: sampling, STL writing, mesh extraction
# ============================================================

def sample_uniform(rng, bounds):
    lo, hi = bounds
    return float(rng.uniform(lo, hi))

def sample_int(rng, bounds):
    lo, hi = bounds
    return int(rng.integers(int(lo), int(hi) + 1))

def json_dumps(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True)

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def write_binary_stl(path, vertices, faces, solid_name="mesh"):
    """Fast vectorized binary STL writer without trimesh dependency."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    vertices = np.asarray(vertices, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)

    tri_vertices = vertices[faces]  # (n_faces, 3, 3)
    p0 = tri_vertices[:, 0, :]
    p1 = tri_vertices[:, 1, :]
    p2 = tri_vertices[:, 2, :]
    normals = np.cross(p1 - p0, p2 - p0)
    norms = np.linalg.norm(normals, axis=1)
    valid = norms > 1e-12
    normals[valid] = normals[valid] / norms[valid, None]
    normals[~valid] = 0.0

    # Binary STL record: normal(3f), vertices(9f), attribute uint16
    records = np.zeros(len(faces), dtype=[
        ("normal", "<f4", (3,)),
        ("vertices", "<f4", (3, 3)),
        ("attr", "<u2"),
    ])
    records["normal"] = normals.astype(np.float32)
    records["vertices"] = tri_vertices.astype(np.float32)

    header = (solid_name[:70]).encode("ascii", errors="ignore")
    header = header + b" " * (80 - len(header))
    with open(path, "wb") as f:
        f.write(header)
        f.write(struct.pack("<I", len(faces)))
        records.tofile(f)
    return path


def estimate_mask_vf(mask):
    return float(np.mean(mask.astype(bool)))


def mesh_from_binary_mask(mask, size_mm=8.0):
    """Convert boolean voxel mask to closed triangular mesh using low/high quality marching cubes."""
    mask = np.asarray(mask, dtype=bool)
    if np.count_nonzero(mask) == 0:
        return np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.int32)

    # Padding closes boundary surfaces.
    padded = np.pad(mask.astype(np.float32), pad_width=1, mode="constant", constant_values=0)
    spacing = (size_mm / mask.shape[0], size_mm / mask.shape[1], size_mm / mask.shape[2])

    step_size = max(1, int(globals().get("MARCHING_CUBES_STEP_SIZE", 1)))
    verts, faces, normals, values = measure.marching_cubes(
        padded,
        level=0.5,
        spacing=spacing,
        step_size=step_size,
        allow_degenerate=False,
    )

    # Remove padding offset.
    verts -= np.array(spacing)
    verts = np.clip(verts, 0.0, size_mm)
    return verts.astype(np.float32), faces.astype(np.int32)


def keep_largest_component(mask):
    lab, n = label(mask)
    if n == 0:
        return mask
    counts = np.bincount(lab.ravel())
    counts[0] = 0
    return lab == int(np.argmax(counts))


def count_connected_components(mask):
    """Return the number of 6-neighbor connected solid components."""
    _, n = label(np.asarray(mask, dtype=bool))
    return int(n)


def connected_component_sizes(mask):
    """Return component sizes excluding background."""
    lab, n = label(np.asarray(mask, dtype=bool))
    if n == 0:
        return []
    counts = np.bincount(lab.ravel())
    return [int(counts[i]) for i in range(1, n + 1)]


def _draw_voxel_bridge(mask, p0, p1, radius_vox=1):
    """Draw a 6-neighbor Manhattan voxel bridge between two coordinates.

    A straight rounded line can leave diagonally touching components disconnected
    under 6-neighbor labeling. This function moves one axis at a time, so the
    inserted bridge is guaranteed to be face-connected.
    """
    mask = np.asarray(mask, dtype=bool).copy()
    p0 = np.asarray(np.rint(p0), dtype=int)
    p1 = np.asarray(np.rint(p1), dtype=int)
    n = np.asarray(mask.shape, dtype=int)
    p0 = np.clip(p0, 0, n - 1)
    p1 = np.clip(p1, 0, n - 1)

    coords = []
    cur = p0.copy()
    coords.append(cur.copy())

    # Move along axes in descending gap order. This gives the shortest
    # Manhattan bridge while preserving 6-neighbor continuity.
    axis_order = list(np.argsort(-np.abs(p1 - p0)))
    for ax in axis_order:
        step = 1 if p1[ax] > cur[ax] else -1
        while cur[ax] != p1[ax]:
            cur = cur.copy()
            cur[ax] += step
            coords.append(cur.copy())

    coords = np.asarray(coords, dtype=int)
    coords = np.clip(coords, 0, n - 1)
    bridge = np.zeros_like(mask, dtype=bool)
    bridge[coords[:, 0], coords[:, 1], coords[:, 2]] = True

    r = max(0, int(radius_vox) - 1)
    if r > 0:
        bridge = binary_dilation(bridge, iterations=r)

    before = int(mask.sum())
    mask |= bridge
    added = int(mask.sum()) - before
    return mask, added


def repair_mask_connectivity(mask, mode="bridge", bridge_radius_vox=1, min_component_voxels=1, max_bridges=200):
    """Force a binary solid mask to become one connected component.

    mode="bridge" connects disconnected islands to the largest component using
    shortest voxel bridges. mode="largest" simply keeps the largest component.
    TPMS is not repaired unless this function is explicitly called.
    """
    mask = np.asarray(mask, dtype=bool).copy()
    info = {
        "component_count_before": count_connected_components(mask),
        "component_count_after": None,
        "connectivity_repaired": False,
        "connectivity_mode": str(mode),
        "bridges_added": 0,
        "voxels_added_by_bridges": 0,
        "small_components_removed": 0,
    }
    if info["component_count_before"] <= 1:
        info["component_count_after"] = info["component_count_before"]
        return mask, info

    lab, ncomp = label(mask)
    counts = np.bincount(lab.ravel())
    counts[0] = 0

    # Remove dust components if requested. Use 1 to keep and connect every island.
    min_size = max(1, int(min_component_voxels))
    if min_size > 1:
        remove_ids = [i for i in range(1, ncomp + 1) if counts[i] < min_size]
        if remove_ids:
            remove_mask = np.isin(lab, remove_ids)
            mask[remove_mask] = False
            info["small_components_removed"] = int(len(remove_ids))
            lab, ncomp = label(mask)
            counts = np.bincount(lab.ravel()) if ncomp > 0 else np.array([0])
            if len(counts) > 0:
                counts[0] = 0

    if ncomp <= 1:
        info["component_count_after"] = int(ncomp)
        info["connectivity_repaired"] = True
        return mask, info

    if str(mode).lower() == "largest":
        repaired = keep_largest_component(mask)
        info["component_count_after"] = count_connected_components(repaired)
        info["connectivity_repaired"] = True
        return repaired, info

    # Bridge mode: iteratively connect the nearest remaining component to the
    # current connected body. This preserves more geometry than keep-largest.
    largest_id = int(np.argmax(counts))
    connected = (lab == largest_id)
    remaining_ids = [i for i in range(1, ncomp + 1) if i != largest_id and counts[i] > 0]
    repaired = mask.copy()
    max_bridges = max(1, int(max_bridges))

    for _ in range(min(max_bridges, len(remaining_ids))):
        lab_cur, n_cur = label(repaired & (~connected))
        if n_cur == 0:
            break

        # Distance to current connected body; nearest indices give bridge endpoint.
        dist, inds = distance_transform_edt(~connected, return_indices=True)

        best = None
        for cid in range(1, n_cur + 1):
            coords = np.argwhere(lab_cur == cid)
            if coords.size == 0:
                continue
            dvals = dist[coords[:, 0], coords[:, 1], coords[:, 2]]
            k = int(np.argmin(dvals))
            p = coords[k]
            q = np.array([inds[0, p[0], p[1], p[2]], inds[1, p[0], p[1], p[2]], inds[2, p[0], p[1], p[2]]], dtype=int)
            d = float(dvals[k])
            if best is None or d < best[0]:
                best = (d, p, q)

        if best is None:
            break
        _, p, q = best
        repaired, added = _draw_voxel_bridge(repaired, p, q, radius_vox=bridge_radius_vox)
        info["bridges_added"] += 1
        info["voxels_added_by_bridges"] += int(added)
        connected = keep_largest_component(repaired)
        if count_connected_components(repaired) <= 1:
            break

    info["component_count_after"] = count_connected_components(repaired)
    info["connectivity_repaired"] = info["component_count_after"] <= 1
    return repaired, info


def repair_lattice_voxel_connectivity(mask, params, generator_type):
    """Apply connectivity repair only to Lattice and Voxel; TPMS is excluded."""
    if generator_type not in ["lattice", "voxel"]:
        return np.asarray(mask, dtype=bool), {
            "component_count_before": count_connected_components(mask),
            "component_count_after": count_connected_components(mask),
            "connectivity_repaired": None,
            "connectivity_mode": "not_applied_to_tpms",
            "bridges_added": 0,
            "voxels_added_by_bridges": 0,
            "small_components_removed": 0,
        }
    if not bool(params.get("force_connected", FORCE_CONNECTED_LATTICE_VOXEL)):
        return np.asarray(mask, dtype=bool), {
            "component_count_before": count_connected_components(mask),
            "component_count_after": count_connected_components(mask),
            "connectivity_repaired": False,
            "connectivity_mode": "disabled",
            "bridges_added": 0,
            "voxels_added_by_bridges": 0,
            "small_components_removed": 0,
        }
    return repair_mask_connectivity(
        mask,
        mode=params.get("connectivity_repair_mode", CONNECTIVITY_REPAIR_MODE),
        bridge_radius_vox=int(params.get("connectivity_bridge_radius_vox", CONNECTIVITY_BRIDGE_RADIUS_VOX)),
        min_component_voxels=int(params.get("connectivity_min_component_voxels", CONNECTIVITY_MIN_COMPONENT_VOXELS)),
        max_bridges=int(params.get("connectivity_max_bridges", CONNECTIVITY_MAX_BRIDGES)),
    )


def enforce_vf_by_rank(score, target_vf=0.30, available=None, prefer_high=True):
    """Select target fraction of voxels by score ranking."""
    score = np.asarray(score)
    if available is None:
        available = np.ones(score.shape, dtype=bool)
    else:
        available = np.asarray(available, dtype=bool)

    idx = np.flatnonzero(available.ravel())
    n_total = score.size
    k = int(round(target_vf * n_total))
    k = max(1, min(k, len(idx)))
    values = score.ravel()[idx]
    order = np.argsort(values)
    if prefer_high:
        chosen = idx[order[-k:]]
    else:
        chosen = idx[order[:k]]
    mask = np.zeros(score.size, dtype=bool)
    mask[chosen] = True
    return mask.reshape(score.shape)


def approximate_lattice_vf_from_edges(nodes, edges, radius, size_mm=8.0):
    vol = 0.0
    for i, j in edges:
        L = np.linalg.norm(nodes[j] - nodes[i])
        vol += math.pi * radius * radius * L
    return float(vol / (size_mm ** 3))


def try_export_step_from_stl(stl_path, stp_path):
    """Optional FreeCAD mesh-to-shape STEP export. Best used only for selected candidates."""
    try:
        import FreeCAD
        import Mesh
        import Part
        mesh = Mesh.Mesh(str(stl_path))
        shape = Part.Shape()
        shape.makeShapeFromMesh(mesh.Topology, 0.1)
        solid = Part.makeSolid(shape)
        solid.exportStep(str(stp_path))
        return True, "ok"
    except Exception as e:
        return False, str(e)


In [ ]:

# ============================================================
# Cell 3. Lattice / strut graph generator
# ============================================================
# Important fix:
# - Lattice struts are no longer exported as separate capped cylinders.
# - Instead, struts and node regions are voxelized as one implicit union,
#   then a single mesh is extracted by marching cubes.
# - This gives smooth node junctions.
# - When a strut or node reaches the unit-cell boundary, the geometry is
#   clipped by the axis-aligned box planes of the 8 mm cube, so tiled parts
#   can be connected later without oblique strut-axis cut faces.


def generate_periodic_face_nodes(rng, size_mm, n_face, axis):
    """Generate matched node pairs on opposite faces for one axis."""
    pts = []
    uv = rng.random((n_face, 2)) * size_mm
    for a, b in uv:
        p0 = np.zeros(3)
        p1 = np.zeros(3)
        if axis == 0:  # x faces
            p0[:] = [0.0, a, b]
            p1[:] = [size_mm, a, b]
        elif axis == 1:  # y faces
            p0[:] = [a, 0.0, b]
            p1[:] = [a, size_mm, b]
        else:  # z faces
            p0[:] = [a, b, 0.0]
            p1[:] = [a, b, size_mm]
        pts.extend([p0, p1])
    return pts


def _face_pattern_points(rng, size_mm, n_points, margin_frac=0.12):
    """Create a reusable 2D face-contact pattern in local face coordinates."""
    n_points = int(max(1, n_points))
    margin = float(size_mm) * float(margin_frac)
    lo, hi = margin, float(size_mm) - margin
    if hi <= lo:
        lo, hi = 0.0, float(size_mm)
    return rng.uniform(lo, hi, size=(n_points, 2))


def _place_face_pattern(pattern_uv, size_mm, face):
    """Map a common 2D pattern onto a cube face.

    Face names:
        x0, x1, y0, y1, z0, z1

    The local coordinates are consistently interpreted so that identical
    pattern_uv gives the same contact-node layout on every requested face.
    """
    pts = []
    size = float(size_mm)
    for u, v in np.asarray(pattern_uv, dtype=float):
        if face == "x0":
            pts.append([0.0, u, v])
        elif face == "x1":
            pts.append([size, u, v])
        elif face == "y0":
            pts.append([u, 0.0, v])
        elif face == "y1":
            pts.append([u, size, v])
        elif face == "z0":
            pts.append([u, v, 0.0])
        elif face == "z1":
            pts.append([u, v, size])
        else:
            raise ValueError(f"Unknown face: {face}")
    return pts


def generate_symmetric_contact_face_nodes(rng, size_mm, mode, n_nodes):
    """Generate face nodes with exact contact-pattern symmetry.

    - periodic_isotropic:
      all six cube faces receive the same local 2D contact-node pattern.
    - periodic_orthotropic:
      the four side faces receive the same local 2D pattern, while top/bottom
      receive another common local 2D pattern.
    """
    pts = []
    mode = str(mode)
    if mode == "periodic_isotropic":
        # Six equivalent faces. Keep the pattern compact enough that total node
        # count is still controlled by node_count_range.
        n_face = max(2, int(round(n_nodes / 18)))
        iso_pattern = _face_pattern_points(rng, size_mm, n_face)
        for face in ["x0", "x1", "y0", "y1", "z0", "z1"]:
            pts += _place_face_pattern(iso_pattern, size_mm, face)
    elif mode == "periodic_orthotropic":
        # Side group = 4 equivalent faces; vertical group = top/bottom pair.
        n_side = max(2, int(round(n_nodes / 24)))
        n_top_bottom = max(3, int(round(n_nodes / 16)))
        side_pattern = _face_pattern_points(rng, size_mm, n_side)
        z_pattern = _face_pattern_points(rng, size_mm, n_top_bottom)
        for face in ["x0", "x1", "y0", "y1"]:
            pts += _place_face_pattern(side_pattern, size_mm, face)
        for face in ["z0", "z1"]:
            pts += _place_face_pattern(z_pattern, size_mm, face)
    return pts


def _canonical_face_layer(mask, face, d=0):
    """Return a boundary/contact layer in a common local 2D coordinate system.

    Local convention:
    - x0/x1 faces: (u, v) = (y, z)
    - y0/y1 faces: (u, v) = (x, z)
    - z0/z1 faces: (u, v) = (x, y)

    This avoids the previous bug where x/y/z faces were copied/compared without
    an explicit local-coordinate convention.
    """
    mask = np.asarray(mask, dtype=bool)
    d = int(d)
    if face == "x0":
        return mask[d, :, :].copy()
    if face == "x1":
        return mask[-1 - d, :, :].copy()
    if face == "y0":
        return mask[:, d, :].copy()
    if face == "y1":
        return mask[:, -1 - d, :].copy()
    if face == "z0":
        return mask[:, :, d].copy()
    if face == "z1":
        return mask[:, :, -1 - d].copy()
    raise ValueError(f"Unknown face name: {face}")


def _set_canonical_face_layer(mask, face, pattern, d=0):
    """Write a canonical 2D pattern to a selected boundary/contact layer."""
    pattern = np.asarray(pattern, dtype=bool)
    d = int(d)
    if face == "x0":
        mask[d, :, :] = pattern
    elif face == "x1":
        mask[-1 - d, :, :] = pattern
    elif face == "y0":
        mask[:, d, :] = pattern
    elif face == "y1":
        mask[:, -1 - d, :] = pattern
    elif face == "z0":
        mask[:, :, d] = pattern
    elif face == "z1":
        mask[:, :, -1 - d] = pattern
    else:
        raise ValueError(f"Unknown face name: {face}")
    return mask


def _local_face_slices(mask):
    """Backward-compatible outer-face accessor in canonical local coordinates."""
    return {face: _canonical_face_layer(mask, face, d=0) for face in ["x0", "x1", "y0", "y1", "z0", "z1"]}


def _stabilize_contact_pattern_edges(pattern):
    """Remove edge-line conflicts so face assignments stay deterministic.

    A cube edge belongs to two faces. If both faces write arbitrary edge pixels,
    the final edge can depend on assignment order. Clearing the 2D pattern edge
    lines makes the six/six or four/two face constraints exactly reproducible.
    """
    pattern = np.asarray(pattern, dtype=bool).copy()
    if pattern.shape[0] >= 2:
        pattern[0, :] = False
        pattern[-1, :] = False
    if pattern.shape[1] >= 2:
        pattern[:, 0] = False
        pattern[:, -1] = False
    return pattern


def _contact_depth(mask, depth_vox=1):
    n = int(np.asarray(mask).shape[0])
    return max(1, min(int(depth_vox), max(1, n // 8)))


def enforce_contact_face_symmetry(mask, mode, depth_vox=1):
    """Force only Lattice/Voxel contact layers to satisfy the required face rule.

    Rules implemented here:
    - periodic_isotropic: x0, x1, y0, y1, z0, z1 have the same canonical pattern.
    - periodic_orthotropic: x0, x1, y0, y1 share one side pattern; z0, z1 share
      another top/bottom pattern.

    TPMS is not passed through this function in the generation pipeline.
    """
    if not bool(globals().get("ENFORCE_CONTACT_FACE_SYMMETRY", True)):
        return mask

    mask = np.asarray(mask, dtype=bool).copy()
    mode = str(mode)
    depth = _contact_depth(mask, depth_vox)

    if mode == "periodic_isotropic":
        face_group = ["x0", "x1", "y0", "y1", "z0", "z1"]
        for d in range(depth):
            pattern = np.zeros_like(_canonical_face_layer(mask, "x0", d), dtype=bool)
            for face in face_group:
                pattern |= _canonical_face_layer(mask, face, d)
            pattern = _stabilize_contact_pattern_edges(pattern)
            for face in face_group:
                _set_canonical_face_layer(mask, face, pattern, d)

    elif mode == "periodic_orthotropic":
        side_faces = ["x0", "x1", "y0", "y1"]
        z_faces = ["z0", "z1"]
        for d in range(depth):
            side_pattern = np.zeros_like(_canonical_face_layer(mask, "x0", d), dtype=bool)
            for face in side_faces:
                side_pattern |= _canonical_face_layer(mask, face, d)
            side_pattern = _stabilize_contact_pattern_edges(side_pattern)

            z_pattern = np.zeros_like(_canonical_face_layer(mask, "z0", d), dtype=bool)
            for face in z_faces:
                z_pattern |= _canonical_face_layer(mask, face, d)
            z_pattern = _stabilize_contact_pattern_edges(z_pattern)

            for face in side_faces:
                _set_canonical_face_layer(mask, face, side_pattern, d)
            for face in z_faces:
                _set_canonical_face_layer(mask, face, z_pattern, d)

    return mask


def contact_face_symmetry_report(mask, mode, depth_vox=1):
    """Quantify face-rule mismatch over the enforced contact depth."""
    mask = np.asarray(mask, dtype=bool)
    mode = str(mode)
    depth = _contact_depth(mask, depth_vox)

    def mismatch(a, b):
        return int(np.count_nonzero(np.asarray(a, dtype=bool) ^ np.asarray(b, dtype=bool)))

    mismatches = {}
    if mode == "periodic_isotropic":
        for d in range(depth):
            ref = _canonical_face_layer(mask, "x0", d)
            for face in ["x1", "y0", "y1", "z0", "z1"]:
                mismatches[f"layer{d}_x0_vs_{face}"] = mismatch(ref, _canonical_face_layer(mask, face, d))
        ok = all(v == 0 for v in mismatches.values())

    elif mode == "periodic_orthotropic":
        for d in range(depth):
            side_ref = _canonical_face_layer(mask, "x0", d)
            for face in ["x1", "y0", "y1"]:
                mismatches[f"layer{d}_side_x0_vs_{face}"] = mismatch(side_ref, _canonical_face_layer(mask, face, d))
            z_ref = _canonical_face_layer(mask, "z0", d)
            mismatches[f"layer{d}_top_vs_bottom"] = mismatch(z_ref, _canonical_face_layer(mask, "z1", d))
        ok = all(v == 0 for v in mismatches.values())

    else:
        ok = None

    return ok, mismatches




# ============================================================
# Strict 3D symmetry helpers for Lattice/Voxel masks
# ============================================================
# The previous version only copied the outer contact layers. That can make the
# boundary report pass while the actual 3D unit cell still does not look
# isotropic/orthotropic. These helpers symmetrize the full voxel mask.

def _axis_symmetry_views(arr, mode):
    """Return transformed views/copies representing the requested 3D symmetry group.

    periodic_isotropic:
        all axis permutations + axis flips are allowed. This makes x/y/z
        statistically/geometrically equivalent at the voxel-mask level.
    periodic_orthotropic:
        x and y directions are equivalent; z is independent but top/bottom are
        mirrored. This gives four equivalent side faces and one equivalent
        top-bottom pair.
    """
    arr = np.asarray(arr)
    mode = str(mode)
    outs = []

    if mode == "periodic_isotropic":
        from itertools import permutations, product
        for perm in permutations((0, 1, 2)):
            a = np.transpose(arr, perm)
            for flips in product([False, True], repeat=3):
                b = a
                for ax, do_flip in enumerate(flips):
                    if do_flip:
                        b = np.flip(b, axis=ax)
                outs.append(np.asarray(b))

    elif mode == "periodic_orthotropic":
        # D4-like xy symmetry + z mirror. z remains a distinct material axis.
        base_ops = [
            arr,
            np.swapaxes(arr, 0, 1),
            np.flip(arr, axis=0),
            np.flip(arr, axis=1),
            np.flip(np.swapaxes(arr, 0, 1), axis=0),
            np.flip(np.swapaxes(arr, 0, 1), axis=1),
            np.flip(np.flip(arr, axis=0), axis=1),
            np.flip(np.flip(np.swapaxes(arr, 0, 1), axis=0), axis=1),
        ]
        for a in base_ops:
            outs.append(np.asarray(a))
            outs.append(np.asarray(np.flip(a, axis=2)))
    else:
        outs = [arr]

    # Remove accidental duplicates to reduce compute while preserving symmetry.
    unique = []
    seen = set()
    for a in outs:
        key = (a.shape, a.strides, a.__array_interface__["data"][0] if np.shares_memory(a, arr) else hash(a.tobytes()[:64]))
        # Do not rely on the key for mathematical uniqueness; it is only a fast filter.
        unique.append(np.asarray(a))
    return unique


def symmetrize_scalar_field(field, mode):
    """Average a scalar field over the symmetry group before thresholding."""
    mode = str(mode)
    if mode not in ["periodic_isotropic", "periodic_orthotropic"]:
        return np.asarray(field)
    views = _axis_symmetry_views(np.asarray(field, dtype=np.float32), mode)
    acc = np.zeros_like(field, dtype=np.float32)
    for v in views:
        acc += np.asarray(v, dtype=np.float32)
    acc /= max(1, len(views))
    acc = (acc - np.mean(acc)) / (np.std(acc) + 1e-12)
    return acc.astype(np.float32)


def symmetrize_binary_mask(mask, mode, target_vf=None, method="score_threshold"):
    """Make a full 3D binary mask obey the requested global symmetry.

    method="or": keeps every voxel that appears in any symmetric copy. This is
    the strongest connectivity-preserving option but can increase VF.

    method="score_threshold": sums all symmetric copies and keeps score levels
    closest to the requested VF. Because the threshold is applied to the
    invariant score field, the result remains exactly symmetric.
    """
    mode = str(mode)
    if mode not in ["periodic_isotropic", "periodic_orthotropic"]:
        return np.asarray(mask, dtype=bool)

    mask = np.asarray(mask, dtype=bool)
    views = _axis_symmetry_views(mask.astype(np.uint8), mode)
    score = np.zeros(mask.shape, dtype=np.uint16)
    for v in views:
        score += np.asarray(v, dtype=np.uint16)

    method = str(method).lower()
    if method == "or" or target_vf is None:
        out = score > 0
    else:
        target = float(target_vf)
        unique_scores = np.unique(score)
        unique_scores = unique_scores[unique_scores > 0]
        if len(unique_scores) == 0:
            out = mask.copy()
        else:
            best_t, best_err = unique_scores[0], 1e9
            for t in unique_scores:
                vf = float(np.mean(score >= t))
                err = abs(vf - target)
                if err < best_err:
                    best_t, best_err = t, err
            out = score >= best_t

    return np.asarray(out, dtype=bool)


def strongest_connectivity_repair(mask, params, mode):
    """Guarantee one connected solid body while preserving symmetry.

    Components are bridged to the nearest center voxel, then the bridge itself is
    symmetrized. This avoids the common failure where a one-sided bridge fixes
    connectivity but breaks isotropic/orthotropic appearance.
    """
    mask = np.asarray(mask, dtype=bool).copy()
    bridge_radius = _safe_positive_int_scalar(params.get("connectivity_bridge_radius_vox", CONNECTIVITY_BRIDGE_RADIUS_VOX), CONNECTIVITY_BRIDGE_RADIUS_VOX)
    max_bridges = _safe_positive_int_scalar(params.get("connectivity_max_bridges", CONNECTIVITY_MAX_BRIDGES), CONNECTIVITY_MAX_BRIDGES)

    info = {"strong_bridges_added": 0, "strong_bridge_voxels_added": 0}
    for _ in range(8):
        lab, ncomp = label(mask)
        if ncomp <= 1:
            break
        center = (np.array(mask.shape) - 1) / 2.0
        center_idx = np.rint(center).astype(int)

        # Make sure the exact center region is solid. This creates a common hub
        # so all symmetrized bridges meet at one place.
        hub = np.zeros_like(mask, dtype=bool)
        hub[tuple(center_idx)] = True
        hub = binary_dilation(hub, iterations=max(1, bridge_radius))
        before = int(mask.sum())
        mask |= hub
        info["strong_bridge_voxels_added"] += int(mask.sum()) - before

        lab, ncomp = label(mask)
        if ncomp <= 1:
            break
        sizes = np.bincount(lab.ravel())
        sizes[0] = 0
        component_ids = [i for i in range(1, ncomp + 1) if sizes[i] > 0]
        component_ids = sorted(component_ids, key=lambda i: sizes[i], reverse=True)

        added_this_round = 0
        for cid in component_ids[:max_bridges]:
            coords = np.argwhere(lab == cid)
            if coords.size == 0:
                continue
            d2 = np.sum((coords.astype(float) - center) ** 2, axis=1)
            p = coords[int(np.argmin(d2))]
            mask, added = _draw_voxel_bridge(mask, p, center_idx, radius_vox=bridge_radius)
            info["strong_bridges_added"] += 1
            info["strong_bridge_voxels_added"] += int(added)
            added_this_round += int(added)
        # Duplicate the repair bridges according to the selected symmetry group.
        mask = symmetrize_binary_mask(mask, mode, target_vf=None, method="or")
        mask = binary_closing(mask, iterations=1)
        if added_this_round == 0 and count_connected_components(mask) > 1:
            # Last-resort: use a slightly thicker hub/bridge.
            bridge_radius += 1

    return mask, info

def _safe_positive_int_scalar(x, default=1):
    if isinstance(x, (list, tuple, np.ndarray)):
        if len(x) == 0:
            return int(default)
        x = x[0]
    try:
        if pd.isna(x):
            return int(default)
    except Exception:
        pass
    try:
        return max(1, int(round(float(x))))
    except Exception:
        return int(default)


def _connect_components_through_interior(mask, bridge_radius_vox=1, max_bridges=200):
    """Connect components using center-directed bridges that avoid relying on faces.

    This is used only after face-rule enforcement when ordinary closest-point
    repair repeatedly creates bridges that are later removed from contact faces.
    """
    mask = np.asarray(mask, dtype=bool).copy()
    lab, ncomp = label(mask)
    if ncomp <= 1:
        return mask, {"interior_bridges_added": 0, "interior_voxels_added_by_bridges": 0}

    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    root_label = int(np.argmax(sizes))
    center = (np.array(mask.shape, dtype=float) - 1.0) / 2.0

    def closest_to_center(component_label):
        coords = np.argwhere(lab == component_label)
        if coords.size == 0:
            return None
        dist = np.sum((coords.astype(float) - center) ** 2, axis=1)
        return coords[int(np.argmin(dist))]

    root = closest_to_center(root_label)
    if root is None:
        return mask, {"interior_bridges_added": 0, "interior_voxels_added_by_bridges": 0}

    bridge_radius_vox = _safe_positive_int_scalar(bridge_radius_vox, 1)
    max_bridges = _safe_positive_int_scalar(max_bridges, 200)

    bridges = 0
    added_total = 0
    for component_label in range(1, ncomp + 1):
        if component_label == root_label:
            continue
        p = closest_to_center(component_label)
        if p is None:
            continue
        mask, added = _draw_voxel_bridge(mask, p, root, radius_vox=bridge_radius_vox)
        added_total += int(added)
        bridges += 1
        if bridges >= max_bridges:
            break

    return mask, {
        "interior_bridges_added": int(bridges),
        "interior_voxels_added_by_bridges": int(added_total),
    }


def finalize_lattice_voxel_mask(mask, params, generator_type, mode_key, max_iter=6):
    """Finalize Lattice/Voxel masks with strict global symmetry + connectivity.

    Stronger than the previous boundary-only correction:
    1) symmetrize the full 3D mask for isotropic/orthotropic Lattice/Voxel modes,
    2) enforce exact contact-face equality over several voxel layers,
    3) repair disconnected components using center-directed symmetric bridges,
    4) re-apply symmetry and face rules, then validate both conditions.
    """
    generator_type = str(generator_type).lower()
    if generator_type not in ["lattice", "voxel"]:
        return np.asarray(mask, dtype=bool), {
            "finalizer_skipped": True,
            "reason": "TPMS_or_non_lattice_voxel_generator",
        }

    mode = str(params.get(mode_key, "stochastic"))
    depth = int(params.get("contact_surface_depth_vox", CONTACT_SURFACE_DEPTH_VOX))
    target_vf = float(params.get("target_vf", TARGET_VF))
    strict = bool(params.get("strict_global_symmetry", STRICT_GLOBAL_SYMMETRY))
    method = str(params.get("strict_symmetry_method", STRICT_SYMMETRY_METHOD))

    mask = np.asarray(mask, dtype=bool).copy()
    all_info = {
        "finalizer_skipped": False,
        "face_rule_mode": mode,
        "strict_global_symmetry": bool(strict and mode in ["periodic_isotropic", "periodic_orthotropic"]),
        "face_rule_depth_vox": int(_contact_depth(mask, depth)),
    }

    if strict and mode in ["periodic_isotropic", "periodic_orthotropic"]:
        # For the initial mask, use score-thresholding to avoid exploding VF.
        mask = symmetrize_binary_mask(mask, mode, target_vf=target_vf, method=method)

    for it in range(int(max_iter)):
        mask = enforce_contact_face_symmetry(mask, mode, depth_vox=depth)

        # Strong repair first; it makes a single hub-connected body and then
        # mirrors the repair, so symmetry is preserved rather than destroyed.
        if count_connected_components(mask) > 1:
            mask, strong_info = strongest_connectivity_repair(mask, params, mode)
            for k, v in strong_info.items():
                all_info[f"iter{it}_{k}"] = v

        mask, info = repair_lattice_voxel_connectivity(mask, params, generator_type=generator_type)
        for k, v in info.items():
            all_info[f"iter{it}_{k}"] = v

        if strict and mode in ["periodic_isotropic", "periodic_orthotropic"]:
            # After bridges are added, use OR-symmetry to preserve connectivity.
            mask = symmetrize_binary_mask(mask, mode, target_vf=None, method="or")
            mask = binary_closing(mask, iterations=1)

        mask = enforce_contact_face_symmetry(mask, mode, depth_vox=depth)

        ok, mismatches = contact_face_symmetry_report(mask, mode, depth_vox=depth)
        ncomp = count_connected_components(mask)
        all_info[f"iter{it}_post_face_symmetry_ok"] = None if ok is None else bool(ok)
        all_info[f"iter{it}_post_component_count"] = int(ncomp)
        all_info[f"iter{it}_vf"] = float(estimate_mask_vf(mask))
        if (ok is None or ok) and ncomp <= 1:
            break

    # Final hard pass.
    if strict and mode in ["periodic_isotropic", "periodic_orthotropic"]:
        mask = symmetrize_binary_mask(mask, mode, target_vf=None, method="or")
        mask = binary_closing(mask, iterations=1)
    mask = enforce_contact_face_symmetry(mask, mode, depth_vox=depth)
    if count_connected_components(mask) > 1:
        mask, strong_info = strongest_connectivity_repair(mask, params, mode)
        for k, v in strong_info.items():
            all_info[f"final_{k}"] = v
        if strict and mode in ["periodic_isotropic", "periodic_orthotropic"]:
            mask = symmetrize_binary_mask(mask, mode, target_vf=None, method="or")
            mask = binary_closing(mask, iterations=1)
        mask = enforce_contact_face_symmetry(mask, mode, depth_vox=depth)

    ok, mismatches = contact_face_symmetry_report(mask, mode, depth_vox=depth)
    ncomp = count_connected_components(mask)
    all_info["final_contact_face_symmetry_ok"] = None if ok is None else bool(ok)
    all_info["final_contact_face_mismatch_voxels"] = json_dumps(mismatches)
    all_info["final_component_count"] = int(ncomp)
    all_info["final_actual_vf_est"] = float(estimate_mask_vf(mask))
    return mask, all_info


def _reflect_points_isotropic(base_points, size_mm):
    """Reflect points from a 1/8 octant model [0, L/2]^3 to the full cube."""
    size = float(size_mm)
    mids = np.array([size/2.0, size/2.0, size/2.0], dtype=float)
    transforms = [(sx, sy, sz) for sx in [0,1] for sy in [0,1] for sz in [0,1]]
    pts = []
    maps = []
    for p in np.asarray(base_points, dtype=float):
        mp = {}
        for sx, sy, sz in transforms:
            q = np.array([
                p[0] if sx == 0 else size - p[0],
                p[1] if sy == 0 else size - p[1],
                p[2] if sz == 0 else size - p[2],
            ], dtype=float)
            key = tuple(np.round(q, 6))
            mp[(sx, sy, sz)] = key
            pts.append(key)
        maps.append(mp)
    uniq = pd.DataFrame(list(dict.fromkeys(pts)), columns=["x","y","z"])
    nodes = uniq[["x","y","z"]].values.astype(float)
    key_to_idx = {tuple(np.round(p,6)): i for i,p in enumerate(nodes)}
    groups = [{k: key_to_idx[v] for k,v in mp.items()} for mp in maps]
    return nodes, groups, transforms

def _reflect_points_orthotropic(base_points, size_mm):
    """Reflect points from a 1/4 model [0, L/2]x[0, L/2]x[0, L] to the full cube."""
    size = float(size_mm)
    transforms = [(sx, sy) for sx in [0,1] for sy in [0,1]]
    pts = []
    maps = []
    for p in np.asarray(base_points, dtype=float):
        mp = {}
        for sx, sy in transforms:
            q = np.array([
                p[0] if sx == 0 else size - p[0],
                p[1] if sy == 0 else size - p[1],
                p[2],
            ], dtype=float)
            key = tuple(np.round(q, 6))
            mp[(sx, sy)] = key
            pts.append(key)
        maps.append(mp)
    uniq = pd.DataFrame(list(dict.fromkeys(pts)), columns=["x","y","z"])
    nodes = uniq[["x","y","z"]].values.astype(float)
    key_to_idx = {tuple(np.round(p,6)): i for i,p in enumerate(nodes)}
    groups = [{k: key_to_idx[v] for k,v in mp.items()} for mp in maps]
    return nodes, groups, transforms


def _augment_base_points_for_fullmodel_connectivity(base, size, kind, rng):
    """Add invariant seam nodes so reflected copies can share common graph anchors.

    isotropic:
        add symmetry-plane nodes on x=L/2, y=L/2, z=L/2 and the center node.
        These nodes are invariant under the corresponding reflections and make it
        possible to connect all 8 octants through shared seam nodes.
    orthotropic:
        add x/y symmetry-plane seam nodes and a central spine line x=L/2,y=L/2.
        This makes the 4 reflected quarter-models share a common connected spine.
    """
    base = np.asarray(base, dtype=float)
    pts = [base]
    eps = 0.08 * size
    if kind == "isotropic":
        n_seam = max(3, int(round(len(base) / 6)))
        uv = _face_pattern_points(rng, size / 2.0, n_seam, margin_frac=0.12)
        seam = []
        for u, v in uv:
            seam.extend([[size/2.0, u, v], [u, size/2.0, v], [u, v, size/2.0]])
        seam.append([size/2.0, size/2.0, size/2.0])
        pts.append(np.asarray(seam, dtype=float))
    elif kind == "orthotropic":
        n_spine = max(4, int(round(len(base) / 8)))
        zvals = np.linspace(eps, size - eps, n_spine)
        seam = [[size/2.0, size/2.0, z] for z in zvals]
        n_plane = max(2, int(round(len(base) / 10)))
        uv = _face_pattern_points(rng, size, n_plane, margin_frac=0.12)
        for u, z in uv:
            seam.append([size/2.0, 0.5*u, z])
            seam.append([0.5*u, size/2.0, z])
        pts.append(np.asarray(seam, dtype=float))
    out = np.vstack(pts)
    df = pd.DataFrame(np.round(out, 6), columns=["x","y","z"]).drop_duplicates()
    return df[["x","y","z"]].values.astype(float)


def _symmetry_seam_indices(points, size, kind, tol=None):
    pts = np.asarray(points, dtype=float)
    if tol is None:
        tol = max(1e-6, 1e-4 * float(size))
    if kind == "isotropic":
        m = (
            np.isclose(pts[:, 0], size / 2.0, atol=tol)
            | np.isclose(pts[:, 1], size / 2.0, atol=tol)
            | np.isclose(pts[:, 2], size / 2.0, atol=tol)
        )
    elif kind == "orthotropic":
        m = (
            np.isclose(pts[:, 0], size / 2.0, atol=tol)
            | np.isclose(pts[:, 1], size / 2.0, atol=tol)
        )
    else:
        m = np.zeros(len(pts), dtype=bool)
    return np.where(m)[0].tolist()


def _connect_components_via_seams(points, edges, seam_idx, target_edges=None):
    """Make the base graph connected by wiring each component to the seam backbone."""
    pts = np.asarray(points, dtype=float)
    n = len(pts)
    if n <= 1:
        return []
    D = squareform(pdist(pts))
    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from([tuple(map(int, e)) for e in edges])
    edge_set = set(tuple(sorted(map(int, e))) for e in edges)

    if not seam_idx:
        # Fallback: choose the node closest to the bbox center as seam anchor.
        center = pts.mean(axis=0, keepdims=True)
        seam_idx = [int(np.argmin(np.sum((pts - center) ** 2, axis=1)))]

    # First, connect seam nodes into one backbone.
    seam_idx = sorted(set(int(i) for i in seam_idx))
    if len(seam_idx) >= 2:
        seam_order = sorted(seam_idx, key=lambda i: (pts[i, 2], pts[i, 1], pts[i, 0]))
        for a, b in zip(seam_order[:-1], seam_order[1:]):
            e = tuple(sorted((int(a), int(b))))
            if e not in edge_set:
                edge_set.add(e)
                G.add_edge(*e)

    # Then connect each disconnected component to the nearest seam node.
    changed = True
    while changed and not nx.is_connected(G):
        changed = False
        comps = list(nx.connected_components(G))
        seam_comp = None
        for comp in comps:
            if any(i in comp for i in seam_idx):
                seam_comp = set(comp)
                break
        if seam_comp is None:
            seam_comp = {seam_idx[0]}
        for comp in comps:
            comp = set(comp)
            if comp == seam_comp:
                continue
            best = None
            for i in comp:
                j = min(seam_idx, key=lambda s: D[i, s])
                cand = (D[i, j], int(i), int(j))
                if (best is None) or (cand[0] < best[0]):
                    best = cand
            if best is not None:
                _, a, b = best
                e = tuple(sorted((a, b)))
                if e not in edge_set:
                    edge_set.add(e)
                    G.add_edge(*e)
                    changed = True
            # update seam component after each bridge
            if nx.is_connected(G):
                break
            comps2 = list(nx.connected_components(G))
            for c in comps2:
                if any(i in c for i in seam_idx):
                    seam_comp = set(c)
                    break

    # Optional: if target_edges requested, add short local edges up to target.
    if target_edges is not None and len(edge_set) < int(target_edges):
        candidate = []
        for i in range(n):
            order = np.argsort(D[i])[1:min(n, 12)]
            for j in order:
                if i < j:
                    candidate.append((D[i, j], int(i), int(j)))
        candidate.sort(key=lambda x: x[0])
        for _, i, j in candidate:
            e = tuple(sorted((i, j)))
            if e not in edge_set:
                edge_set.add(e)
                if len(edge_set) >= int(target_edges):
                    break
    return sorted(edge_set)


def generate_lattice_nodes(params, seed=0):
    rng = np.random.default_rng(seed)
    size = float(params["size_mm"])
    n_nodes = int(params["node_count"])
    mode = str(params["lattice_mode"])
    perturb = float(params.get("node_perturbation", 0.05))

    if mode == "periodic_isotropic":
        n_base = max(3, int(math.ceil(n_nodes / 8.0)))
        interior = rng.random((n_base, 3)) * (size / 2.0)
        n_face = max(2, int(round(n_base / 4)))
        uv = _face_pattern_points(rng, size / 2.0, n_face, margin_frac=0.08)
        face_pts = []
        for u, v in uv:
            face_pts.extend([[0.0, u, v], [u, 0.0, v], [u, v, 0.0]])
        base = np.vstack([interior, np.asarray(face_pts, dtype=float)])
        if perturb > 0:
            m = np.all(base > 1e-9, axis=1)
            base[m] += rng.normal(0, perturb * size * 0.2, base[m].shape)
            base = np.clip(base, 0.0, size / 2.0)
        base = _augment_base_points_for_fullmodel_connectivity(base, size, "isotropic", rng)
        nodes, groups, transforms = _reflect_points_isotropic(base, size)
        params["_lattice_symmetry_kind"] = "isotropic"
        params["_lattice_base_points"] = np.asarray(base, dtype=float)
        params["_lattice_symmetry_groups"] = groups
        params["_lattice_symmetry_transforms"] = transforms
        return nodes

    if mode == "periodic_orthotropic":
        n_base = max(4, int(math.ceil(n_nodes / 4.0)))
        interior = np.column_stack([
            rng.random(n_base) * (size / 2.0),
            rng.random(n_base) * (size / 2.0),
            rng.random(n_base) * size,
        ])
        n_side = max(2, int(round(n_base / 5)))
        uv_side = _face_pattern_points(rng, size, n_side, margin_frac=0.08)
        side_pts = [[0.0, u * 0.5, v] for u, v in uv_side] + [[u * 0.5, 0.0, v] for u, v in uv_side]
        base = np.vstack([interior, np.asarray(side_pts, dtype=float)])
        if perturb > 0:
            m = np.all(base[:, :2] > 1e-9, axis=1)
            base[m] += rng.normal(0, perturb * size * 0.2, base[m].shape)
            base[:, 0] = np.clip(base[:, 0], 0.0, size / 2.0)
            base[:, 1] = np.clip(base[:, 1], 0.0, size / 2.0)
            base[:, 2] = np.clip(base[:, 2], 0.0, size)
        base = _augment_base_points_for_fullmodel_connectivity(base, size, "orthotropic", rng)
        nodes, groups, transforms = _reflect_points_orthotropic(base, size)
        params["_lattice_symmetry_kind"] = "orthotropic"
        params["_lattice_base_points"] = np.asarray(base, dtype=float)
        params["_lattice_symmetry_groups"] = groups
        params["_lattice_symmetry_transforms"] = transforms
        return nodes

    pts = rng.random((n_nodes, 3)) * size
    if perturb > 0:
        pts += rng.normal(0, perturb * size, pts.shape)
        pts = np.clip(pts, 0, size)
    df = pd.DataFrame(np.round(np.asarray(pts, dtype=float), 6), columns=["x", "y", "z"]).drop_duplicates()
    params["_lattice_symmetry_kind"] = "stochastic"
    params["_lattice_base_points"] = df[["x", "y", "z"]].values.astype(float)
    return df[["x", "y", "z"]].values.astype(float)


def _select_edges_on_points(points, params):
    rng = np.random.default_rng(params.get("seed", 0))
    n = len(points)
    if n <= 1:
        return []
    target_edges = min(int(params["strut_count"]), n * (n - 1) // 2)
    k = int(params.get("connectivity_k", 4))
    vertical_bias = float(params.get("vertical_bias", 0.5))
    diagonal_bias = float(params.get("diagonal_bias", 0.5))

    D = squareform(pdist(points))
    candidate = []
    for i in range(n):
        order = np.argsort(D[i])[1:min(n, k + 10)]
        for j in order:
            if i < j:
                v = points[j] - points[i]
                L = np.linalg.norm(v)
                if L <= 1e-9:
                    continue
                z_align = abs(v[2]) / L
                diag_score = 1.0 - abs(z_align - 0.5) * 2.0
                weight = L / (0.20 + vertical_bias * z_align + diagonal_bias * max(0, diag_score))
                weight *= rng.uniform(0.95, 1.05)
                candidate.append((weight, i, j))
    candidate = sorted(candidate, key=lambda x: x[0])

    G = nx.Graph()
    G.add_nodes_from(range(n))
    edges = []

    for _, i, j in candidate:
        if len(edges) >= target_edges:
            break
        if not nx.has_path(G, i, j):
            G.add_edge(i, j)
            edges.append((i, j))

    existing = set(tuple(sorted(e)) for e in edges)
    for _, i, j in candidate:
        if len(edges) >= target_edges:
            break
        e = tuple(sorted((i, j)))
        if e not in existing:
            existing.add(e)
            edges.append(e)
    return edges


def select_lattice_edges(nodes, params):
    kind = params.get("_lattice_symmetry_kind", "stochastic")
    if kind in ["isotropic", "orthotropic"]:
        base_points = np.asarray(params.get("_lattice_base_points", nodes), dtype=float)
        base_edges = _select_edges_on_points(base_points, params)
        seam_idx = _symmetry_seam_indices(base_points, float(params["size_mm"]), kind)
        target_edges = min(int(params["strut_count"]), len(base_points) * (len(base_points) - 1) // 2)
        base_edges = _connect_components_via_seams(base_points, base_edges, seam_idx, target_edges=target_edges)

        groups = params.get("_lattice_symmetry_groups", [])
        transforms = params.get("_lattice_symmetry_transforms", [])
        full_edges = set()
        for i0, j0 in base_edges:
            gi = groups[i0]
            gj = groups[j0]
            for t in transforms:
                if t in gi and t in gj:
                    a, b = gi[t], gj[t]
                    if a != b:
                        full_edges.add(tuple(sorted((int(a), int(b)))))
        return sorted(full_edges)
    return _select_edges_on_points(nodes, params)

def _distance_sq_segment_to_grid(X, Y, Z, p0, p1):
    """Vectorized squared distance from all grid points to a 3D line segment.
    Supports NumPy or CuPy arrays depending on the current generation backend.
    """
    xp = cp.get_array_module(X) if (CUPY_AVAILABLE and cp is not None and hasattr(cp, "get_array_module")) else np
    p0 = xp.asarray(p0, dtype=xp.float32)
    p1 = xp.asarray(p1, dtype=xp.float32)
    ab = p1 - p0
    ab2 = xp.sum(ab * ab)
    if float(ab2) <= 1e-12:
        return (X - p0[0])**2 + (Y - p0[1])**2 + (Z - p0[2])**2
    t = ((X - p0[0]) * ab[0] + (Y - p0[1]) * ab[1] + (Z - p0[2]) * ab[2]) / ab2
    t = xp.clip(t, 0.0, 1.0)
    qx = p0[0] + t * ab[0]
    qy = p0[1] + t * ab[1]
    qz = p0[2] + t * ab[2]
    return (X - qx)**2 + (Y - qy)**2 + (Z - qz)**2


def build_lattice_mask(nodes, edges, radius, size_mm, grid_n=72, node_blend_factor=1.2, close_iters=1):
    """Build a voxelized union of cylinders + node spheres within the 8-mm box.

    This function is now backend-aware:
    - NumPy on CPU
    - CuPy on GPU when compute_backend allows it
    """
    xp = _xp_for_generation() if "_xp_for_generation" in globals() else np
    g = xp.linspace(0.0, size_mm, int(grid_n), dtype=xp.float32)
    X, Y, Z = xp.meshgrid(g, g, g, indexing="ij")
    mask = xp.zeros((grid_n, grid_n, grid_n), dtype=bool)

    r2 = xp.float32(radius * radius)
    node_r = xp.float32(max(radius * node_blend_factor, radius))
    node_r2 = xp.float32(node_r * node_r)

    for i, j in edges:
        d2 = _distance_sq_segment_to_grid(X, Y, Z, nodes[i], nodes[j])
        mask |= (d2 <= r2)

    for p in nodes:
        p = xp.asarray(p, dtype=xp.float32)
        d2 = (X - p[0])**2 + (Y - p[1])**2 + (Z - p[2])**2
        mask |= (d2 <= node_r2)

    if CUPY_AVAILABLE and cp is not None and isinstance(mask, cp.ndarray):
        mask = cp.asnumpy(mask)

    if close_iters > 0:
        mask = binary_closing(mask, iterations=int(close_iters))

    return np.asarray(mask, dtype=bool)


def _make_trimesh_cylinder_between(p0, p1, radius, sections=20):
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    if np.linalg.norm(p1 - p0) <= 1e-12:
        return None
    try:
        return trimesh.creation.cylinder(radius=float(radius), segment=np.vstack([p0, p1]), sections=int(sections))
    except Exception:
        return None


def _make_trimesh_node_sphere(center, radius, subdivisions=2):
    try:
        m = trimesh.creation.icosphere(subdivisions=int(subdivisions), radius=float(radius))
        m.apply_translation(np.asarray(center, dtype=float))
        return m
    except Exception:
        return None



def _make_cylinder_mesh_between(p0, p1, radius=0.2, sections=20):
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    v = p1 - p0
    L = float(np.linalg.norm(v))
    if L <= 1e-12:
        return np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.int32)
    ez = v / L
    tmp = np.array([1.0, 0.0, 0.0]) if abs(ez[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    ex = np.cross(ez, tmp)
    ex /= (np.linalg.norm(ex) + 1e-12)
    ey = np.cross(ez, ex)
    angles = np.linspace(0, 2*np.pi, int(sections), endpoint=False)
    circle = np.stack([np.cos(angles), np.sin(angles)], axis=1)

    ring0 = p0[None, :] + radius * (circle[:, 0:1] * ex[None, :] + circle[:, 1:2] * ey[None, :])
    ring1 = p1[None, :] + radius * (circle[:, 0:1] * ex[None, :] + circle[:, 1:2] * ey[None, :])
    verts = np.vstack([ring0, ring1, p0[None, :], p1[None, :]]).astype(np.float32)

    faces = []
    n = int(sections)
    for i in range(n):
        j = (i + 1) % n
        faces.append([i, j, n + j])
        faces.append([i, n + j, n + i])
    c0 = 2 * n
    c1 = 2 * n + 1
    for i in range(n):
        j = (i + 1) % n
        faces.append([c0, j, i])
        faces.append([c1, n + i, n + j])
    return verts, np.asarray(faces, dtype=np.int32)

def build_lattice_direct_mesh(nodes, edges, radius, size_mm, sections=20, node_blend_factor=1.0):
    """Build lattice as direct node+strut mesh with constant strut thickness.

    This intentionally uses only strut cylinders, not voxel masks, so periodic_isotropic
    and periodic_orthotropic boundaries remain line/strut based rather than voxel-like.
    """
    all_v = []
    all_f = []
    offset = 0
    for i, j in edges:
        v, f = _make_cylinder_mesh_between(nodes[i], nodes[j], radius=float(radius), sections=int(sections))
        if len(v) == 0 or len(f) == 0:
            continue
        all_v.append(v)
        all_f.append(f + offset)
        offset += len(v)
    if not all_v:
        return np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.int32)
    verts = np.vstack(all_v).astype(np.float32)
    faces = np.vstack(all_f).astype(np.int32)
    verts = np.clip(verts, 0.0, float(size_mm))
    return verts, faces



def calibrate_lattice_radius(nodes, edges, params):
    """Fast analytic radius calibration for direct strut lattices.

    This avoids repeated voxel-mask calibration loops and keeps final lattice generation
    node/strut based rather than voxel dominated.
    """
    size = float(params["size_mm"])
    target_vf = float(params.get("target_vf", TARGET_VF))
    min_radius = float(params.get("min_thickness_mm", 0.2)) / 2.0

    total_length = sum(np.linalg.norm(nodes[j] - nodes[i]) for i, j in edges)
    if total_length <= 1e-12:
        radius = min_radius
    else:
        radius = math.sqrt(max(target_vf, 1e-8) * (size ** 3) / (math.pi * total_length))
        radius = max(radius, min_radius)

    # light correction for likely overlap at nodes
    radius *= 0.97
    actual_vf = approximate_lattice_vf_from_edges(nodes, edges, radius, size_mm=size)
    return float(radius), float(actual_vf)



def generate_lattice_candidate(params):
    seed = int(params["seed"])
    nodes = generate_lattice_nodes(params, seed=seed)
    edges = select_lattice_edges(nodes, params)

    radius, actual_vf = calibrate_lattice_radius(nodes, edges, params)

    verts, faces = build_lattice_direct_mesh(
        nodes,
        edges,
        radius=float(radius),
        size_mm=float(params["size_mm"]),
        sections=int(params.get("cylinder_segments", CYLINDER_SEGMENTS)),
        node_blend_factor=float(params.get("node_blend_factor", 1.0)),
    )
    mesh_method = "direct_node_strut_mesh"

    G = nx.Graph()
    G.add_nodes_from(range(len(nodes)))
    G.add_edges_from(edges)
    lengths = [np.linalg.norm(nodes[j] - nodes[i]) for i, j in edges]

    mode = str(params.get("lattice_mode", "stochastic"))
    if mode == "periodic_isotropic":
        contact_ok = True
        contact_mismatch = {"symmetry_type": "1/8_reflection_xyz", "note": "outer x/y/z boundaries mirrored from one octant"}
    elif mode == "periodic_orthotropic":
        contact_ok = True
        contact_mismatch = {"symmetry_type": "1/4_reflection_xy", "note": "outer x/y boundaries mirrored; z boundaries independent"}
    else:
        contact_ok = None
        contact_mismatch = {}

    ncomp_graph = nx.number_connected_components(G) if len(nodes) else 0

    quick_desc = {
        "actual_vf_est": float(actual_vf),
        "node_count_actual": int(len(nodes)),
        "strut_count_actual": int(len(edges)),
        "strut_radius_mm": float(radius),
        "strut_diameter_mm": float(2 * radius),
        "mean_node_degree": float(np.mean([d for _, d in G.degree()])) if len(nodes) > 0 else np.nan,
        "strut_length_mean": float(np.mean(lengths)) if lengths else np.nan,
        "mesh_method": mesh_method,
        "lattice_grid_n": int(params.get("lattice_grid_n", LATTICE_GRID_N)),
        "node_blend_factor": float(params.get("node_blend_factor", LATTICE_NODE_BLEND_FACTOR)),
        "contact_face_symmetry_ok": bool(contact_ok) if contact_ok is not None else None,
        "contact_face_mismatch_voxels": json_dumps(contact_mismatch),
        "component_count_before_repair": int(ncomp_graph),
        "connected_components": int(ncomp_graph),
        "component_count_after_repair": int(ncomp_graph),
        "is_single_connected_component": bool(int(ncomp_graph) <= 1),
        "connectivity_repaired": False,
        "connectivity_mode": "graph_direct",
        "bridges_added": 0,
        "voxels_added_by_bridges": 0,
    }
    return verts, faces, quick_desc



# ----------------------------------------------------------------
# Stronger symmetry-constrained lattice helpers
# ----------------------------------------------------------------
def _isclosev(a, b, tol):
    return abs(float(a) - float(b)) <= float(tol)

def _round_key3(p, nd=6):
    p = np.asarray(p, dtype=float)
    return tuple(np.round(p, nd).tolist())

def _build_face_graph_from_uv(uv, k=2):
    uv = np.asarray(uv, dtype=float)
    n = len(uv)
    if n <= 1:
        return []
    D = squareform(pdist(uv))
    edges = set()
    kk = max(1, min(int(k), n - 1))
    for i in range(n):
        order = np.argsort(D[i])[1:1 + kk]
        for j in order:
            edges.add(tuple(sorted((int(i), int(j)))))
    return sorted(edges)

def _nearest_indices(src_pts, ref_pts):
    src = np.asarray(src_pts, dtype=float)
    ref = np.asarray(ref_pts, dtype=float)
    if len(src) == 0 or len(ref) == 0:
        return []
    D = ((src[:, None, :] - ref[None, :, :]) ** 2).sum(axis=2)
    return np.argmin(D, axis=1).tolist()

def _find_group_index(node_groups, pred):
    out = []
    for i, p in enumerate(node_groups):
        if pred(np.asarray(p, dtype=float)):
            out.append(i)
    return out

def _enforce_boundary_face_edge_constraints(base_points, base_edges, params):
    """
    Enforce exact boundary-face correspondence constraints on the BASE graph before reflection.

    isotropic:
      - x0, y0, z0 base faces must share the same 2D face graph
      - each face node also receives one deterministic connector to the corresponding seam plane
      - this guarantees that, after 1/8 reflection, all six outer faces have the same boundary node/strut pattern

    orthotropic:
      - x0 and y0 side faces share the same side-face graph
      - z0 and z1 share the same top/bottom graph
      - deterministic connectors attach side faces to x/y seam planes and top/bottom faces to the central spine
    """
    pts = np.asarray(base_points, dtype=float)
    size = float(params["size_mm"])
    tol = max(1e-6, 1e-4 * size)
    edge_set = set(tuple(sorted(map(int, e))) for e in base_edges)

    meta = params.get("_lattice_boundary_meta", {}) or {}
    kind = str(params.get("_lattice_symmetry_kind", "stochastic"))

    def add_edge(i, j):
        i, j = int(i), int(j)
        if i == j:
            return
        edge_set.add(tuple(sorted((i, j))))

    if kind == "isotropic":
        face_x0 = meta.get("x0", {}).get("indices", [])
        face_y0 = meta.get("y0", {}).get("indices", [])
        face_z0 = meta.get("z0", {}).get("indices", [])
        uv = np.asarray(meta.get("face_pattern_uv", []), dtype=float)
        face_graph = _build_face_graph_from_uv(uv, k=2)
        for a, b in face_graph:
            if a < len(face_x0) and b < len(face_x0): add_edge(face_x0[a], face_x0[b])
            if a < len(face_y0) and b < len(face_y0): add_edge(face_y0[a], face_y0[b])
            if a < len(face_z0) and b < len(face_z0): add_edge(face_z0[a], face_z0[b])

        seam_x = meta.get("xmid", [])
        seam_y = meta.get("ymid", [])
        seam_z = meta.get("zmid", [])
        for i, j_idx in zip(face_x0, _nearest_indices(pts[face_x0], pts[seam_x] if seam_x else pts[[np.argmin(np.abs(pts[:,0]-size/2))]])):
            if seam_x: add_edge(i, seam_x[j_idx])
        for i, j_idx in zip(face_y0, _nearest_indices(pts[face_y0], pts[seam_y] if seam_y else pts[[np.argmin(np.abs(pts[:,1]-size/2))]])):
            if seam_y: add_edge(i, seam_y[j_idx])
        for i, j_idx in zip(face_z0, _nearest_indices(pts[face_z0], pts[seam_z] if seam_z else pts[[np.argmin(np.abs(pts[:,2]-size/2))]])):
            if seam_z: add_edge(i, seam_z[j_idx])

        center_idx = meta.get("center_idx", None)
        if center_idx is not None:
            for seam_group in [seam_x, seam_y, seam_z]:
                for s in seam_group:
                    add_edge(center_idx, s)

    elif kind == "orthotropic":
        face_x0 = meta.get("x0", {}).get("indices", [])
        face_y0 = meta.get("y0", {}).get("indices", [])
        uv_side = np.asarray(meta.get("side_pattern_uv", []), dtype=float)
        side_graph = _build_face_graph_from_uv(uv_side, k=2)
        for a, b in side_graph:
            if a < len(face_x0) and b < len(face_x0): add_edge(face_x0[a], face_x0[b])
            if a < len(face_y0) and b < len(face_y0): add_edge(face_y0[a], face_y0[b])

        face_z0 = meta.get("z0", {}).get("indices", [])
        face_z1 = meta.get("z1", {}).get("indices", [])
        uv_tb = np.asarray(meta.get("topbottom_pattern_uv", []), dtype=float)
        tb_graph = _build_face_graph_from_uv(uv_tb, k=2)
        for a, b in tb_graph:
            if a < len(face_z0) and b < len(face_z0): add_edge(face_z0[a], face_z0[b])
            if a < len(face_z1) and b < len(face_z1): add_edge(face_z1[a], face_z1[b])

        seam_x = meta.get("xmid", [])
        seam_y = meta.get("ymid", [])
        spine = meta.get("spine", [])
        for i, j_idx in zip(face_x0, _nearest_indices(pts[face_x0], pts[seam_x] if seam_x else pts[[np.argmin(np.abs(pts[:,0]-size/2))]])):
            if seam_x: add_edge(i, seam_x[j_idx])
        for i, j_idx in zip(face_y0, _nearest_indices(pts[face_y0], pts[seam_y] if seam_y else pts[[np.argmin(np.abs(pts[:,1]-size/2))]])):
            if seam_y: add_edge(i, seam_y[j_idx])
        for group in [face_z0, face_z1]:
            for i, j_idx in zip(group, _nearest_indices(pts[group], pts[spine] if spine else pts[[np.argmin(np.sum((pts[:,:2]-size/2.0)**2, axis=1))]])):
                if spine: add_edge(i, spine[j_idx])

        if len(spine) >= 2:
            spine_order = sorted(spine, key=lambda idx: pts[idx, 2])
            for a, b in zip(spine_order[:-1], spine_order[1:]):
                add_edge(a, b)

    return sorted(edge_set)

def _prepare_isotropic_boundary_meta(base, size, uv_face):
    pts = np.asarray(base, dtype=float)
    tol = max(1e-6, 1e-4 * size)
    meta = {}
    meta["face_pattern_uv"] = np.asarray(uv_face, dtype=float).tolist()
    # because generation appends x0, y0, z0 in this exact repeated order
    n = len(uv_face)
    start = len(pts) - (3*n + 1 + 3*n)  # before seam points appended: face nodes begin before seams; not robust
    return meta


def generate_lattice_nodes(params, seed=0):
    rng = np.random.default_rng(seed)
    size = float(params["size_mm"])
    n_nodes = int(params["node_count"])
    mode = str(params["lattice_mode"])
    perturb = float(params.get("node_perturbation", 0.05))

    if mode == "periodic_isotropic":
        n_base = max(3, int(math.ceil(n_nodes / 8.0)))
        interior = rng.random((n_base, 3)) * (size / 2.0)
        n_face = max(3, int(round(n_base / 4)))
        uv = _face_pattern_points(rng, size / 2.0, n_face, margin_frac=0.08)

        face_x0 = np.asarray([[0.0, u, v] for u, v in uv], dtype=float)
        face_y0 = np.asarray([[u, 0.0, v] for u, v in uv], dtype=float)
        face_z0 = np.asarray([[u, v, 0.0] for u, v in uv], dtype=float)

        base_core = np.vstack([interior, face_x0, face_y0, face_z0])
        if perturb > 0:
            m = np.all(base_core > 1e-9, axis=1)
            base_core[m] += rng.normal(0, perturb * size * 0.2, base_core[m].shape)
            base_core = np.clip(base_core, 0.0, size / 2.0)

        base = _augment_base_points_for_fullmodel_connectivity(base_core, size, "isotropic", rng)
        # recover exact face-node groups from rounded coordinates
        pts = np.asarray(base, dtype=float)
        tol = max(1e-6, 1e-4 * size)
        face_x0_idx = [i for i,p in enumerate(pts) if _isclosev(p[0], 0.0, tol) and (p[1] > tol) and (p[2] > tol)]
        face_y0_idx = [i for i,p in enumerate(pts) if _isclosev(p[1], 0.0, tol) and (p[0] > tol) and (p[2] > tol)]
        face_z0_idx = [i for i,p in enumerate(pts) if _isclosev(p[2], 0.0, tol) and (p[0] > tol) and (p[1] > tol)]
        seam_x = [i for i,p in enumerate(pts) if _isclosev(p[0], size/2.0, tol)]
        seam_y = [i for i,p in enumerate(pts) if _isclosev(p[1], size/2.0, tol)]
        seam_z = [i for i,p in enumerate(pts) if _isclosev(p[2], size/2.0, tol)]
        center_idx = next((i for i,p in enumerate(pts) if _isclosev(p[0], size/2.0, tol) and _isclosev(p[1], size/2.0, tol) and _isclosev(p[2], size/2.0, tol)), None)

        nodes, groups, transforms = _reflect_points_isotropic(base, size)
        params["_lattice_symmetry_kind"] = "isotropic"
        params["_lattice_base_points"] = np.asarray(base, dtype=float)
        params["_lattice_symmetry_groups"] = groups
        params["_lattice_symmetry_transforms"] = transforms
        params["_lattice_boundary_meta"] = {
            "x0": {"indices": face_x0_idx},
            "y0": {"indices": face_y0_idx},
            "z0": {"indices": face_z0_idx},
            "face_pattern_uv": np.asarray(uv, dtype=float).tolist(),
            "xmid": seam_x,
            "ymid": seam_y,
            "zmid": seam_z,
            "center_idx": center_idx,
        }
        return nodes

    if mode == "periodic_orthotropic":
        n_base = max(4, int(math.ceil(n_nodes / 4.0)))
        interior = np.column_stack([
            rng.random(n_base) * (size / 2.0),
            rng.random(n_base) * (size / 2.0),
            rng.random(n_base) * size,
        ])
        n_side = max(3, int(round(n_base / 5)))
        uv_side = _face_pattern_points(rng, size, n_side, margin_frac=0.08)
        face_x0 = np.asarray([[0.0, 0.5*u, v] for u, v in uv_side], dtype=float)
        face_y0 = np.asarray([[0.5*u, 0.0, v] for u, v in uv_side], dtype=float)

        n_tb = max(3, int(round(n_base / 6)))
        uv_tb = _face_pattern_points(rng, size / 2.0, n_tb, margin_frac=0.08)
        face_z0 = np.asarray([[u, v, 0.0] for u, v in uv_tb], dtype=float)
        face_z1 = np.asarray([[u, v, size] for u, v in uv_tb], dtype=float)

        base_core = np.vstack([interior, face_x0, face_y0, face_z0, face_z1])
        if perturb > 0:
            m = np.all(base_core[:, :2] > 1e-9, axis=1)
            base_core[m] += rng.normal(0, perturb * size * 0.2, base_core[m].shape)
            base_core[:, 0] = np.clip(base_core[:, 0], 0.0, size / 2.0)
            base_core[:, 1] = np.clip(base_core[:, 1], 0.0, size / 2.0)
            base_core[:, 2] = np.clip(base_core[:, 2], 0.0, size)

        base = _augment_base_points_for_fullmodel_connectivity(base_core, size, "orthotropic", rng)
        pts = np.asarray(base, dtype=float)
        tol = max(1e-6, 1e-4 * size)
        face_x0_idx = [i for i,p in enumerate(pts) if _isclosev(p[0], 0.0, tol) and (p[1] > tol) and (p[2] > tol) and (p[2] < size-tol)]
        face_y0_idx = [i for i,p in enumerate(pts) if _isclosev(p[1], 0.0, tol) and (p[0] > tol) and (p[2] > tol) and (p[2] < size-tol)]
        face_z0_idx = [i for i,p in enumerate(pts) if _isclosev(p[2], 0.0, tol) and (p[0] > tol) and (p[1] > tol)]
        face_z1_idx = [i for i,p in enumerate(pts) if _isclosev(p[2], size, tol) and (p[0] > tol) and (p[1] > tol)]
        seam_x = [i for i,p in enumerate(pts) if _isclosev(p[0], size/2.0, tol)]
        seam_y = [i for i,p in enumerate(pts) if _isclosev(p[1], size/2.0, tol)]
        spine = [i for i,p in enumerate(pts) if _isclosev(p[0], size/2.0, tol) and _isclosev(p[1], size/2.0, tol)]

        nodes, groups, transforms = _reflect_points_orthotropic(base, size)
        params["_lattice_symmetry_kind"] = "orthotropic"
        params["_lattice_base_points"] = np.asarray(base, dtype=float)
        params["_lattice_symmetry_groups"] = groups
        params["_lattice_symmetry_transforms"] = transforms
        params["_lattice_boundary_meta"] = {
            "x0": {"indices": face_x0_idx},
            "y0": {"indices": face_y0_idx},
            "z0": {"indices": face_z0_idx},
            "z1": {"indices": face_z1_idx},
            "side_pattern_uv": np.asarray(uv_side, dtype=float).tolist(),
            "topbottom_pattern_uv": np.asarray(uv_tb, dtype=float).tolist(),
            "xmid": seam_x,
            "ymid": seam_y,
            "spine": spine,
        }
        return nodes

    pts = rng.random((n_nodes, 3)) * size
    if perturb > 0:
        pts += rng.normal(0, perturb * size, pts.shape)
        pts = np.clip(pts, 0, size)
    df = pd.DataFrame(np.round(np.asarray(pts, dtype=float), 6), columns=["x", "y", "z"]).drop_duplicates()
    params["_lattice_symmetry_kind"] = "stochastic"
    params["_lattice_base_points"] = df[["x", "y", "z"]].values.astype(float)
    params["_lattice_boundary_meta"] = {}
    return df[["x", "y", "z"]].values.astype(float)

def select_lattice_edges(nodes, params):
    kind = params.get("_lattice_symmetry_kind", "stochastic")
    if kind in ["isotropic", "orthotropic"]:
        base_points = np.asarray(params.get("_lattice_base_points", nodes), dtype=float)
        base_edges = _select_edges_on_points(base_points, params)
        base_edges = _enforce_boundary_face_edge_constraints(base_points, base_edges, params)
        seam_idx = _symmetry_seam_indices(base_points, float(params["size_mm"]), kind)
        target_edges = min(int(params["strut_count"]), len(base_points) * (len(base_points) - 1) // 2)
        base_edges = _connect_components_via_seams(base_points, base_edges, seam_idx, target_edges=target_edges)

        groups = params.get("_lattice_symmetry_groups", [])
        transforms = params.get("_lattice_symmetry_transforms", [])
        full_edges = set()
        for i0, j0 in base_edges:
            gi = groups[i0]
            gj = groups[j0]
            for t in transforms:
                if t in gi and t in gj:
                    a, b = gi[t], gj[t]
                    if a != b:
                        full_edges.add(tuple(sorted((int(a), int(b)))))
        return sorted(full_edges)
    return _select_edges_on_points(nodes, params)


In [ ]:

# ============================================================
# Cell 4. TPMS / SDF generator with basic and advanced non-overlapping combinations
# ============================================================

from scipy.ndimage import binary_erosion
import threading
_GPU_CONTEXT = threading.local()

def _set_generation_gpu_device(device_id):
    _GPU_CONTEXT.device_id = int(device_id)

def _get_generation_gpu_device():
    return int(getattr(_GPU_CONTEXT, "device_id", GPU_DEVICE_ID))

def _xp_for_generation():
    if USE_GPU_FOR_GENERATION and CUPY_AVAILABLE:
        try:
            cp.cuda.Device(_get_generation_gpu_device()).use()
        except Exception:
            pass
        return cp
    return np


def _to_numpy_array(a):
    if CUPY_AVAILABLE and cp is not None and isinstance(a, cp.ndarray):
        return cp.asnumpy(a)
    return np.asarray(a)


def make_tpms_grid(n=96, size_mm=8.0, frequency=2.0, anisotropy=(1.0, 1.0, 1.0)):
    xp = _xp_for_generation()
    x = xp.linspace(0, 2*xp.pi*frequency*anisotropy[0], n)
    y = xp.linspace(0, 2*xp.pi*frequency*anisotropy[1], n)
    z = xp.linspace(0, 2*xp.pi*frequency*anisotropy[2], n)
    return xp.meshgrid(x, y, z, indexing="ij")


def tpms_field(tpms_type, X, Y, Z):
    xp = cp.get_array_module(X) if (CUPY_AVAILABLE and cp is not None) else np
    t = tpms_type.lower()
    if t == "gyroid":
        F = xp.sin(X)*xp.cos(Y) + xp.sin(Y)*xp.cos(Z) + xp.sin(Z)*xp.cos(X)
    elif t == "primitive":
        F = xp.cos(X) + xp.cos(Y) + xp.cos(Z)
    elif t == "diamond":
        F = (xp.sin(X)*xp.sin(Y)*xp.sin(Z) + xp.sin(X)*xp.cos(Y)*xp.cos(Z)
             + xp.cos(X)*xp.sin(Y)*xp.cos(Z) + xp.cos(X)*xp.cos(Y)*xp.sin(Z))
    elif t == "iwp":
        F = 2*(xp.cos(X)*xp.cos(Y) + xp.cos(Y)*xp.cos(Z) + xp.cos(Z)*xp.cos(X)) - (xp.cos(2*X) + xp.cos(2*Y) + xp.cos(2*Z))
    elif t == "neovius":
        F = 3*(xp.cos(X) + xp.cos(Y) + xp.cos(Z)) + 4*xp.cos(X)*xp.cos(Y)*xp.cos(Z)
    elif t == "lidinoid":
        F = (0.5*(xp.sin(2*X)*xp.cos(Y)*xp.sin(Z) + xp.sin(2*Y)*xp.cos(Z)*xp.sin(X) + xp.sin(2*Z)*xp.cos(X)*xp.sin(Y))
             - 0.5*(xp.cos(2*X)*xp.cos(2*Y) + xp.cos(2*Y)*xp.cos(2*Z) + xp.cos(2*Z)*xp.cos(2*X)) + 0.15)
    elif t == "split_p":
        F = 1.1*(xp.cos(X) + xp.cos(Y) + xp.cos(Z)) + 0.25*(xp.cos(2*X) + xp.cos(2*Y) + xp.cos(2*Z))
    elif t == "double_gyroid":
        G = xp.sin(X)*xp.cos(Y) + xp.sin(Y)*xp.cos(Z) + xp.sin(Z)*xp.cos(X)
        F = G**2 - 0.35
    elif t == "frd":
        F = 4*xp.cos(X)*xp.cos(Y)*xp.cos(Z) - (xp.cos(2*X)*xp.cos(2*Y) + xp.cos(2*Y)*xp.cos(2*Z) + xp.cos(2*Z)*xp.cos(2*X))
    elif t == "fischer_koch_s":
        F = (xp.cos(2*X)*xp.sin(Y)*xp.cos(Z) + xp.cos(X)*xp.cos(2*Y)*xp.sin(Z) + xp.sin(X)*xp.cos(Y)*xp.cos(2*Z))
    elif t == "fischer_koch_c":
        F = (xp.cos(2*X)*xp.cos(Y)*xp.sin(Z) + xp.sin(X)*xp.cos(2*Y)*xp.cos(Z) + xp.cos(X)*xp.sin(Y)*xp.cos(2*Z))
    elif t == "pw_hybrid":
        Fp = xp.cos(X) + xp.cos(Y) + xp.cos(Z)
        Fg = xp.sin(X)*xp.cos(Y) + xp.sin(Y)*xp.cos(Z) + xp.sin(Z)*xp.cos(X)
        F = 0.55*Fp + 0.45*Fg
    elif t == "srs_like":
        F = xp.cos(X)*xp.sin(Y) + xp.cos(Y)*xp.sin(Z) + xp.cos(Z)*xp.sin(X)
    elif t == "karcher_k_like":
        F = xp.cos(X)*xp.cos(Y) + xp.cos(Y)*xp.cos(Z) + xp.cos(Z)*xp.cos(X) - xp.sin(X)*xp.sin(Y)*xp.sin(Z)
    else:
        raise ValueError(f"Unknown TPMS type: {tpms_type}")
    # Normalize for robust quantile/offset handling.
    F = (F - xp.mean(F)) / (xp.std(F) + 1e-12)
    return F


def enforce_min_hole_size(mask, min_hole_size_vox=1):
    """Remove pores smaller than the requested voxel-scale hole size.

    For DLP printing, tiny isolated holes often close due to over-curing.
    This operation fills/removes small pore features instead of keeping
    unprintable holes in the design.
    """
    min_hole_size_vox = int(min_hole_size_vox)
    if min_hole_size_vox <= 1:
        return mask
    pores = ~np.asarray(mask, dtype=bool)
    pores = binary_opening(pores, iterations=max(1, min_hole_size_vox - 1))
    return ~pores

# ------------------------------------------------------------
# Multi-wall 조합 템플릿
# ------------------------------------------------------------
MULTIWALL_MODE_TEMPLATES = {
    "A+B":              ["solid_A", "solid_B"],
    "Wall+A":           ["wall", "solid_A"],
    "Wall+B":           ["wall", "solid_B"],
    "Double-wall":      ["wall", "wall"],
    "Double-wall+A":    ["wall", "wall", "solid_A"],
    "Double-wall+B":    ["wall", "wall", "solid_B"],
    "Double-wall+A+B":  ["wall", "wall", "solid_A", "solid_B"],
}

MIN_COMPONENT_VF = 0.20
MAX_ISLANDS_BEFORE_SKIP_REPAIR = 400


# ------------------------------------------------------------
# 두께(mm) 기반 mask 생성
# ------------------------------------------------------------
# ------------------------------------------------------------
# 목표 VF를 정확히 맞추는 mask 생성 (bisection 역산)
#   두께를 랜덤으로 주고 사후 보정하는 대신,
#   각 TPMS 함수에서 목표 VF가 나오는 band 폭을 이진탐색으로 찾음.
# ------------------------------------------------------------
def _raw_mask_from_band(F, mode, half_band, level_shift=0.0):
    if mode == "wall":
        return np.abs(F - level_shift) <= half_band
    elif mode == "solid_A":
        return (F >= level_shift) & (F <= level_shift + 2 * half_band)
    elif mode == "solid_B":
        return (F <= level_shift) & (F >= level_shift - 2 * half_band)
    else:
        raise ValueError(f"Unknown TPMS mode: {mode}")

def make_mask_for_target_vf(F, mode, target_vf, occupied=None, gap_voxels=0,
                            level_shift=0.0, tol=0.005, max_iter=40):
    """목표 VF에 맞는 mask를 bisection으로 생성.
    반환: (mask, 달성 VF, 사용된 half_band)"""
    forbidden = None
    if occupied is not None and np.any(occupied):
        forbidden = occupied.copy()
        if gap_voxels > 0:
            forbidden = binary_dilation(forbidden, iterations=int(gap_voxels))

    total = F.size

    def vf_at(hb):
        m = _raw_mask_from_band(F, mode, hb, level_shift)
        if forbidden is not None:
            m = m & ~forbidden
        return float(m.sum()) / total, m

    # band 폭 탐색 범위: F는 정규화되어 있으므로 0 ~ (F 범위)로 충분
    lo, hi = 0.0, float(np.abs(F - level_shift).max()) + 1e-6
    best_m = None; best_vf = 0.0; best_hb = hi
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        vf, m = vf_at(mid)
        best_m, best_vf, best_hb = m, vf, mid
        if abs(vf - target_vf) <= tol:
            break
        if vf < target_vf:
            lo = mid   # 더 두껍게
        else:
            hi = mid   # 더 얇게
    return best_m, best_vf, best_hb

def make_tpms_component_mask(F, mode, thickness_mm, voxel_size_mm, occupied=None, gap_voxels=0, level_shift=0.0):
    gx, gy, gz = np.gradient(F, voxel_size_mm)
    grad_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    mean_grad = max(float(np.mean(grad_mag)), 1e-6)
    half_band = (thickness_mm / 2.0) * mean_grad

    if mode == "wall":
        raw_mask = np.abs(F - level_shift) <= half_band
    elif mode == "solid_A":
        raw_mask = (F >= level_shift) & (F <= level_shift + 2 * half_band)
    elif mode == "solid_B":
        raw_mask = (F <= level_shift) & (F >= level_shift - 2 * half_band)
    else:
        raise ValueError(f"Unknown TPMS mode: {mode}")

    mask = raw_mask.copy()
    if occupied is not None and np.any(occupied):
        forbidden = occupied.copy()
        if gap_voxels > 0:
            forbidden = binary_dilation(forbidden, iterations=int(gap_voxels))
        mask &= ~forbidden
    return mask


# ------------------------------------------------------------
# closed-cell 자동 복구
# ------------------------------------------------------------
def repair_closed_cells(mask):
    void = ~np.asarray(mask, dtype=bool)
    lab_v, n = label(void)
    if n == 0:
        return mask
    boundary_ids = set()
    for face in [lab_v[0, :, :], lab_v[-1, :, :], lab_v[:, 0, :], lab_v[:, -1, :], lab_v[:, :, 0], lab_v[:, :, -1]]:
        boundary_ids.update(np.unique(face).tolist())
    boundary_ids.discard(0)
    trapped_ids = [i for i in range(1, n + 1) if i not in boundary_ids]
    if not trapped_ids:
        return mask
    trapped_mask = np.isin(lab_v, trapped_ids)
    return mask | trapped_mask


# ------------------------------------------------------------
# 두께 편차 / 프린팅 가능성 체크
# ------------------------------------------------------------
def check_thickness_consistency(comps, max_deviation_ratio=1.5):
    thicknesses = [c["thickness_mm"] for c in comps]
    t_min, t_max = min(thicknesses), max(thicknesses)
    if t_min <= 0:
        return False
    return (t_max - t_min) / t_min <= max_deviation_ratio

def check_open_cell(mask):
    void = ~np.asarray(mask, dtype=bool)
    lab_v, n = label(void)
    if n == 0:
        return True
    for comp_id in range(1, n + 1):
        comp_mask = (lab_v == comp_id)
        touches = (
            comp_mask[0, :, :].any() or comp_mask[-1, :, :].any() or
            comp_mask[:, 0, :].any() or comp_mask[:, -1, :].any() or
            comp_mask[:, :, 0].any() or comp_mask[:, :, -1].any()
        )
        if not touches:
            return False
    return True

def check_no_island(mask):
    return count_connected_components(mask) <= 1

def check_printability(mask, comps, target_vf, vf_range=(0.10, 0.80), max_thickness_deviation=1.5):
    reasons = []
    if not (vf_range[0] <= target_vf <= vf_range[1]):
        reasons.append("vf_out_of_range")
    if not check_thickness_consistency(comps, max_thickness_deviation):
        reasons.append("thickness_deviation_too_large")
    if not check_no_island(mask):
        reasons.append("island_detected")
    # closed-cell 검사 비활성화 (VF 정확성 우선 정책). 닫힌 셀 허용.
    # if not check_open_cell(mask):
    #     reasons.append("closed_cell_detected")
    return (len(reasons) == 0), reasons


# ------------------------------------------------------------
# 전체 mask 조립 (erosion 방식 VF 보정)
# ------------------------------------------------------------
def generate_tpms_mask_from_components(params):
    seed = int(params["seed"])
    rng = np.random.default_rng(seed)
    n = int(params.get("grid_n", TPMS_GRID_N))
    size_mm = params.get("size_mm", SIZE_MM)
    frequency = float(params.get("frequency", 2.0))
    anisotropy = tuple(params.get("anisotropy_xyz", [1.0, 1.0, 1.0]))
    noise_amp = float(params.get("noise_amp", 0.0))
    gap_mm = float(params.get("component_gap_mm", 0.0))
    voxel_size_mm = size_mm / n
    gap_voxels = int(round(gap_mm / voxel_size_mm))
    components = params["components"]

    X, Y, Z = make_tpms_grid(n=n, size_mm=size_mm, frequency=frequency, anisotropy=anisotropy)
    occupied = np.zeros((n, n, n), dtype=bool)
    target_vf = float(params.get("target_vf", TARGET_VF))

    # 목표 VF를 component 개수로 나눠 배분 (겹침 고려해 누적 목표로 접근)
    n_comp = len(components)
    for ci, comp in enumerate(components):
        F = _to_numpy_array(tpms_field(comp["tpms_type"], X, Y, Z))
        if noise_amp > 0:
            F = F + noise_amp * rng.normal(size=F.shape)
            F = (F - np.mean(F)) / (np.std(F) + 1e-12)

        # 이번 component까지 채워야 할 누적 목표 VF
        cumulative_target = target_vf * (ci + 1) / n_comp
        current_vf = estimate_mask_vf(occupied)
        # 이번 component가 추가로 채워야 하는 양
        remaining = max(cumulative_target - current_vf, 0.0)
        if remaining <= 0:
            continue

        m, achieved, hb = make_mask_for_target_vf(
            F,
            mode=comp["mode"],
            target_vf=remaining,
            occupied=occupied,
            gap_voxels=gap_voxels,
            level_shift=float(comp.get("level_shift", 0.0)),
            tol=0.005,
        )
        occupied |= m

    # 미세 구멍 제거
    min_hole_size_vox = int(params.get("min_hole_size_vox", 1))
    if min_hole_size_vox > 1:
        occupied = enforce_min_hole_size(occupied, min_hole_size_vox=min_hole_size_vox)

    # 최종 VF 미세 보정 (bisection이 이미 맞췄지만, 구멍제거 등으로 약간 어긋난 경우만)
    actual = estimate_mask_vf(occupied)
    if actual > target_vf + VF_TOLERANCE:
        cur = occupied.copy()
        n_islands_ref = count_connected_components(cur)
        for _ in range(4):
            if estimate_mask_vf(cur) <= target_vf + VF_TOLERANCE:
                break
            eroded = binary_erosion(cur, iterations=1)
            if not np.any(eroded):
                break
            if count_connected_components(eroded) > n_islands_ref:
                break
            cur = eroded
        occupied = cur
    elif actual < target_vf - VF_TOLERANCE:
        cur = occupied.copy()
        for _ in range(4):
            cur_vf = estimate_mask_vf(cur)
            if cur_vf >= target_vf - VF_TOLERANCE:
                break
            deficit = target_vf - cur_vf
            add_k = int(round(deficit * cur.size))
            dilated = binary_dilation(cur, iterations=1)
            cands = np.flatnonzero((dilated & ~cur).ravel())
            if len(cands) == 0:
                break
            add_k = min(add_k, len(cands))
            add_idx = rng.choice(cands, size=add_k, replace=False)
            flat = cur.ravel()
            flat[add_idx] = True
            cur = flat.reshape(cur.shape)
        occupied = cur

    return occupied

# ------------------------------------------------------------
# 최종 candidate 생성 (island 안전장치 + RuntimeError로 Cell 7 호환)
# ------------------------------------------------------------
def generate_tpms_candidate(params):
    mask = generate_tpms_mask_from_components(params)
    n_islands_before = count_connected_components(mask)

    if n_islands_before > MAX_ISLANDS_BEFORE_SKIP_REPAIR:
        raise RuntimeError(f"not_printable: too_many_islands_skipped_repair (islands={n_islands_before})")

    mask, repair_info = repair_mask_connectivity(
        mask, mode="bridge", bridge_radius_vox=3, min_component_voxels=1, max_bridges=500,
    )
    # closed-cell 복구 비활성화: neovius 등에서 닫힌 공극을 채우면 VF가 목표를 크게 초과함.
    # VF 정확성(불변 조건)을 위해 복구를 생략하고, 닫힌 셀은 허용한다.

    ok, reasons = check_printability(mask, params["components"], params["target_vf"])
    if not ok:
        raise RuntimeError(f"not_printable: {reasons}")

    verts, faces = mesh_from_binary_mask(mask, size_mm=params.get("size_mm", SIZE_MM))
    quick_desc = {
        "actual_vf_est": estimate_mask_vf(mask),
        "voxel_grid_n": int(mask.shape[0]),
        "component_count": len(params["components"]),
        "component_modes": "+".join([c["mode"] for c in params["components"]]),
        "component_tpms_types": "+".join([c["tpms_type"] for c in params["components"]]),
        "component_thicknesses_mm": json_dumps([round(c["thickness_mm"], 3) for c in params["components"]]),
        "multiwall_combo": params.get("multiwall_combo", ""),
        "min_hole_size_vox": int(params.get("min_hole_size_vox", 1)),
        "compute_backend": DEVICE,
        "islands_before_repair": n_islands_before,
    }
    return verts, faces, quick_desc



In [ ]:

# ============================================================
# Cell 5. Voxel morphogenesis generator
# ============================================================

def periodic_fourier_field(n, num_terms=16, anisotropy=(1.0, 1.0, 1.0), seed=0):
    rng = np.random.default_rng(seed)
    xp = _xp_for_generation() if "_xp_for_generation" in globals() else np
    x = xp.linspace(0, 2*xp.pi, n, endpoint=False)
    X, Y, Z = xp.meshgrid(x, x, x, indexing="ij")
    F = xp.zeros((n, n, n), dtype=xp.float32)
    for _ in range(num_terms):
        kx = rng.integers(1, 5)
        ky = rng.integers(1, 5)
        kz = rng.integers(1, 5)
        phase = rng.uniform(0, 2*np.pi)
        amp = rng.normal(0, 1) / math.sqrt(num_terms)
        F += amp * xp.sin(kx*anisotropy[0]*X + ky*anisotropy[1]*Y + kz*anisotropy[2]*Z + phase)
    F = (F - xp.mean(F)) / (xp.std(F) + 1e-12)
    if CUPY_AVAILABLE and cp is not None and isinstance(F, cp.ndarray):
        F = cp.asnumpy(F)
    return np.asarray(F, dtype=np.float32)


def stochastic_gaussian_field(n, sigma=(3.0, 3.0, 3.0), seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(size=(n, n, n))
    F = gaussian_filter(noise, sigma=sigma, mode="reflect")
    F = (F - np.mean(F)) / (np.std(F) + 1e-12)
    return F


def enforce_periodic_faces(mask, mode):
    """Backward-compatible wrapper.

    periodic_isotropic now means all six contact faces are identical.
    periodic_orthotropic now means four side faces are identical and the
    top/bottom faces are identical.
    """
    return enforce_contact_face_symmetry(
        mask,
        mode,
        depth_vox=int(globals().get("CONTACT_SURFACE_DEPTH_VOX", 1)),
    )


def limit_max_thickness_approx(mask, max_thickness_vox):
    """Approximate maximum local thickness control by eroding extremely thick cores."""
    if max_thickness_vox is None or max_thickness_vox <= 0:
        return mask
    dist = distance_transform_edt(mask)
    too_thick = dist > float(max_thickness_vox)
    if np.any(too_thick):
        mask = mask.copy()
        # Remove only part of very thick core to avoid destroying connectivity.
        mask[too_thick] = False
        mask = binary_closing(mask, iterations=1)
    return mask


def enforce_voxel_min_hole_size(mask, min_hole_size_vox=1):
    min_hole_size_vox = int(min_hole_size_vox)
    if min_hole_size_vox <= 1:
        return mask
    pores = ~np.asarray(mask, dtype=bool)
    pores = binary_opening(pores, iterations=max(1, min_hole_size_vox - 1))
    return ~pores


def generate_voxel_mask(params):
    seed = int(params["seed"])
    n = int(params.get("grid_n", VOXEL_GRID_N))
    mode = params["voxel_mode"]
    anisotropy_z = float(params.get("anisotropy_z", 1.0))

    if mode in ["periodic_isotropic", "periodic_orthotropic"]:
        anisotropy = (1.0, 1.0, anisotropy_z)
        F = periodic_fourier_field(
            n=n,
            num_terms=int(params.get("num_fourier_terms", 16)),
            anisotropy=anisotropy,
            seed=seed,
        )
    else:
        sigma_base = float(params.get("sigma", 4.0))
        sigma = (sigma_base, sigma_base, sigma_base * anisotropy_z)
        F = stochastic_gaussian_field(n=n, sigma=sigma, seed=seed)

    # Build isotropic/orthotropic behavior at the scalar-field stage.
    # This is the important change: the full 3D field is symmetrized before
    # thresholding, not merely corrected on the outer faces afterward.
    if mode in ["periodic_isotropic", "periodic_orthotropic"] and bool(params.get("strict_global_symmetry", STRICT_GLOBAL_SYMMETRY)):
        F = symmetrize_scalar_field(F, mode)

    mask = enforce_vf_by_rank(F, target_vf=TARGET_VF, prefer_high=True)

    closing_iter = int(params.get("closing_iter", 0))
    opening_iter = int(params.get("opening_iter", 0))
    if closing_iter > 0:
        mask = binary_closing(mask, iterations=closing_iter)
    if opening_iter > 0:
        if mode in ["periodic_isotropic", "periodic_orthotropic"] and bool(params.get("strict_global_symmetry", STRICT_GLOBAL_SYMMETRY)):
            # Opening can delete thin symmetric connections completely. Skip it
            # for strict periodic masks; connectivity repair handles artifacts.
            pass
        else:
            mask = binary_opening(mask, iterations=opening_iter)

    # Approximate minimum solid feature size by closing/opening.
    min_thick = int(params.get("min_thickness_vox", 1))
    if min_thick > 1:
        mask = binary_closing(mask, iterations=min_thick - 1)
        if mode in ["periodic_isotropic", "periodic_orthotropic"] and bool(params.get("strict_global_symmetry", STRICT_GLOBAL_SYMMETRY)):
            # Do not open strict symmetric masks; it can remove the whole
            # thin connected skeleton in low-resolution screening mode.
            pass
        else:
            mask = binary_opening(mask, iterations=max(0, min_thick - 2))

    # Approximate minimum printable pore/hole size.
    # Small pores are filled because they are likely to close during DLP over-curing.
    min_hole = int(params.get("min_hole_size_vox", 1))
    if min_hole > 1:
        mask = enforce_voxel_min_hole_size(mask, min_hole_size_vox=min_hole)

    mask, conn_info = finalize_lattice_voxel_mask(
        mask,
        params,
        generator_type="voxel",
        mode_key="voxel_mode",
        max_iter=6,
    )

    # For strict periodic modes, max-thickness trimming can delete the symmetric
    # hub/bridges and may even return an empty mask. Skip it and prioritize
    # symmetry + one-piece connectivity.
    if mode in ["periodic_isotropic", "periodic_orthotropic"] and bool(params.get("strict_global_symmetry", STRICT_GLOBAL_SYMMETRY)):
        conn_info["max_thickness_trim_skipped_to_preserve_symmetry_connectivity"] = True
    else:
        mask = limit_max_thickness_approx(mask, int(params.get("max_thickness_vox", 0)))
    mask, conn_info_2 = finalize_lattice_voxel_mask(
        mask,
        params,
        generator_type="voxel",
        mode_key="voxel_mode",
        max_iter=6,
    )
    conn_info.update({f"after_thickness_{k}": v for k, v in conn_info_2.items()})

    # Do not re-threshold strict periodic masks after bridges are made; that was
    # one cause of disconnected exports. We record the VF deviation instead.
    if abs(estimate_mask_vf(mask) - TARGET_VF) > VF_TOLERANCE:
        if mode in ["periodic_isotropic", "periodic_orthotropic"] and bool(params.get("strict_global_symmetry", STRICT_GLOBAL_SYMMETRY)):
            conn_info["vf_correction_skipped_to_preserve_symmetry_connectivity"] = True
            conn_info["vf_after_strict_symmetry"] = float(estimate_mask_vf(mask))
        else:
            mask = enforce_vf_by_rank(F, target_vf=TARGET_VF, prefer_high=True)
            mask, conn_info_3 = finalize_lattice_voxel_mask(
                mask,
                params,
                generator_type="voxel",
                mode_key="voxel_mode",
                max_iter=6,
            )
            conn_info.update({f"after_vf_{k}": v for k, v in conn_info_3.items()})

    params["_last_connectivity_info"] = conn_info
    return mask


def generate_voxel_candidate(params):
    mask = generate_voxel_mask(params)
    conn_info = params.get("_last_connectivity_info", {})
    verts, faces = mesh_from_binary_mask(mask, size_mm=params.get("size_mm", SIZE_MM))
    lab, ncomp = label(mask)
    contact_ok, contact_mismatch = contact_face_symmetry_report(mask, params.get("voxel_mode", "stochastic"), depth_vox=int(params.get("contact_surface_depth_vox", CONTACT_SURFACE_DEPTH_VOX)))
    quick_desc = {
        "actual_vf_est": estimate_mask_vf(mask),
        "voxel_grid_n": int(mask.shape[0]),
        "connected_components": int(ncomp),
        "slice_vf_mean_z": float(np.mean(mask.mean(axis=(0, 1)))),
        "slice_vf_std_z": float(np.std(mask.mean(axis=(0, 1)))),
        "min_thickness_vox": int(params.get("min_thickness_vox", 1)),
        "min_hole_size_vox": int(params.get("min_hole_size_vox", 1)),
        "compute_backend": DEVICE,
        "contact_face_symmetry_ok": bool(contact_ok) if contact_ok is not None else None,
        "contact_face_mismatch_voxels": json_dumps(contact_mismatch),
        "component_count_before_repair": int(conn_info.get("component_count_before", int(ncomp))),
        "component_count_after_repair": int(ncomp),
        "is_single_connected_component": bool(int(ncomp) <= 1),
        "connectivity_repaired": bool(int(ncomp) <= 1),
        "connectivity_mode": str(conn_info.get("connectivity_mode", "bridge")),
        "bridges_added": int(conn_info.get("bridges_added", 0)) + int(conn_info.get("after_thickness_bridges_added", 0)) + int(conn_info.get("after_vf_bridges_added", 0)) + int(conn_info.get("final_bridges_added", 0)),
        "voxels_added_by_bridges": int(conn_info.get("voxels_added_by_bridges", 0)) + int(conn_info.get("after_thickness_voxels_added_by_bridges", 0)) + int(conn_info.get("after_vf_voxels_added_by_bridges", 0)) + int(conn_info.get("final_voxels_added_by_bridges", 0)),
    }
    return verts, faces, quick_desc


In [ ]:

# ============================================================
# Cell 6. Candidate table builder
# ============================================================

def make_lattice_params(lattice_mode, cfg, idx, rng):
    p = {
        "generator_type": "lattice",
        "lattice_mode": lattice_mode,
        "size_mm": SIZE_MM,
        "target_vf": sample_uniform(rng, [0.45, 0.55]),
        "stl_mesh_size_mm": STL_MESH_SIZE_MM,
        "cylinder_segments": CYLINDER_SEGMENTS,
        "enforce_contact_face_symmetry": bool(USER_STRUCTURE_CONFIG["lattice"].get("enforce_contact_face_symmetry", ENFORCE_CONTACT_FACE_SYMMETRY)),
        "contact_surface_depth_vox": int(USER_STRUCTURE_CONFIG["lattice"].get("contact_surface_depth_vox", CONTACT_SURFACE_DEPTH_VOX)),
        "node_count": sample_int(rng, cfg["node_count_range"]),
        "strut_count": sample_int(rng, cfg["strut_count_range"]),
        "min_thickness_mm": sample_uniform(rng, cfg["min_thickness_mm_range"]),
        "connectivity_k": sample_int(rng, cfg["connectivity_k_range"]),
        "vertical_bias": sample_uniform(rng, cfg["vertical_bias_range"]),
        "diagonal_bias": sample_uniform(rng, cfg["diagonal_bias_range"]),
        "node_perturbation": sample_uniform(rng, cfg["node_perturbation_range"]),
        "seed": int(rng.integers(0, 2**31 - 1)),
        "lattice_grid_n": LATTICE_GRID_N,
        "node_blend_factor": LATTICE_NODE_BLEND_FACTOR,
        "lattice_binary_closing_iters": LATTICE_BINARY_CLOSING_ITERS,
        "lattice_radius_search_iters": LATTICE_RADIUS_SEARCH_ITERS,
        "force_connected": FORCE_CONNECTED_LATTICE_VOXEL,
        "connectivity_repair_mode": CONNECTIVITY_REPAIR_MODE,
        "connectivity_bridge_radius_vox": CONNECTIVITY_BRIDGE_RADIUS_VOX,
        "connectivity_min_component_voxels": CONNECTIVITY_MIN_COMPONENT_VOXELS,
        "connectivity_max_bridges": CONNECTIVITY_MAX_BRIDGES,
        "connectivity_retry_after_contact_symmetry": CONNECTIVITY_RETRY_AFTER_CONTACT_SYMMETRY,
    }
    if lattice_mode == "periodic_orthotropic":
        p["orthotropic_z_scale"] = sample_uniform(rng, cfg.get("orthotropic_z_scale_range", [1.0, 1.0]))
    p["candidate_id"] = f"LAT_{lattice_mode}_{idx:06d}"
    return p



def make_tpms_basic_params(tpms_type, mode, cfg, idx, rng):
    comp = random_tpms_component(rng, tpms_type=tpms_type, mode=mode, cfg=cfg)
    p = {
        "generator_type": "tpms",
        "tpms_stage": "basic",
        "size_mm": SIZE_MM,
        "target_vf": sample_uniform(rng, [0.45, 0.55]),
        "stl_mesh_size_mm": STL_MESH_SIZE_MM,
        "grid_n": TPMS_GRID_N,
        "frequency": sample_uniform(rng, cfg["frequency_range"]),
        "anisotropy_xyz": [
            sample_uniform(rng, cfg["anisotropy_xyz_range"]),
            sample_uniform(rng, cfg["anisotropy_xyz_range"]),
            sample_uniform(rng, cfg["anisotropy_xyz_range"]),
        ],
        "noise_amp": sample_uniform(rng, cfg["noise_amp_range"]),
        "tpms_thickness_mm": sample_uniform(rng, cfg.get("tpms_thickness_mm_range", [0.18, 0.60])),
        "min_hole_size_vox": sample_int(rng, cfg.get("min_hole_size_vox_range", [1, 1])),
        "component_gap_mm": 0.0,
        "components": [comp],
        "seed": int(rng.integers(0, 2**31 - 1)),
        "lattice_grid_n": LATTICE_GRID_N,
        "node_blend_factor": LATTICE_NODE_BLEND_FACTOR,
        "lattice_binary_closing_iters": LATTICE_BINARY_CLOSING_ITERS,
        "lattice_radius_search_iters": LATTICE_RADIUS_SEARCH_ITERS,
    }
    p["candidate_id"] = f"TPMS_basic_{tpms_type}_{mode}_{idx:06d}"
    return p


def random_tpms_component(rng, tpms_type=None, mode=None, cfg=None):
    if tpms_type is None:
        tpms_type = str(rng.choice(TPMS_LIBRARY))
    if mode is None:
        mode = str(rng.choice(TPMS_BASIC_MODES))
    thickness_range = cfg.get("tpms_thickness_mm_range", [0.85, 1.15]) if cfg else [0.85, 1.15]
    return {
        "tpms_type": tpms_type,
        "mode": mode,
        "level_shift": float(rng.uniform(-0.35, 0.35)),
        "thickness_mm": sample_uniform(rng, thickness_range),
    }


def make_tpms_adv_params(stage, cfg, idx, rng, combo_name=None, tpms_type=None):
    if combo_name is None:
        candidates = [name for name, modes in MULTIWALL_MODE_TEMPLATES.items()
                      if len(modes) == int(cfg["n_components"])]
        combo_name = str(rng.choice(candidates))

    chosen_modes = MULTIWALL_MODE_TEMPLATES[combo_name]
    n_comp = len(chosen_modes)
    if tpms_type is None:
        tpms_type = str(rng.choice(TPMS_LIBRARY))

    comps = [random_tpms_component(rng, tpms_type=tpms_type, mode=m, cfg=cfg) for m in chosen_modes]

    min_total_vf = min(max(0.10, MIN_COMPONENT_VF * n_comp), 0.75)
    target_vf = sample_uniform(rng, [0.45, 0.55])

    p = {
        "generator_type": "tpms",
        "tpms_stage": stage,
        "multiwall_combo": combo_name,
        "size_mm": SIZE_MM,
        "target_vf": target_vf,
        "stl_mesh_size_mm": STL_MESH_SIZE_MM,
        "grid_n": TPMS_GRID_N,
        "frequency": sample_uniform(rng, cfg["frequency_range"]),
        "anisotropy_xyz": [sample_uniform(rng, cfg["anisotropy_xyz_range"]) for _ in range(3)],
        "noise_amp": sample_uniform(rng, cfg["noise_amp_range"]),
        "min_hole_size_vox": sample_int(rng, cfg.get("min_hole_size_vox_range", [1, 1])),
        "component_gap_mm": sample_uniform(rng, cfg["component_gap_mm_range"]) if "component_gap_mm_range" in cfg else 0.0,
        "components": comps,
        "seed": int(rng.integers(0, 2**31 - 1)),
        "lattice_grid_n": LATTICE_GRID_N,
        "node_blend_factor": LATTICE_NODE_BLEND_FACTOR,
        "lattice_binary_closing_iters": LATTICE_BINARY_CLOSING_ITERS,
        "lattice_radius_search_iters": LATTICE_RADIUS_SEARCH_ITERS,
    }
    p["candidate_id"] = f"TPMS_{stage}_{tpms_type}_{combo_name}_{idx:06d}".replace("/", "_").replace(" ", "_").replace("+", "-")
    return p


def make_voxel_params(voxel_mode, cfg, idx, rng):
    p = {
        "generator_type": "voxel",
        "voxel_mode": voxel_mode,
        "size_mm": SIZE_MM,
        "target_vf": sample_uniform(rng, [0.45, 0.55]),
        "stl_mesh_size_mm": STL_MESH_SIZE_MM,
        "grid_n": VOXEL_GRID_N,
        "min_thickness_vox": sample_int(rng, cfg["min_thickness_vox_range"]),
        "min_hole_size_vox": sample_int(rng, cfg.get("min_hole_size_vox_range", [1, 1])),
        "max_thickness_vox": sample_int(rng, cfg["max_thickness_vox_range"]),
        "closing_iter": sample_int(rng, cfg["closing_iter_range"]),
        "opening_iter": sample_int(rng, cfg["opening_iter_range"]),
        "anisotropy_z": sample_uniform(rng, cfg["anisotropy_z_range"]),
        "seed": int(rng.integers(0, 2**31 - 1)),
        "lattice_grid_n": LATTICE_GRID_N,
        "node_blend_factor": LATTICE_NODE_BLEND_FACTOR,
        "lattice_binary_closing_iters": LATTICE_BINARY_CLOSING_ITERS,
        "lattice_radius_search_iters": LATTICE_RADIUS_SEARCH_ITERS,
        "force_connected": FORCE_CONNECTED_LATTICE_VOXEL,
        "connectivity_repair_mode": CONNECTIVITY_REPAIR_MODE,
        "connectivity_bridge_radius_vox": CONNECTIVITY_BRIDGE_RADIUS_VOX,
        "connectivity_min_component_voxels": CONNECTIVITY_MIN_COMPONENT_VOXELS,
        "connectivity_max_bridges": CONNECTIVITY_MAX_BRIDGES,
        "connectivity_retry_after_contact_symmetry": CONNECTIVITY_RETRY_AFTER_CONTACT_SYMMETRY,
    }
    if voxel_mode in ["periodic_isotropic", "periodic_orthotropic"]:
        p["num_fourier_terms"] = sample_int(rng, cfg["num_fourier_terms_range"])
        p["sigma"] = np.nan
    else:
        p["sigma"] = sample_uniform(rng, cfg["sigma_xyz_range"])
        p["num_fourier_terms"] = np.nan
    p["candidate_id"] = f"VOX_{voxel_mode}_{idx:06d}"
    return p


def flatten_candidate_params(p):
    row = dict(p)
    row["parameter_json"] = json_dumps(p)
    if "components" in row:
        row["components_json"] = json_dumps(row["components"])
        row["component_count"] = len(row["components"])
        row["component_modes"] = "+".join([c["mode"] for c in row["components"]])
        row["component_tpms_types"] = "+".join([c["tpms_type"] for c in row["components"]])
        del row["components"]
    else:
        row["components_json"] = ""
        row["component_count"] = 0
        row["component_modes"] = ""
        row["component_tpms_types"] = ""
    if "anisotropy_xyz" in row:
        row["anisotropy_xyz_json"] = json_dumps(row["anisotropy_xyz"])
        del row["anisotropy_xyz"]
    else:
        row["anisotropy_xyz_json"] = ""
    return row


def unflatten_candidate_row(row):
    p = json.loads(row["parameter_json"])
    return p


def build_candidate_table(seed=42):
    rng = np.random.default_rng(seed)
    params = []

    # Lattice candidates.
    for lattice_mode, cfg in LATTICE_GENERATION_CONFIG.items():
        for i in range(int(cfg["n_candidates"])):
            params.append(make_lattice_params(lattice_mode, cfg, i, rng))

    # TPMS basic: every equation × wall/solid_A/solid_B × count.
    basic_cfg = TPMS_GENERATION_CONFIG["basic"]
    idx = 0
    for tpms_type in TPMS_LIBRARY:
        for mode in TPMS_BASIC_MODES:
            for _ in range(int(basic_cfg["n_candidates_per_tpms_mode"])):
                params.append(make_tpms_basic_params(tpms_type, mode, basic_cfg, idx, rng))
                idx += 1

    # TPMS advanced 2/3/4 component non-overlapping combinations.
    # TPMS: Multi-wall 템플릿 기반 (14타입 x 7조합)
    tpms_cfg = {
        "tpms_thickness_mm_range": [0.8, 2.5],
        "frequency_range": [5.0, 5.0],
        "anisotropy_xyz_range": [0.90, 1.20],
        "noise_amp_range": [0.0, 0.015],
        "component_gap_mm_range": [0.05, 0.35],
        "min_hole_size_vox_range": [1, 1],
    }
    idx = 0
    for tpms_type in TPMS_LIBRARY:
        for combo in MULTIWALL_MODE_TEMPLATES:
            n_comp = len(MULTIWALL_MODE_TEMPLATES[combo])
            cfg_for_stage = dict(tpms_cfg)
            cfg_for_stage["n_components"] = n_comp
            params.append(make_tpms_adv_params("multiwall", cfg_for_stage, idx, rng, combo_name=combo, tpms_type=tpms_type))
            idx += 1

    # Voxel candidates.
    for voxel_mode, cfg in VOXEL_GENERATION_CONFIG.items():
        for i in range(int(cfg["n_candidates"])):
            params.append(make_voxel_params(voxel_mode, cfg, i, rng))

    df = pd.DataFrame([flatten_candidate_params(p) for p in params])
    return df

candidate_df = build_candidate_table(seed=RANDOM_SEED)

master_csv = DIR_TABLE / "candidate_parameter_table_master.csv"
master_xlsx = DIR_TABLE / "candidate_parameter_table_master.xlsx"
candidate_df.to_csv(master_csv, index=False, encoding="utf-8-sig")
candidate_df.to_excel(master_xlsx, index=False)

print("Candidate table saved:", master_csv)
print("Total candidates:", len(candidate_df))

print(candidate_df["generator_type"].value_counts())
display(candidate_df.head(10))


In [ ]:

# ============================================================
# Cell 7. Run one candidate + batch generation
# ============================================================

def run_single_candidate_from_row(row_dict):
    row = dict(row_dict)
    p = json.loads(row["parameter_json"])
    candidate_id = p["candidate_id"]
    generator_type = p["generator_type"]

    stl_dir = DIR_GEOM / generator_type / "STL"
    stl_path = stl_dir / f"{candidate_id}.stl"
    stp_path = DIR_GEOM / generator_type / "STP_selected_only" / f"{candidate_id}.stp"

    if stl_path.exists() and RESUME_MODE and not OVERWRITE_EXISTING:
        return {
            "candidate_id": candidate_id,
            "generator_type": generator_type,
            "status": "skipped_existing",
            "stl_path": str(stl_path),
            "stp_path": str(stp_path) if stp_path.exists() else "",
            "actual_vf_est": np.nan,
            "vf_error": np.nan,
            "error_message": "",
        }

    try:
        if generator_type == "lattice":
            verts, faces, quick = generate_lattice_candidate(p)
        elif generator_type == "tpms":
            verts, faces, quick = generate_tpms_candidate(p)
        elif generator_type == "voxel":
            verts, faces, quick = generate_voxel_candidate(p)
        else:
            raise ValueError(f"Unknown generator_type: {generator_type}")

        if len(verts) == 0 or len(faces) == 0:
            raise RuntimeError("Empty mesh generated")

        if EXPORT_STL:
            write_binary_stl(stl_path, verts, faces, solid_name=candidate_id)

        stp_status = "not_requested"
        if EXPORT_STP:
            ok, msg = try_export_step_from_stl(stl_path, stp_path)
            stp_status = "ok" if ok else f"failed: {msg}"

        actual_vf = safe_float(quick.get("actual_vf_est", np.nan))
        result = {
            "candidate_id": candidate_id,
            "generator_type": generator_type,
            "status": "ok",
            "stl_path": str(stl_path),
            "stp_path": str(stp_path) if EXPORT_STP else "",
            "stp_status": stp_status,
            "actual_vf_est": actual_vf,
            "vf_error": actual_vf - TARGET_VF if not np.isnan(actual_vf) else np.nan,
            "error_message": "",
        }
        result.update(quick)
        return result

    except Exception as e:
        return {
            "candidate_id": candidate_id,
            "generator_type": generator_type,
            "status": "failed",
            "stl_path": "",
            "stp_path": "",
            "stp_status": "",
            "actual_vf_est": np.nan,
            "vf_error": np.nan,
            "error_message": repr(e),
        }



def run_batch_generation(candidate_df, n_workers=N_WORKERS):
    """
    Hybrid batch runner.

    - CPU-heavy lattice/voxel jobs use process-based parallelism when requested.
    - GPU-eligible TPMS jobs can run concurrently with CPU jobs so CPU and GPU are both utilized.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import time

    rows = candidate_df.to_dict("records")
    total = len(rows)
    n_workers = int(max(1, n_workers))

    print("=" * 80)
    print(f"Running {total} candidates")
    print(f"n_workers={n_workers} | backend={DEVICE} | GPU={USE_GPU_FOR_GENERATION} | fast={globals().get('FAST_SCREENING_MODE', False)} | mc_step={globals().get('MARCHING_CUBES_STEP_SIZE', 1)}")
    print(f"parallel_backend={PARALLEL_BACKEND} | save_every_n={SAVE_EVERY_N}")
    print(f"Result folder: {OUTPUT_ROOT.resolve()}")
    print("=" * 80)

    results = []
    t0 = time.time()

    def _save_progress(force=False):
        if not results:
            return
        done = len(results)
        if force or done % SAVE_EVERY_N == 0:
            progress_df = pd.DataFrame(results)
            progress_df.to_csv(DIR_LOG / "progress_log.csv", index=False, encoding="utf-8-sig")
            try:
                progress_df.to_excel(DIR_LOG / "progress_log.xlsx", index=False)
            except Exception:
                pass

    def _print_progress(i, res):
        elapsed = time.time() - t0
        rate = i / max(elapsed, 1e-9)
        remain = (total - i) / max(rate, 1e-9)
        print(f"[{i:4d}/{total}] {res.get('candidate_id','')} | {res.get('status','')} | elapsed={elapsed/60:.1f} min | ETA={remain/60:.1f} min")

    def _run_rows_serial(row_list):
        out=[]
        for row in row_list:
            out.append(run_single_candidate_from_row(row))
        return out

    def _run_rows_joblib(row_list, n_jobs):
        return Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
            delayed(run_single_candidate_from_row)(row) for row in row_list
        )

    def _run_rows_thread(row_list, n_jobs):
        out=[]
        with ThreadPoolExecutor(max_workers=n_jobs) as ex:
            futs={ex.submit(run_single_candidate_from_row,row): row.get("candidate_id","") for row in row_list}
            for fut in as_completed(futs):
                cid=futs[fut]
                try:
                    out.append(fut.result())
                except Exception as e:
                    out.append({
                        "candidate_id": cid, "generator_type": "unknown", "status": "failed",
                        "stl_path": "", "stp_path": "", "stp_status": "",
                        "actual_vf_est": np.nan, "vf_error": np.nan, "error_message": repr(e),
                    })
        return out

    cpu_rows = rows
    gpu_rows = []
    if USE_GPU_FOR_GENERATION:
        gpu_rows = [r for r in rows if str(json.loads(r["parameter_json"]).get("generator_type","")) == "tpms"]
        cpu_rows = [r for r in rows if str(json.loads(r["parameter_json"]).get("generator_type","")) != "tpms"]

    # Launch CPU and GPU work concurrently when possible
    batches = []
    if gpu_rows and cpu_rows:
        with ThreadPoolExecutor(max_workers=2) as ex:
            cpu_fn = _run_rows_joblib if PARALLEL_BACKEND in ["loky","process","processes"] and n_workers > 1 else (_run_rows_thread if n_workers > 1 else _run_rows_serial)
            fut_cpu = ex.submit(cpu_fn, cpu_rows, n_workers) if cpu_fn != _run_rows_serial else ex.submit(cpu_fn, cpu_rows)
            fut_gpu = ex.submit(_run_rows_serial, gpu_rows)
            for fut in as_completed([fut_cpu, fut_gpu]):
                batches.extend(fut.result())
    else:
        if n_workers == 1:
            batches = _run_rows_serial(rows)
        elif PARALLEL_BACKEND in ["loky","process","processes"]:
            batches = _run_rows_joblib(rows, n_workers)
        elif PARALLEL_BACKEND in ["thread", "threads", "threading"]:
            batches = _run_rows_thread(rows, n_workers)
        else:
            batches = _run_rows_serial(rows)

    for i, res in enumerate(batches, 1):
        results.append(res)
        if (i == 1) or (i % PROGRESS_PRINT_EVERY_N == 0) or (i == total):
            _print_progress(i, res)
        _save_progress(force=False)

    _save_progress(force=True)

    res_df = pd.DataFrame(results)
    res_df.to_csv(DIR_LOG / "generation_result_log.csv", index=False, encoding="utf-8-sig")
    res_df.to_excel(DIR_LOG / "generation_result_log.xlsx", index=False)

    merged = candidate_df.merge(res_df, on=["candidate_id", "generator_type"], how="left")
    merged.to_csv(DIR_LOG / "candidate_table_with_generation_results.csv", index=False, encoding="utf-8-sig")
    merged.to_excel(DIR_LOG / "candidate_table_with_generation_results.xlsx", index=False)

    failed = res_df[res_df["status"].eq("failed")]
    if len(failed):
        failed.to_csv(DIR_LOG / "failed_log.csv", index=False, encoding="utf-8-sig")

    print("Done.")
    print(res_df["status"].value_counts(dropna=False))
    print("Result folder:", OUTPUT_ROOT.resolve())
    return merged, res_df


def verify_generated_files(result_df):
    """Check whether STL/STP files physically exist and summarize counts by folder."""
    df = result_df.copy()
    if "stl_path" not in df.columns:
        df["stl_exists"] = False
    else:
        df["stl_exists"] = df["stl_path"].fillna("").map(lambda x: Path(str(x)).exists() if str(x) else False)

    if "stp_path" not in df.columns:
        df["stp_exists"] = False
    else:
        df["stp_exists"] = df["stp_path"].fillna("").map(lambda x: Path(str(x)).exists() if str(x) else False)

    summary = (
        df.groupby(["generator_type", "status"], dropna=False)
        .agg(
            n_candidates=("candidate_id", "count"),
            n_stl_files=("stl_exists", "sum"),
            n_stp_files=("stp_exists", "sum"),
        )
        .reset_index()
    )

    file_inventory = []
    for gen in ["lattice", "tpms", "voxel"]:
        stl_dir = DIR_GEOM / gen / "STL"
        stp_dir = DIR_GEOM / gen / "STP_selected_only"
        file_inventory.append({
            "generator_type": gen,
            "stl_dir": str(stl_dir),
            "stl_count_on_disk": len(list(stl_dir.glob("*.stl"))) if stl_dir.exists() else 0,
            "stp_dir": str(stp_dir),
            "stp_count_on_disk": len(list(stp_dir.glob("*.stp"))) if stp_dir.exists() else 0,
        })
    inventory_df = pd.DataFrame(file_inventory)

    summary.to_csv(DIR_LOG / "generation_file_check_summary.csv", index=False, encoding="utf-8-sig")
    inventory_df.to_csv(DIR_LOG / "generation_file_inventory.csv", index=False, encoding="utf-8-sig")
    df.to_csv(DIR_LOG / "generation_result_log_with_file_check.csv", index=False, encoding="utf-8-sig")

    print("\nFile existence check")
    print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
    print(summary)
    print("\nFolder inventory")
    print(inventory_df)
    return df, summary, inventory_df


# Auto-run block.
# Previous version only ran candidate_df.head(6), so most users saw a normal finish
# even though the full candidate table was not generated.
if AUTO_RUN_GENERATION:
    if RUN_GENERATION_MODE == "preview":
        active_df = candidate_df.head(PREVIEW_N).copy()
        active_workers = 1
        print(f"RUN_GENERATION_MODE='preview': running only {len(active_df)} candidates.")
    elif RUN_GENERATION_MODE == "all":
        active_df = candidate_df.copy()
        # Use configured worker count directly. Previous logic forced 1 worker in fast-screening mode,
        # which under-utilized high-end CPUs and made generation look artificially slow.
        active_workers = max(1, int(N_WORKERS))
        print(f"RUN_GENERATION_MODE='all': running all {len(active_df)} candidates.")
    elif RUN_GENERATION_MODE == "none":
        active_df = None
        active_workers = 1
        print("RUN_GENERATION_MODE='none': candidate table only. No geometry generation was run.")
    else:
        raise ValueError("RUN_GENERATION_MODE must be 'preview', 'all', or 'none'.")

    if active_df is not None and len(active_df) > 0:
        merged_result_df, generation_log_df = run_batch_generation(active_df, n_workers=active_workers)
        generation_log_checked_df, file_check_summary_df, file_inventory_df = verify_generated_files(generation_log_df)
        display(generation_log_checked_df.head(20))
        display(file_check_summary_df)
        display(file_inventory_df)
else:
    print("AUTO_RUN_GENERATION=False: candidate table was built, but no STL generation was run.")
    print("To generate all candidates manually, run:")
    print("merged_result_df, generation_log_df = run_batch_generation(candidate_df, n_workers=N_WORKERS)")



# ----------------------------------------------------------------
# Performance-max generation runner with multi-GPU TPMS dispatch
# ----------------------------------------------------------------
def run_single_candidate_from_row(row_dict):
    row = dict(row_dict)
    p = json.loads(row["parameter_json"])
    candidate_id = p["candidate_id"]
    generator_type = p["generator_type"]

    if "_gpu_device_id" in row and "USE_GPU_FOR_GENERATION" in globals() and USE_GPU_FOR_GENERATION:
        try:
            _set_generation_gpu_device(int(row["_gpu_device_id"]))
        except Exception:
            pass

    stl_dir = DIR_GEOM / generator_type / "STL"
    stl_path = stl_dir / f"{candidate_id}.stl"
    stp_path = DIR_GEOM / generator_type / "STP_selected_only" / f"{candidate_id}.stp"

    if stl_path.exists() and RESUME_MODE and not OVERWRITE_EXISTING:
        return {
            "candidate_id": candidate_id,
            "generator_type": generator_type,
            "status": "skipped_existing",
            "stl_path": str(stl_path),
            "stp_path": str(stp_path) if stp_path.exists() else "",
            "actual_vf_est": np.nan,
            "vf_error": np.nan,
            "error_message": "",
        }

    try:
        if generator_type == "lattice":
            verts, faces, quick = generate_lattice_candidate(p)
        elif generator_type == "tpms":
            verts, faces, quick = generate_tpms_candidate(p)
        elif generator_type == "voxel":
            verts, faces, quick = generate_voxel_candidate(p)
        else:
            raise ValueError(f"Unknown generator_type: {generator_type}")

        if len(verts) == 0 or len(faces) == 0:
            raise RuntimeError("Empty mesh generated")

        if EXPORT_STL:
            write_binary_stl(stl_path, verts, faces, solid_name=candidate_id)

        stp_status = "not_requested"
        if EXPORT_STP:
            ok, msg = try_export_step_from_stl(stl_path, stp_path)
            stp_status = "ok" if ok else f"failed: {msg}"

        actual_vf = safe_float(quick.get("actual_vf_est", np.nan))
        result = {
            "candidate_id": candidate_id,
            "generator_type": generator_type,
            "status": "ok",
            "stl_path": str(stl_path),
            "stp_path": str(stp_path) if EXPORT_STP else "",
            "stp_status": stp_status,
            "actual_vf_est": actual_vf,
            "vf_error": actual_vf - TARGET_VF if not np.isnan(actual_vf) else np.nan,
            "error_message": "",
        }
        result.update(quick)
        if "_gpu_device_id" in row:
            result["gpu_device_id"] = int(row["_gpu_device_id"])
        return result

    except Exception as e:
        return {
            "candidate_id": candidate_id,
            "generator_type": generator_type,
            "status": "failed",
            "stl_path": "",
            "stp_path": "",
            "stp_status": "",
            "actual_vf_est": np.nan,
            "vf_error": np.nan,
            "error_message": repr(e),
            "gpu_device_id": int(row.get("_gpu_device_id", -1)),
        }


def run_batch_generation(candidate_df, n_workers=N_WORKERS):
    """
    Hybrid performance-max batch runner.

    - lattice + voxel: CPU process pool (loky) to push CPU utilization
    - tpms: dispatched across multiple GPU device lanes when available
    - CPU pool and GPU lanes run concurrently
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import time

    rows = candidate_df.to_dict("records")
    total = len(rows)
    n_workers = int(max(1, n_workers))

    print("=" * 80)
    print(f"Running {total} candidates")
    print(f"n_workers={n_workers} | backend={DEVICE} | GPU={USE_GPU_FOR_GENERATION} | mc_step={globals().get('MARCHING_CUBES_STEP_SIZE', 1)}")
    print(f"parallel_backend={PARALLEL_BACKEND} | save_every_n={SAVE_EVERY_N} | gpu_devices={globals().get('GPU_DEVICE_IDS', [GPU_DEVICE_ID]) if USE_GPU_FOR_GENERATION else []}")
    print(f"Result folder: {OUTPUT_ROOT.resolve()}")
    print("=" * 80)

    results = []
    t0 = time.time()

    def _save_progress(force=False):
        if not results:
            return
        done = len(results)
        if force or done % SAVE_EVERY_N == 0:
            progress_df = pd.DataFrame(results)
            progress_df.to_csv(DIR_LOG / "progress_log.csv", index=False, encoding="utf-8-sig")
            try:
                progress_df.to_excel(DIR_LOG / "progress_log.xlsx", index=False)
            except Exception:
                pass

    def _print_progress(i, res):
        elapsed = time.time() - t0
        rate = i / max(elapsed, 1e-9)
        remain = (total - i) / max(rate, 1e-9)
        print(f"[{i:4d}/{total}] {res.get('candidate_id','')} | {res.get('status','')} | elapsed={elapsed/60:.1f} min | ETA={remain/60:.1f} min")

    def _run_rows_serial(row_list):
        return [run_single_candidate_from_row(r) for r in row_list]

    def _run_rows_joblib(row_list, n_jobs):
        return Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
            delayed(run_single_candidate_from_row)(row) for row in row_list
        )

    def _run_rows_thread(row_list, n_jobs):
        out = []
        with ThreadPoolExecutor(max_workers=n_jobs) as ex:
            futs = {ex.submit(run_single_candidate_from_row, row): row.get("candidate_id", "") for row in row_list}
            for fut in as_completed(futs):
                cid = futs[fut]
                try:
                    out.append(fut.result())
                except Exception as e:
                    out.append({
                        "candidate_id": cid, "generator_type": "unknown", "status": "failed",
                        "stl_path": "", "stp_path": "", "stp_status": "",
                        "actual_vf_est": np.nan, "vf_error": np.nan, "error_message": repr(e),
                    })
        return out

    gpu_rows = []
    cpu_rows = rows
    if USE_GPU_FOR_GENERATION:
        gpu_rows = [r for r in rows if str(json.loads(r["parameter_json"]).get("generator_type", "")) == "tpms"]
        cpu_rows = [r for r in rows if str(json.loads(r["parameter_json"]).get("generator_type", "")) != "tpms"]

    cpu_fn = _run_rows_joblib if PARALLEL_BACKEND in ["loky","process","processes"] and n_workers > 1 else (_run_rows_thread if n_workers > 1 else _run_rows_serial)
    batches = []

    def _gpu_dispatch(row_list):
        if not row_list:
            return []
        devs = list(globals().get("GPU_DEVICE_IDS", [GPU_DEVICE_ID]))
        if not devs:
            devs = [GPU_DEVICE_ID]
        lanes = [[] for _ in devs]
        for i, row in enumerate(row_list):
            rr = dict(row)
            rr["_gpu_device_id"] = int(devs[i % len(devs)])
            lanes[i % len(devs)].append(rr)
        out = []
        with ThreadPoolExecutor(max_workers=len(devs)) as ex:
            futs = [ex.submit(_run_rows_serial, lane) for lane in lanes if lane]
            for fut in as_completed(futs):
                out.extend(fut.result())
        return out

    if gpu_rows and cpu_rows:
        with ThreadPoolExecutor(max_workers=2) as ex:
            fut_cpu = ex.submit(cpu_fn, cpu_rows, n_workers) if cpu_fn != _run_rows_serial else ex.submit(cpu_fn, cpu_rows)
            fut_gpu = ex.submit(_gpu_dispatch, gpu_rows)
            for fut in as_completed([fut_cpu, fut_gpu]):
                batches.extend(fut.result())
    else:
        if gpu_rows:
            batches = _gpu_dispatch(gpu_rows)
        else:
            if n_workers == 1:
                batches = _run_rows_serial(rows)
            elif PARALLEL_BACKEND in ["loky","process","processes"]:
                batches = _run_rows_joblib(rows, n_workers)
            elif PARALLEL_BACKEND in ["thread","threads","threading"]:
                batches = _run_rows_thread(rows, n_workers)
            else:
                batches = _run_rows_serial(rows)

    for i, res in enumerate(batches, 1):
        results.append(res)
        if (i == 1) or (i % PROGRESS_PRINT_EVERY_N == 0) or (i == total):
            _print_progress(i, res)
        _save_progress(force=False)

    _save_progress(force=True)

    res_df = pd.DataFrame(results)
    res_df.to_csv(DIR_LOG / "generation_result_log.csv", index=False, encoding="utf-8-sig")
    try:
        res_df.to_excel(DIR_LOG / "generation_result_log.xlsx", index=False)
    except Exception:
        pass

    merged = candidate_df.merge(res_df, on=["candidate_id", "generator_type"], how="left")
    merged.to_csv(DIR_LOG / "candidate_table_with_generation_results.csv", index=False, encoding="utf-8-sig")
    try:
        merged.to_excel(DIR_LOG / "candidate_table_with_generation_results.xlsx", index=False)
    except Exception:
        pass

    failed = res_df[res_df["status"].eq("failed")]
    if len(failed):
        failed.to_csv(DIR_LOG / "failed_log.csv", index=False, encoding="utf-8-sig")

    print("Done.")
    print(res_df["status"].value_counts(dropna=False))
    return merged, res_df


RUN_GENERATION_MODE='all': running all 434 candidates.
Running 434 candidates
n_workers=1 | backend=cpu | GPU=False | fast=False | mc_step=1
parallel_backend=thread | save_every_n=5
Result folder: /content/drive/MyDrive/CMSL_results/Result_Batch_FastScreening_LowQualitySTL_260706_022507
[   1/434] TPMS_basic_gyroid_wall_000000 | ok | elapsed=29.1 min | ETA=12601.5 min
[   5/434] TPMS_basic_gyroid_wall_000004 | ok | elapsed=29.1 min | ETA=2497.0 min
[  10/434] TPMS_basic_gyroid_solid_A_000009 | ok | elapsed=29.1 min | ETA=1234.0 min
[  15/434] TPMS_basic_gyroid_solid_A_000014 | ok | elapsed=29.1 min | ETA=813.0 min
[  20/434] TPMS_basic_gyroid_solid_B_000019 | ok | elapsed=29.1 min | ETA=602.5 min
[  25/434] TPMS_basic_primitive_wall_000024 | ok | elapsed=29.1 min | ETA=476.2 min
[  30/434] TPMS_basic_primitive_wall_000029 | ok | elapsed=29.1 min | ETA=392.0 min
[  35/434] TPMS_basic_primitive_solid_A_000034 | ok | elapsed=29.1 min | ETA=331.8 min
[  40/434] TPMS_basic_primitive_solid_A

,candidate_id,generator_type,status,stl_path,stp_path,stp_status,actual_vf_est,vf_error,error_message,voxel_grid_n,component_count,component_modes,component_tpms_types,component_thicknesses_mm,multiwall_combo,min_hole_size_vox,compute_backend,islands_before_repair,stl_exists,stp_exists
0,TPMS_basic_gyroid_wall_000000,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.275398,-0.024602,,150.0,1.0,wall,gyroid,[0.364],,2.0,cpu,2.0,True,False
1,TPMS_basic_gyroid_wall_000001,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.264675,-0.035325,,150.0,1.0,wall,gyroid,[0.336],,2.0,cpu,1.0,True,False
2,TPMS_basic_gyroid_wall_000002,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.323125,0.023125,,150.0,1.0,wall,gyroid,[0.445],,1.0,cpu,2.0,True,False
3,TPMS_basic_gyroid_wall_000003,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.300026,0.000026,,150.0,1.0,wall,gyroid,[0.245],,2.0,cpu,1.0,True,False
4,TPMS_basic_gyroid_wall_000004,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.265246,-0.034754,,150.0,1.0,wall,gyroid,[0.38],,2.0,cpu,1.0,True,False
5,TPMS_basic_gyroid_wall_000005,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.287437,-0.012563,,150.0,1.0,wall,gyroid,[0.343],,2.0,cpu,1.0,True,False
6,TPMS_basic_gyroid_wall_000006,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.254107,-0.045893,,150.0,1.0,wall,gyroid,[0.508],,2.0,cpu,4.0,True,False
7,TPMS_basic_gyroid_wall_000007,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.299359,-0.000641,,150.0,1.0,wall,gyroid,[0.447],,1.0,cpu,2.0,True,False
8,TPMS_basic_gyroid_solid_A_000008,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.278022,-0.021978,,150.0,1.0,solid_A,gyroid,[0.278],,2.0,cpu,1.0,True,False
9,TPMS_basic_gyroid_solid_A_000009,tpms,ok,/content/drive/MyDrive/CMSL_results/Result_Bat...,,not_requested,0.279017,-0.020983,,150.0,1.0,solid_A,gyroid,[0.522],,2.0,cpu,1.0,True,False


,generator_type,status,n_candidates,n_stl_files,n_stp_files
0,tpms,failed,58,0,0
1,tpms,ok,376,376,0


,generator_type,stl_dir,stl_count_on_disk,stp_dir,stp_count_on_disk
0,lattice,/content/drive/MyDrive/CMSL_results/Result_Bat...,0,/content/drive/MyDrive/CMSL_results/Result_Bat...,0
1,tpms,/content/drive/MyDrive/CMSL_results/Result_Bat...,376,/content/drive/MyDrive/CMSL_results/Result_Bat...,0
2,voxel,/content/drive/MyDrive/CMSL_results/Result_Bat...,0,/content/drive/MyDrive/CMSL_results/Result_Bat...,0


In [ ]:

# ============================================================
# Cell 7A. Boundary-condition validation helpers
# ============================================================

def validate_lattice_voxel_boundary_conditions(params_or_row, regenerate_mask=True):
    """Validate Isotropic/Orthotropic face constraints for one Lattice/Voxel candidate.

    Returns a dictionary containing:
    - boundary_condition_applicable: False for TPMS
    - contact_face_symmetry_ok: exact Boolean result for the requested rule
    - connected_components: number of 3D solid components after finalization
    - mismatch_voxels: per-face mismatch counts; all must be zero when OK
    """
    if isinstance(params_or_row, pd.Series):
        p = json.loads(params_or_row["parameter_json"])
    elif isinstance(params_or_row, dict) and "parameter_json" in params_or_row:
        p = json.loads(params_or_row["parameter_json"])
    else:
        p = dict(params_or_row)

    generator_type = str(p.get("generator_type", "")).lower()
    if generator_type == "tpms":
        return {
            "candidate_id": p.get("candidate_id", ""),
            "generator_type": generator_type,
            "boundary_condition_applicable": False,
            "reason": "TPMS is intentionally excluded from Isotropic/Orthotropic face constraints.",
        }

    if generator_type == "lattice":
        mode = p.get("lattice_mode", "stochastic")
        if regenerate_mask:
            nodes = generate_lattice_nodes(p, seed=int(p["seed"]))
            edges = select_lattice_edges(nodes, p)
            _, mask, _ = calibrate_lattice_radius(nodes, edges, p)
        else:
            raise ValueError("regenerate_mask=False requires a mask-based extension, not implemented here.")
    elif generator_type == "voxel":
        mode = p.get("voxel_mode", "stochastic")
        if regenerate_mask:
            mask = generate_voxel_mask(p)
        else:
            raise ValueError("regenerate_mask=False requires a mask-based extension, not implemented here.")
    else:
        raise ValueError(f"Unsupported generator_type: {generator_type}")

    ok, mismatches = contact_face_symmetry_report(
        mask,
        mode,
        depth_vox=int(p.get("contact_surface_depth_vox", CONTACT_SURFACE_DEPTH_VOX)),
    )
    global_symmetry_ok = None
    global_symmetry_mismatch_voxels = None
    if mode in ["periodic_isotropic", "periodic_orthotropic"]:
        sym_or = symmetrize_binary_mask(mask, mode, target_vf=None, method="or")
        # If the mask is already symmetric, OR-symmetrization adds no voxels.
        global_symmetry_mismatch_voxels = int(np.count_nonzero(sym_or ^ mask))
        global_symmetry_ok = bool(global_symmetry_mismatch_voxels == 0)

    return {
        "candidate_id": p.get("candidate_id", ""),
        "generator_type": generator_type,
        "mode": mode,
        "boundary_condition_applicable": mode in ["periodic_isotropic", "periodic_orthotropic"],
        "contact_face_symmetry_ok": None if ok is None else bool(ok),
        "global_symmetry_ok": global_symmetry_ok,
        "global_symmetry_mismatch_voxels": global_symmetry_mismatch_voxels,
        "mismatch_voxels": mismatches,
        "connected_components": int(count_connected_components(mask)),
        "actual_vf_est": float(estimate_mask_vf(mask)),
    }


def validate_candidate_table_boundary_conditions(candidate_df, max_per_group=3):
    """Quick validation over representative Lattice/Voxel candidates only."""
    rows = []
    for generator_type in ["lattice", "voxel"]:
        sub = candidate_df[candidate_df["generator_type"].eq(generator_type)].head(int(max_per_group))
        for _, row in sub.iterrows():
            rows.append(validate_lattice_voxel_boundary_conditions(row))
    return pd.DataFrame(rows)



In [ ]:
# ============================================================
# Cell 7B. 모델별 상세 요약 + 압축시험용 물성 엑셀 자동 생성
#   - 성공/실패 전부 포함
#   - 실제 메쉬 부피 / 상대밀도, 프린팅 적합성 등급, 시편 메타데이터 추가
#   - 저장: 03_generation_logs/model_summary.xlsx
# ============================================================
import os
import json as _json
import numpy as np
import pandas as pd
from scipy.ndimage import label as _ndlabel

# --- 프린팅 재료 밀도 (g/cm^3). DLP 광경화 수지 기준 기본값. 필요시 수정 ---
PRINT_MATERIAL_DENSITY_G_CM3 = 1.15
# --- 압축 하중 방향 (기록용). TPMS는 등방적이나 실제 시험축 명시 ---
COMPRESSION_LOAD_AXIS = "Z"

def _read_stl_binary(path):
    """바이너리 STL을 읽어 (V, F) 반환. 실패 시 None."""
    try:
        with open(path, "rb") as f:
            f.read(80)
            n_tri = int.from_bytes(f.read(4), "little")
            data = np.frombuffer(f.read(), dtype=np.uint8)
        # 각 삼각형 = 50 bytes
        tris = data[: n_tri * 50].reshape(n_tri, 50)
        verts = tris[:, 12:48].copy().view(np.float32).reshape(n_tri, 3, 3)
        V = verts.reshape(-1, 3).astype(np.float64)
        F = np.arange(n_tri * 3).reshape(n_tri, 3)
        return V, F
    except Exception:
        return None

def _signed_volume(V, F):
    tri = V[F]
    vol = np.einsum("ij,ij->i", tri[:, 0], np.cross(tri[:, 1], tri[:, 2])) / 6.0
    return abs(float(np.sum(vol)))

def _mesh_metrics_from_stl(stl_path, size_mm):
    """STL에서 실제 부피(mm^3), 상대밀도, 예상질량(g) 계산."""
    out = {"mesh_volume_mm3": np.nan, "relative_density": np.nan,
           "est_mass_g": np.nan, "bbox_mm": ""}
    if not stl_path or not os.path.exists(stl_path):
        return out
    res = _read_stl_binary(stl_path)
    if res is None:
        return out
    V, F = res
    vol_mm3 = _signed_volume(V, F)
    bbox = V.max(axis=0) - V.min(axis=0)
    total_vol = float(size_mm) ** 3 if size_mm and not np.isnan(size_mm) else np.nan
    rel_density = vol_mm3 / total_vol if total_vol and total_vol > 0 else np.nan
    mass_g = vol_mm3 * 1e-3 * PRINT_MATERIAL_DENSITY_G_CM3  # mm^3 -> cm^3 -> g
    out.update({
        "mesh_volume_mm3": round(vol_mm3, 3),
        "relative_density": round(rel_density, 4) if not np.isnan(rel_density) else np.nan,
        "est_mass_g": round(mass_g, 4),
        "bbox_mm": f"{bbox[0]:.2f} x {bbox[1]:.2f} x {bbox[2]:.2f}",
    })
    return out

def _printability_grade(row, mesh):
    """프린팅 적합성 등급: A(양호)/B(주의)/C(부적합)/-(실패).
    - island(복구전) 많음, 두께 얇음, 부피/상대밀도 이상 등을 종합."""
    if str(row.get("status")) != "ok":
        return "-", "generation_failed"
    reasons = []
    grade = "A"
    isl = row.get("islands_before_repair", np.nan)
    try:
        isl = float(isl)
    except Exception:
        isl = np.nan
    if not np.isnan(isl) and isl > 30:
        grade = "B"; reasons.append(f"island_before_repair={int(isl)}")
    if not np.isnan(isl) and isl > 150:
        grade = "C"; reasons.append("island_very_high")
    rd = mesh.get("relative_density", np.nan)
    if not np.isnan(rd) and (rd < 0.35 or rd > 0.65):
        # 목표 VF 0.45~0.55 대비 메쉬 상대밀도가 크게 벗어나면 주의
        if grade == "A":
            grade = "B"
        reasons.append(f"relative_density={rd}")
    return grade, "; ".join(reasons) if reasons else "ok"

def build_model_summary_excel():
    src_csv = DIR_LOG / "candidate_table_with_generation_results.csv"
    if not src_csv.exists():
        print("[요약 스킵] 생성 결과 로그 없음:", src_csv)
        return None
    df = pd.read_csv(src_csv)

    def safe(row, key, default=""):
        return row[key] if key in row and pd.notna(row[key]) else default

    rows = []
    for _, r in df.iterrows():
        size_mm = safe(r, "size_mm", np.nan)
        try: size_mm = float(size_mm)
        except Exception: size_mm = np.nan
        freq = safe(r, "frequency", np.nan)
        try:
            n_cell = float(freq); cell_size = size_mm / n_cell if n_cell else np.nan
        except Exception:
            n_cell, cell_size = np.nan, np.nan

        stl_path = str(safe(r, "stl_path", ""))
        stl_size_mb = np.nan
        if stl_path and os.path.exists(stl_path):
            try: stl_size_mb = round(os.path.getsize(stl_path)/(1024*1024), 2)
            except Exception: pass

        mesh = _mesh_metrics_from_stl(stl_path, size_mm)
        grade, grade_reason = _printability_grade(r, mesh)

        target_vf = safe(r, "target_vf", np.nan)
        actual_vf = safe(r, "actual_vf_est", np.nan)

        rows.append({
            "시편ID": safe(r, "candidate_id"),
            "성공여부": safe(r, "status"),
            "실패사유": safe(r, "error_message"),
            "프린팅적합등급": grade,
            "적합성_비고": grade_reason,
            "TPMS함수": safe(r, "component_tpms_types"),
            "모드/조합": safe(r, "multiwall_combo") or safe(r, "component_modes"),
            "component수": safe(r, "component_count"),
            "component별_두께mm": safe(r, "component_thicknesses_mm"),
            "목표VF": target_vf,
            "복셀VF(추정)": actual_vf,
            "메쉬실제부피_mm3": mesh["mesh_volume_mm3"],
            "상대밀도": mesh["relative_density"],
            "예상질량_g": mesh["est_mass_g"],
            "실측bbox_mm": mesh["bbox_mm"],
            "전체크기_mm": size_mm,
            "셀개수(변당)": freq,
            "셀크기_mm": round(cell_size, 3) if pd.notna(cell_size) else np.nan,
            "압축하중축": COMPRESSION_LOAD_AXIS,
            "재료밀도_g_cm3": PRINT_MATERIAL_DENSITY_G_CM3,
            "island(복구전)": safe(r, "islands_before_repair"),
            "seed": safe(r, "seed"),
            "STL크기_MB": stl_size_mb,
            "STL경로": stl_path,
            # 실험 기록용 빈 칸 (실측값 나중에 채우기)
            "실측_최대하중_N": "",
            "실측_압축강도_MPa": "",
            "실측_탄성계수_MPa": "",
            "프린팅_레이어두께_mm": "",
            "비고": "",
        })

    summary = pd.DataFrame(rows)
    order = {"ok": 0, "skipped_existing": 1, "failed": 2}
    summary["_o"] = summary["성공여부"].map(lambda s: order.get(str(s), 3))
    summary = summary.sort_values(["_o", "프린팅적합등급", "시편ID"]).drop(columns=["_o"]).reset_index(drop=True)

    out_csv = DIR_LOG / "model_summary.csv"
    out_xlsx = DIR_LOG / "model_summary.xlsx"
    summary.to_csv(out_csv, index=False, encoding="utf-8-sig")
    summary.to_excel(out_xlsx, index=False)

    n_ok = int((summary["성공여부"] == "ok").sum())
    print("=" * 60)
    print("압축시험용 모델 요약 엑셀 생성 완료")
    print(f"  전체 {len(summary)}개 | 성공 {n_ok}개")
    print(f"  저장: {out_xlsx}")
    if n_ok > 0:
        okdf = summary[summary["성공여부"] == "ok"]
        print("\n[프린팅 적합 등급 분포]")
        print(okdf["프린팅적합등급"].value_counts().to_string())
        rd = pd.to_numeric(okdf["상대밀도"], errors="coerce")
        print(f"\n[상대밀도] 평균 {rd.mean():.3f} | 범위 {rd.min():.3f}~{rd.max():.3f}")
        print("  (실제 압축강도-상대밀도 관계 분석의 핵심 지표)")
    print("=" * 60)
    return summary

model_summary_df = build_model_summary_excel()
if model_summary_df is not None:
    display(model_summary_df.head(15))


In [ ]:

# ============================================================
# Cell 8. Structural descriptor extraction — core config + utilities
# ============================================================
# This version is CPU-stable by default. It avoids CuPy cross-device arrays,
# which caused the previous descriptor table to fail before any descriptor was written.
# Descriptor families are split into separate cells:
#   Cell 8B: Point-based mass distribution descriptors
#   Cell 8C: Surface area/curvature descriptors
#   Cell 8D: Slice-pair image descriptors
#   Cell 8E: Lattice node/strut descriptors
#   Cell 8F: Integrated runner and export

DESCRIPTOR_SCHEMA_VERSION = "structural_descriptor_v4_260509_graph_voxel_descriptor_expanded_harmonized"

from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, re, json, math, struct, warnings, time, copy
import numpy as np
import pandas as pd

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

try:
    from scipy.spatial import cKDTree
    from scipy.ndimage import zoom as ndi_zoom, binary_fill_holes, distance_transform_edt, binary_erosion
    SCIPY_DESCRIPTOR_AVAILABLE = True
except Exception:
    cKDTree = None
    ndi_zoom = None
    binary_fill_holes = None
    SCIPY_DESCRIPTOR_AVAILABLE = False

try:
    from scipy.ndimage import label as ndi_label
except Exception:
    ndi_label = None

try:
    import trimesh
    TRIMESH_AVAILABLE = True
except Exception:
    trimesh = None
    TRIMESH_AVAILABLE = False

try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except Exception:
    nx = None
    NETWORKX_AVAILABLE = False

try:
    from skimage.morphology import skeletonize_3d as sk_skeletonize_3d
    SKIMAGE_SKELETON_AVAILABLE = True
except Exception:
    try:
        from skimage.morphology import skeletonize as sk_skeletonize
        sk_skeletonize_3d = None
        SKIMAGE_SKELETON_AVAILABLE = True
    except Exception:
        sk_skeletonize_3d = None
        sk_skeletonize = None
        SKIMAGE_SKELETON_AVAILABLE = False

try:
    from scipy.ndimage import generate_binary_structure
except Exception:
    generate_binary_structure = None

DESCRIPTOR_DEFAULT_CONFIG = {
    # General execution
    "descriptor_backend": "cpu",             # keep CPU for reproducibility/stability
    "cpu_workers": max(1, min(16, os.cpu_count() or 1)),
    "parallel_backend": "thread",            # "serial" or "thread"

    # Input/output
    "candidate_table_name": "candidate_table.csv",
    "progress_table_name": "progress_log.csv",
    "descriptor_output_folder": "04_structural_descriptors_split",
    "write_csv": True,
    "write_excel": True,
    "write_merged_table": True,
    "save_point_cloud_csv": False,
    "save_curvature_raw_csv": False,
    "save_slice_merge_images": False,
    "slice_image_max_save_per_axis": 12,

    # Point descriptors, based on Parameter_distribution_New.py logic
    "enable_point_descriptors": True,
    "surface_point_count": 3000,
    "surface_point_oversample_factor": 4,
    "interior_grid_n": 32,
    "interior_point_max": 8000,
    "interior_sampling_method": "auto",          # auto | contains | voxel | inward_proxy
    "interior_inward_offset_fraction": 0.018,    # bbox max dimension fraction for proxy points
    "interior_inward_offset_steps": 3,           # multiple inward offsets from surface
    "interior_require_exact": False,             # False keeps proxy points instead of empty descriptors
    "local_range_abs_norm": 0.10,

    # Surface descriptors, based on Curvature Extraction_New.py logic
    "enable_surface_descriptors": True,
    "enable_surface_curvature": True,
    "max_vertices_for_curvature": 80000,
    "curvature_clip_percentile": 99.0,

    # Slice image descriptors, based on parameter_rawdata/Parameter_result/parameter_angle logic
    "enable_slice_descriptors": True,
    "slice_count": 100,                       # requested default: 100 slices
    "slice_image_size": 160,                  # 2D binary image resolution per slice
    "slice_pair_step": 1,
    "slice_axes": ["z", "x", "xy", "xyz"],
    "min_component_pixels": 2,
    "layer_height_mm": None,                  # if None, size_mm/(slice_count-1)

    # Mesh voxelization for slice/interior descriptors
    "voxel_grid_n": 96,
    "voxelize_pitch_factor": 1.0,

    # Lattice descriptors, based on Extract structure variable.py logic
    "enable_lattice_descriptors": True,
    "lattice_coord_key_decimals": 6,
    "lattice_l_over_d_from_radius": True,

    # Additional descriptors
    "enable_additional_descriptors": True,
    "enable_voxel_topology_descriptors": True,
    "voxel_topology_grid_n": 96,
    "run_length_sample_max": 250000,
    "overhang_angle_threshold_deg": 45.0,

    # Advanced graph / topology descriptors added from discussion
    "enable_advanced_graph_descriptors": True,
    "advanced_graph_grid_n": 96,
    "graph_boundary_margin_vox": 1,
    "graph_algebraic_connectivity_max_nodes": 400,
    "graph_path_axes": ["x", "y", "z"],
    "local_density_block_sizes": [4, 8],
    "save_skeleton_graph_csv": False,
}


def _deep_update_dict(base, overrides):
    if not isinstance(overrides, dict):
        return base
    for k, v in overrides.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            _deep_update_dict(base[k], v)
        else:
            base[k] = v
    return base


USER_DESCRIPTOR_CONFIG = globals().get("USER_DESCRIPTOR_CONFIG", {})
DESCRIPTOR_CONFIG = _deep_update_dict(copy.deepcopy(DESCRIPTOR_DEFAULT_CONFIG), USER_DESCRIPTOR_CONFIG)



_AXIS_VEC = {
    "x": np.array([1.0, 0.0, 0.0], dtype=float),
    "y": np.array([0.0, 1.0, 0.0], dtype=float),
    "z": np.array([0.0, 0.0, 1.0], dtype=float),
    "xy": np.array([1.0, 1.0, 0.0], dtype=float) / np.sqrt(2.0),
    "xz": np.array([1.0, 0.0, 1.0], dtype=float) / np.sqrt(2.0),
    "yz": np.array([0.0, 1.0, 1.0], dtype=float) / np.sqrt(2.0),
    "xyz": np.array([1.0, 1.0, 1.0], dtype=float) / np.sqrt(3.0),
}


def _safe_float(x, default=np.nan):
    try:
        v = float(x)
        if math.isfinite(v):
            return v
        return default
    except Exception:
        return default


def _json_load_maybe(x):
    if isinstance(x, dict):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return {}
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return {}
        try:
            return json.loads(s)
        except Exception:
            return {}
    return {}


def _stats(values, prefix, weights=None):
    """Robust descriptive statistics with optional weights.
    Keeps values and weights aligned after NaN filtering.
    """
    arr = np.asarray(values, dtype=float).ravel()
    finite = np.isfinite(arr)
    arr_f = arr[finite]
    out = {
        f"{prefix}_avg": np.nan,
        f"{prefix}_stdev": np.nan,
        f"{prefix}_min": np.nan,
        f"{prefix}_max": np.nan,
        f"{prefix}_p05": np.nan,
        f"{prefix}_p50": np.nan,
        f"{prefix}_p95": np.nan,
        f"{prefix}_count": int(arr_f.size),
    }
    if arr_f.size == 0:
        return out
    if weights is not None:
        w0 = np.asarray(weights, dtype=float).ravel()
        if len(w0) == len(arr):
            w = w0[finite]
            good_w = np.isfinite(w) & (w > 0)
            if np.any(good_w):
                vv = arr_f[good_w]
                ww = w[good_w]
                ww = ww / (np.sum(ww) + 1e-16)
                mu = float(np.sum(vv * ww))
                sig = float(np.sqrt(np.sum(ww * (vv - mu) ** 2)))
                out[f"{prefix}_avg"] = mu
                out[f"{prefix}_stdev"] = sig
            else:
                out[f"{prefix}_avg"] = float(np.mean(arr_f))
                out[f"{prefix}_stdev"] = float(np.std(arr_f))
        else:
            out[f"{prefix}_avg"] = float(np.mean(arr_f))
            out[f"{prefix}_stdev"] = float(np.std(arr_f))
    else:
        out[f"{prefix}_avg"] = float(np.mean(arr_f))
        out[f"{prefix}_stdev"] = float(np.std(arr_f))
    out[f"{prefix}_min"] = float(np.min(arr_f))
    out[f"{prefix}_max"] = float(np.max(arr_f))
    out[f"{prefix}_p05"] = float(np.percentile(arr_f, 5))
    out[f"{prefix}_p50"] = float(np.percentile(arr_f, 50))
    out[f"{prefix}_p95"] = float(np.percentile(arr_f, 95))
    return out


def _weighted_mean_std(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not np.any(m):
        return np.nan, np.nan
    v = v[m]
    w = w[m]
    w = w / (np.sum(w) + 1e-16)
    mu = float(np.sum(v * w))
    sig = float(np.sqrt(np.sum(w * (v - mu) ** 2)))
    return mu, sig


def _descriptor_output_dir():
    base = globals().get("OUTPUT_ROOT", Path.cwd())
    base = Path(base)
    out = base / str(DESCRIPTOR_CONFIG.get("descriptor_output_folder", "04_structural_descriptors_split"))
    out.mkdir(parents=True, exist_ok=True)
    return out


def _candidate_id_from_row(row):
    if isinstance(row, dict):
        return str(row.get("candidate_id", row.get("id", "unknown_candidate")))
    return str(getattr(row, "candidate_id", "unknown_candidate"))


def _generator_type_from_row(row):
    if isinstance(row, dict):
        p = _json_load_maybe(row.get("parameter_json", ""))
        return str(row.get("generator_type", p.get("generator_type", "unknown")))
    return str(getattr(row, "generator_type", "unknown"))


def _get_parameter_dict(row):
    if isinstance(row, dict):
        p = _json_load_maybe(row.get("parameter_json", ""))
        # Some rows may already be flattened candidate parameter rows.
        for k in ["candidate_id", "generator_type", "size_mm", "seed", "strut_radius_mm", "strut_diameter_mm"]:
            if k in row and k not in p:
                p[k] = row[k]
        # Candidate/progress tables may not carry size_mm; fall back to global config.
        if "size_mm" not in p and "SIZE_MM" in globals():
            p["size_mm"] = globals().get("SIZE_MM")
        if "size_mm" not in p and "USER_STRUCTURE_CONFIG" in globals():
            try:
                p["size_mm"] = USER_STRUCTURE_CONFIG.get("size_mm", p.get("size_mm", np.nan))
            except Exception:
                pass
        return p
    return {}


def _resolve_candidate_stl_path(row):
    """Resolve STL path from progress table, candidate table, or OUTPUT_ROOT folder."""
    if isinstance(row, dict):
        path = str(row.get("stl_path", "") or "")
        if path and Path(path).exists():
            return Path(path)
        # If the table came from another PC, rebuild path from current OUTPUT_ROOT.
        cid = _candidate_id_from_row(row)
        g = _generator_type_from_row(row)
        root = Path(globals().get("OUTPUT_ROOT", Path.cwd()))
        guess = root / "02_geometry" / g / "STL" / f"{cid}.stl"
        if guess.exists():
            return guess
        # Search below OUTPUT_ROOT as a final fallback.
        found = list(root.glob(f"**/{cid}.stl"))
        if found:
            return found[0]
        return Path(path) if path else guess
    return Path("")


def _get_descriptor_input_table():
    """Build the best available descriptor input table.
    Priority:
    1) generation_results_df/progress_df/desc source already in memory
    2) CSV files inside OUTPUT_ROOT
    3) CANDIDATE_DF/candidate_df in memory
    """
    for name in ["generation_results_df", "progress_df", "PROGRESS_DF", "candidate_df", "CANDIDATE_DF"]:
        obj = globals().get(name, None)
        if isinstance(obj, pd.DataFrame) and not obj.empty:
            return obj.copy()

    root = Path(globals().get("OUTPUT_ROOT", Path.cwd()))
    paths = [
        root / str(DESCRIPTOR_CONFIG.get("progress_table_name", "progress_log.csv")),
        root / "03_tables" / str(DESCRIPTOR_CONFIG.get("progress_table_name", "progress_log.csv")),
        root / "03_tables" / str(DESCRIPTOR_CONFIG.get("candidate_table_name", "candidate_table.csv")),
        root / str(DESCRIPTOR_CONFIG.get("candidate_table_name", "candidate_table.csv")),
    ]
    for p in paths:
        if p.exists():
            try:
                return pd.read_csv(p)
            except Exception:
                pass

    raise RuntimeError("No candidate/progress table found. Run generation cells first or set CANDIDATE_DF/progress_df.")


def _load_binary_stl_as_mesh(path):
    """Load binary STL written by write_binary_stl. Returns unique vertices and faces."""
    path = Path(path)
    with open(path, "rb") as f:
        header = f.read(80)
        n = struct.unpack("<I", f.read(4))[0]
        dtype = np.dtype([
            ("normal", "<f4", (3,)),
            ("vertices", "<f4", (3, 3)),
            ("attr", "<u2"),
        ])
        records = np.fromfile(f, dtype=dtype, count=n)
    tri = records["vertices"].astype(np.float64)
    flat = tri.reshape(-1, 3)
    rounded = np.round(flat, 8)
    uniq, inv = np.unique(rounded, axis=0, return_inverse=True)
    faces = inv.reshape(-1, 3).astype(np.int64)
    return uniq.astype(np.float64), faces


def _load_ascii_stl_as_mesh(path):
    verts = []
    faces = []
    current = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            s = line.strip().split()
            if len(s) == 4 and s[0].lower() == "vertex":
                current.append([float(s[1]), float(s[2]), float(s[3])])
                if len(current) == 3:
                    base = len(verts)
                    verts.extend(current)
                    faces.append([base, base + 1, base + 2])
                    current = []
    if not verts:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    flat = np.asarray(verts, dtype=float)
    rounded = np.round(flat, 8)
    uniq, inv = np.unique(rounded, axis=0, return_inverse=True)
    return uniq, inv.reshape(-1, 3).astype(int)


def load_stl_mesh_arrays(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"STL not found: {path}")
    # Try binary first. If header/count are inconsistent, fallback to ASCII/trimesh.
    try:
        V, F = _load_binary_stl_as_mesh(path)
        if len(V) > 0 and len(F) > 0:
            return V, F
    except Exception:
        pass
    try:
        V, F = _load_ascii_stl_as_mesh(path)
        if len(V) > 0 and len(F) > 0:
            return V, F
    except Exception:
        pass
    if TRIMESH_AVAILABLE:
        m = trimesh.load_mesh(str(path), force="mesh", process=False)
        return np.asarray(m.vertices, dtype=float), np.asarray(m.faces, dtype=int)
    raise RuntimeError(f"Could not load STL: {path}")


def _face_geometry(V, F):
    tri = V[F]
    v0 = tri[:, 1] - tri[:, 0]
    v1 = tri[:, 2] - tri[:, 0]
    n = np.cross(v0, v1)
    dbl_area = np.linalg.norm(n, axis=1)
    area = 0.5 * dbl_area
    normals = np.zeros_like(n)
    ok = dbl_area > 1e-16
    normals[ok] = n[ok] / dbl_area[ok, None]
    centers = tri.mean(axis=1)
    return tri, area, normals, centers


def _surface_area(V, F):
    _, area, _, _ = _face_geometry(V, F)
    return float(np.sum(area))


def _mesh_volume_signed(V, F):
    tri = V[F]
    vol = np.einsum("ij,ij->i", tri[:, 0], np.cross(tri[:, 1], tri[:, 2])) / 6.0
    return float(np.sum(vol))


def _sample_surface_points(V, F, n_points=3000, rng=None):
    rng = np.random.default_rng(1234) if rng is None else rng
    tri, area, normals, _ = _face_geometry(V, F)
    total = np.sum(area)
    if total <= 0 or len(tri) == 0:
        return np.empty((0, 3)), np.empty((0,)), np.empty((0, 3))
    prob = area / total
    idx = rng.choice(len(tri), size=int(n_points), replace=True, p=prob)
    r1 = np.sqrt(rng.random(len(idx)))
    r2 = rng.random(len(idx))
    pts = ((1 - r1)[:, None] * tri[idx, 0] + (r1 * (1 - r2))[:, None] * tri[idx, 1] + (r1 * r2)[:, None] * tri[idx, 2])
    return pts, area[idx], normals[idx]


def _normalize_points_by_bbox(points, bbox_min=None, bbox_max=None):
    P = np.asarray(points, dtype=float)
    if P.size == 0:
        return P
    if bbox_min is None:
        bbox_min = np.nanmin(P, axis=0)
    if bbox_max is None:
        bbox_max = np.nanmax(P, axis=0)
    center = 0.5 * (bbox_min + bbox_max)
    length = np.maximum(bbox_max - bbox_min, 1e-12)
    return (P - center) / length


def _point_distribution_stats(points_norm, prefix, local_abs=0.10):
    """Parameter_distribution_New.py-compatible X/Y/Z/XY/XZ/YZ/XYZ mean+stdev."""
    P = np.asarray(points_norm, dtype=float)
    out = {}
    if P.size == 0:
        for scope in ["global", "local"]:
            for name in ["x", "y", "z", "xy", "xz", "yz", "xyz"]:
                out[f"{prefix}_{scope}_{name}_avg"] = np.nan
                out[f"{prefix}_{scope}_{name}_stdev"] = np.nan
            out[f"{prefix}_{scope}_point_count"] = 0
        return out

    def fill(scope, Q):
        vals = {
            "x": np.abs(Q[:, 0]),
            "y": np.abs(Q[:, 1]),
            "z": np.abs(Q[:, 2]),
            "xy": np.sqrt(Q[:, 0] ** 2 + Q[:, 1] ** 2),
            "xz": np.sqrt(Q[:, 0] ** 2 + Q[:, 2] ** 2),
            "yz": np.sqrt(Q[:, 1] ** 2 + Q[:, 2] ** 2),
            "xyz": np.sqrt(np.sum(Q ** 2, axis=1)),
        }
        for name, arr in vals.items():
            out[f"{prefix}_{scope}_{name}_avg"] = float(np.mean(arr)) if len(arr) else np.nan
            out[f"{prefix}_{scope}_{name}_stdev"] = float(np.std(arr)) if len(arr) else np.nan
        out[f"{prefix}_{scope}_point_count"] = int(len(Q))

    fill("global", P)
    mask_local = np.all(np.abs(P) <= float(local_abs), axis=1)
    fill("local", P[mask_local])

    if len(P) >= 3:
        C = np.cov(P.T)
        eig = np.sort(np.linalg.eigvalsh(C))[::-1]
        out[f"{prefix}_cov_eig1"] = float(eig[0])
        out[f"{prefix}_cov_eig2"] = float(eig[1])
        out[f"{prefix}_cov_eig3"] = float(eig[2])
        out[f"{prefix}_anisotropy_eig1_over_eig3"] = float(eig[0] / (eig[2] + 1e-16))
        out[f"{prefix}_planarity_eig2_minus_eig3_over_eig1"] = float((eig[1] - eig[2]) / (eig[0] + 1e-16))
    else:
        for k in ["cov_eig1", "cov_eig2", "cov_eig3", "anisotropy_eig1_over_eig3", "planarity_eig2_minus_eig3_over_eig1"]:
            out[f"{prefix}_{k}"] = np.nan
    return out

print(f"Loaded descriptor core: {DESCRIPTOR_SCHEMA_VERSION}")
print(f"cv2={CV2_AVAILABLE}, scipy={SCIPY_DESCRIPTOR_AVAILABLE}, trimesh={TRIMESH_AVAILABLE}")


In [ ]:

# ============================================================
# Cell 8B. Point-based descriptors — fixed interior point generation
# ============================================================
# Implements Parameter_distribution_New.py logic, but creates three point sets:
#   1) surface points: area-weighted approximately uniform surface sampling
#   2) interior points: exact mesh.contains when possible, then voxel centers, then inward-offset proxy
#   3) mass points: interior points when available; otherwise surface+inward proxy
# The previous version returned zero interior points for many generated STLs because trimesh.contains
# can fail when rtree is unavailable or when the mesh is not perfectly watertight.


def _make_trimesh_object(V, F, process=False):
    if not TRIMESH_AVAILABLE:
        return None
    try:
        return trimesh.Trimesh(vertices=np.asarray(V, dtype=float), faces=np.asarray(F, dtype=int), process=process)
    except Exception:
        return None


def _limit_points_evenly(points, max_points=8000, rng=None):
    P = np.asarray(points, dtype=float)
    if len(P) == 0:
        return P
    max_points = int(max_points)
    if len(P) <= max_points:
        return P
    rng = np.random.default_rng(1234) if rng is None else rng
    # Fast spatially balanced downsampling: random pre-sample + farthest-point-like refinement.
    pre_n = min(len(P), max(max_points * 4, max_points))
    pre_idx = rng.choice(len(P), size=pre_n, replace=False)
    Q = P[pre_idx]
    chosen = [int(rng.integers(0, len(Q)))]
    dist2 = np.sum((Q - Q[chosen[0]]) ** 2, axis=1)
    target = min(max_points, len(Q))
    # cap the farthest loop for speed; if max_points is large, random-balanced is enough.
    if target <= 1500:
        for _ in range(1, target):
            j = int(np.argmax(dist2))
            chosen.append(j)
            d = np.sum((Q - Q[j]) ** 2, axis=1)
            dist2 = np.minimum(dist2, d)
        return Q[np.asarray(chosen, dtype=int)]
    return Q[rng.choice(len(Q), size=target, replace=False)]


def _sample_interior_points_by_contains(V, F, grid_n=32, max_points=8000, rng=None):
    rng = np.random.default_rng(1234) if rng is None else rng
    tm = _make_trimesh_object(V, F, process=True)
    if tm is None:
        return np.empty((0, 3), dtype=float), "contains_unavailable"
    bbox_min = np.min(V, axis=0)
    bbox_max = np.max(V, axis=0)
    grid_n = int(max(6, grid_n))
    axes = [np.linspace(bbox_min[i], bbox_max[i], grid_n) for i in range(3)]
    X, Y, Z = np.meshgrid(*axes, indexing="ij")
    pts = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
    try:
        inside = np.asarray(tm.contains(pts), dtype=bool)
    except Exception as e:
        return np.empty((0, 3), dtype=float), f"contains_failed:{type(e).__name__}"
    pts_in = pts[inside]
    return _limit_points_evenly(pts_in, max_points=max_points, rng=rng), "contains"


def _sample_interior_points_by_voxel(V, F, grid_n=64, max_points=8000, rng=None):
    rng = np.random.default_rng(1234) if rng is None else rng
    if not TRIMESH_AVAILABLE:
        return np.empty((0, 3), dtype=float), "voxel_unavailable"
    tm = _make_trimesh_object(V, F, process=True)
    if tm is None:
        return np.empty((0, 3), dtype=float), "voxel_mesh_failed"
    bbox_min = np.min(V, axis=0)
    bbox_max = np.max(V, axis=0)
    ext = np.maximum(bbox_max - bbox_min, 1e-12)
    try:
        pitch = float(np.max(ext) / max(16, int(grid_n))) * float(DESCRIPTOR_CONFIG.get("voxelize_pitch_factor", 1.0))
        vg = tm.voxelized(pitch)
        try:
            vg = vg.fill()
        except Exception:
            pass
        pts = np.asarray(vg.points, dtype=float)
        if len(pts) == 0:
            return np.empty((0, 3), dtype=float), "voxel_empty"
        # Remove extreme boundary points when possible to represent interior mass, not just surface.
        if len(pts) > 30 and cKDTree is not None:
            surf_pts, _, _ = _sample_surface_points(V, F, min(5000, max(1000, len(pts)//2)), rng=rng)
            tree = cKDTree(surf_pts)
            d, _ = tree.query(pts, k=1)
            # keep all if filtering would remove everything; otherwise use points away from surface
            thresh = np.percentile(d, 35)
            core = pts[d >= thresh]
            if len(core) >= max(50, min(max_points, 100)):
                pts = core
        return _limit_points_evenly(pts, max_points=max_points, rng=rng), "voxel_filled"
    except Exception as e:
        return np.empty((0, 3), dtype=float), f"voxel_failed:{type(e).__name__}"


def _sample_inward_proxy_points(V, F, n_surface=3000, max_points=8000, rng=None):
    """Fallback for thin-wall/open/non-watertight STLs.
    Creates proxy interior points by moving surface points toward the structure center.
    This prevents blank mass-distribution descriptors while marking the method explicitly.
    """
    rng = np.random.default_rng(1234) if rng is None else rng
    bbox_min = np.min(V, axis=0)
    bbox_max = np.max(V, axis=0)
    center = 0.5 * (bbox_min + bbox_max)
    ext_max = float(np.max(np.maximum(bbox_max - bbox_min, 1e-12)))
    surf_pts, _, surf_normals = _sample_surface_points(V, F, n_surface, rng=rng)
    if len(surf_pts) == 0:
        return np.empty((0, 3), dtype=float), "proxy_empty_surface"
    normals = np.asarray(surf_normals, dtype=float)
    # Orient normals approximately outward by comparing with bbox-center vector.
    orient = np.sign(np.sum(normals * (surf_pts - center), axis=1))
    orient[orient == 0] = 1.0
    outward = normals * orient[:, None]
    offset0 = float(DESCRIPTOR_CONFIG.get("interior_inward_offset_fraction", 0.018)) * ext_max
    steps = int(max(1, DESCRIPTOR_CONFIG.get("interior_inward_offset_steps", 3)))
    pts = []
    for k in range(1, steps + 1):
        pts.append(surf_pts - outward * (offset0 * k))
    P = np.vstack(pts)
    # Clip to bbox to avoid abnormal offsets from imperfect normals.
    P = np.minimum(np.maximum(P, bbox_min), bbox_max)
    return _limit_points_evenly(P, max_points=max_points, rng=rng), "surface_inward_proxy"


def _sample_interior_points(V, F, grid_n=32, max_points=8000, rng=None):
    """Robust interior point sampler with explicit method/status reporting."""
    rng = np.random.default_rng(1234) if rng is None else rng
    method = str(DESCRIPTOR_CONFIG.get("interior_sampling_method", "auto")).lower()
    diagnostics = []

    if method in ["auto", "contains"]:
        pts, stat = _sample_interior_points_by_contains(V, F, grid_n=grid_n, max_points=max_points, rng=rng)
        diagnostics.append(stat)
        if len(pts) > 0 or method == "contains":
            return pts, stat, diagnostics

    if method in ["auto", "voxel"]:
        pts, stat = _sample_interior_points_by_voxel(V, F, grid_n=max(grid_n, int(DESCRIPTOR_CONFIG.get("voxel_grid_n", 96))), max_points=max_points, rng=rng)
        diagnostics.append(stat)
        if len(pts) > 0 or method == "voxel":
            return pts, stat, diagnostics

    if not bool(DESCRIPTOR_CONFIG.get("interior_require_exact", False)) and method in ["auto", "inward_proxy", "proxy"]:
        pts, stat = _sample_inward_proxy_points(V, F, n_surface=int(DESCRIPTOR_CONFIG.get("surface_point_count", 3000)), max_points=max_points, rng=rng)
        diagnostics.append(stat)
        return pts, stat, diagnostics

    return np.empty((0, 3), dtype=float), "empty", diagnostics


def extract_point_descriptors(V, F, row=None, out_dir=None, rng=None):
    if not DESCRIPTOR_CONFIG.get("enable_point_descriptors", True):
        return {"point_descriptor_status": "disabled"}
    rng = np.random.default_rng(12345) if rng is None else rng
    out = {"point_descriptor_status": "ok"}
    try:
        bbox_min = np.min(V, axis=0)
        bbox_max = np.max(V, axis=0)
        n_surface = int(DESCRIPTOR_CONFIG.get("surface_point_count", 3000))
        surf_pts, surf_w, surf_normals = _sample_surface_points(V, F, n_surface, rng=rng)
        surf_norm = _normalize_points_by_bbox(surf_pts, bbox_min, bbox_max)
        out.update(_point_distribution_stats(surf_norm, "point_surface", DESCRIPTOR_CONFIG.get("local_range_abs_norm", 0.10)))
        out["point_surface_count"] = int(len(surf_pts))

        interior_pts, interior_method, interior_diag = _sample_interior_points(
            V, F,
            grid_n=int(DESCRIPTOR_CONFIG.get("interior_grid_n", 32)),
            max_points=int(DESCRIPTOR_CONFIG.get("interior_point_max", 8000)),
            rng=rng,
        )
        out["point_interior_sampling_method"] = interior_method
        out["point_interior_sampling_diagnostics"] = " | ".join(map(str, interior_diag))
        out["point_interior_count"] = int(len(interior_pts))
        interior_norm = _normalize_points_by_bbox(interior_pts, bbox_min, bbox_max)
        out.update(_point_distribution_stats(interior_norm, "point_interior", DESCRIPTOR_CONFIG.get("local_range_abs_norm", 0.10)))

        # Combined mass distribution: surface + interior/proxy points.
        if len(interior_pts):
            mass_pts = np.vstack([surf_pts, interior_pts]) if len(surf_pts) else interior_pts
        else:
            mass_pts = surf_pts
        mass_norm = _normalize_points_by_bbox(mass_pts, bbox_min, bbox_max)
        out.update(_point_distribution_stats(mass_norm, "point_mass", DESCRIPTOR_CONFIG.get("local_range_abs_norm", 0.10)))
        out["point_mass_count"] = int(len(mass_pts))

        # Additional point-cloud descriptors: radial shell occupancy and quadrant balance.
        if len(mass_norm):
            r = np.sqrt(np.sum(mass_norm ** 2, axis=1))
            out.update(_stats(r, "point_mass_radial"))
            signs = (mass_norm >= 0).astype(int)
            oct_id = signs[:, 0] * 4 + signs[:, 1] * 2 + signs[:, 2]
            counts = np.bincount(oct_id, minlength=8).astype(float)
            probs = counts / (counts.sum() + 1e-16)
            out["point_mass_octant_entropy"] = float(-np.sum(probs[probs > 0] * np.log(probs[probs > 0])))
            out["point_mass_octant_imbalance"] = float(np.max(probs) - np.min(probs))

        if DESCRIPTOR_CONFIG.get("save_point_cloud_csv", False) and out_dir is not None:
            cid = _candidate_id_from_row(row or {})
            pc_dir = Path(out_dir) / "point_clouds"
            pc_dir.mkdir(parents=True, exist_ok=True)
            if len(surf_pts):
                pd.DataFrame(surf_pts, columns=["x", "y", "z"]).to_csv(pc_dir / f"{cid}_surface_points.csv", index=False)
            if len(interior_pts):
                pd.DataFrame(interior_pts, columns=["x", "y", "z"]).to_csv(pc_dir / f"{cid}_interior_points.csv", index=False)
    except Exception as e:
        out["point_descriptor_status"] = "failed"
        out["point_descriptor_error"] = repr(e)
    return out


In [ ]:

# ============================================================
# Cell 8C. Surface descriptors: surface area, compactness, curvature
# ============================================================
# Implements DDG/mixed-Voronoi style surface curvature and area-weighted stats
# inspired by Curvature Extraction_New.py.


def _mesh_edges(F):
    E = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]])
    E = np.sort(E, axis=1)
    E = np.unique(E, axis=0)
    return E


def _cot_angle(u, v):
    cross = np.linalg.norm(np.cross(u, v))
    if cross < 1e-16:
        return 0.0
    return float(np.dot(u, v) / cross)


def _triangle_angles(v0, v1, v2):
    a = np.linalg.norm(v1 - v2)
    b = np.linalg.norm(v2 - v0)
    c = np.linalg.norm(v0 - v1)
    alpha = np.arccos(np.clip((b*b + c*c - a*a) / (2*b*c + 1e-16), -1.0, 1.0))
    beta  = np.arccos(np.clip((c*c + a*a - b*b) / (2*c*a + 1e-16), -1.0, 1.0))
    gamma = np.pi - alpha - beta
    return alpha, beta, gamma, a, b, c


def _mixed_voronoi_areas(V, F):
    tri, face_area, _, _ = _face_geometry(V, F)
    Av = np.zeros(len(V), dtype=float)
    for f, (i0, i1, i2) in enumerate(F):
        v0, v1, v2 = V[i0], V[i1], V[i2]
        alpha, beta, gamma, a, b, c = _triangle_angles(v0, v1, v2)
        obtuse = (alpha > np.pi/2) or (beta > np.pi/2) or (gamma > np.pi/2)
        Af = face_area[f]
        if not obtuse:
            cot_alpha = 1 / np.tan(alpha)
            cot_beta = 1 / np.tan(beta)
            cot_gamma = 1 / np.tan(gamma)
            A0 = (b*b * cot_gamma + c*c * cot_beta) / 8.0
            A1 = (c*c * cot_alpha + a*a * cot_gamma) / 8.0
            A2 = (a*a * cot_beta + b*b * cot_alpha) / 8.0
        else:
            if alpha > np.pi/2:
                A0, A1, A2 = Af/2, Af/4, Af/4
            elif beta > np.pi/2:
                A0, A1, A2 = Af/4, Af/2, Af/4
            else:
                A0, A1, A2 = Af/4, Af/4, Af/2
        Av[i0] += A0; Av[i1] += A1; Av[i2] += A2
    return np.maximum(Av, 1e-16)


def _vertex_angle_sums(V, F):
    sums = np.zeros(len(V), dtype=float)
    for i, j, k in F:
        vi, vj, vk = V[i], V[j], V[k]
        a, b, c, *_ = _triangle_angles(vi, vj, vk)
        sums[i] += a; sums[j] += b; sums[k] += c
    return sums


def _boundary_vertices(F, nV):
    E = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]])
    E = np.sort(E, axis=1)
    E_unique, counts = np.unique(E, axis=0, return_counts=True)
    boundary_edges = E_unique[counts == 1]
    b = np.zeros(nV, dtype=bool)
    if len(boundary_edges):
        b[np.unique(boundary_edges.ravel())] = True
    return b


def _surface_curvature_ddg(V, F):
    Av = _mixed_voronoi_areas(V, F)
    Hn = np.zeros((len(V), 3), dtype=float)
    for i, j, k in F:
        vi, vj, vk = V[i], V[j], V[k]
        # Standard cotangent Laplacian contribution per opposite angle.
        cot_k = _cot_angle(vi - vk, vj - vk)
        cot_j = _cot_angle(vi - vj, vk - vj)
        cot_i = _cot_angle(vj - vi, vk - vi)
        Hn[i] += cot_k * (vj - vi) + cot_j * (vk - vi)
        Hn[j] += cot_i * (vk - vj) + cot_k * (vi - vj)
        Hn[k] += cot_j * (vi - vk) + cot_i * (vj - vk)
    Hn = 0.5 * Hn / Av[:, None]
    H = 0.5 * np.linalg.norm(Hn, axis=1)
    angle_sum = _vertex_angle_sums(V, F)
    boundary = _boundary_vertices(F, len(V))
    full_angle = np.where(boundary, np.pi, 2*np.pi)
    K = (full_angle - angle_sum) / Av
    return H, K, Av


def extract_surface_descriptors(V, F, row=None, out_dir=None):
    if not DESCRIPTOR_CONFIG.get("enable_surface_descriptors", True):
        return {"surface_descriptor_status": "disabled"}
    out = {"surface_descriptor_status": "ok"}
    try:
        bbox_min = np.min(V, axis=0)
        bbox_max = np.max(V, axis=0)
        ext = np.maximum(bbox_max - bbox_min, 1e-12)
        area = _surface_area(V, F)
        vol_signed = _mesh_volume_signed(V, F)
        vol = abs(vol_signed)
        bbox_vol = float(np.prod(ext))
        out.update({
            "surface_area": area,
            "surface_volume_abs": vol,
            "surface_volume_signed": vol_signed,
            "surface_area_to_volume": area / (vol + 1e-16),
            "surface_area_to_bbox_volume": area / (bbox_vol + 1e-16),
            "relative_density_from_mesh_volume": vol / (bbox_vol + 1e-16),
            "bbox_x": float(ext[0]),
            "bbox_y": float(ext[1]),
            "bbox_z": float(ext[2]),
            "bbox_volume": bbox_vol,
            "compactness_sphericity": (np.pi ** (1/3) * (6 * vol) ** (2/3) / (area + 1e-16)) if vol > 0 else np.nan,
        })
        E = _mesh_edges(F)
        edge_lengths = np.linalg.norm(V[E[:, 0]] - V[E[:, 1]], axis=1) if len(E) else np.array([])
        out.update(_stats(edge_lengths, "surface_edge_length"))
        tri, face_area, normals, centers = _face_geometry(V, F)
        out.update(_stats(face_area, "surface_triangle_area"))
        # Normal/orientation descriptors: mean absolute normal components.
        if len(normals):
            for ax, idx in zip(["x", "y", "z"], [0, 1, 2]):
                out[f"surface_normal_abs_{ax}_avg"] = float(np.mean(np.abs(normals[:, idx])))
                out[f"surface_normal_abs_{ax}_stdev"] = float(np.std(np.abs(normals[:, idx])))

        if DESCRIPTOR_CONFIG.get("enable_surface_curvature", True):
            if len(V) <= int(DESCRIPTOR_CONFIG.get("max_vertices_for_curvature", 80000)):
                H, K, Av = _surface_curvature_ddg(V, F)
                clip_p = float(DESCRIPTOR_CONFIG.get("curvature_clip_percentile", 99.0))
                for name, arr in [("mean_curvature", H), ("gaussian_curvature", K)]:
                    finite = arr[np.isfinite(arr)]
                    if len(finite):
                        lim = np.percentile(np.abs(finite), clip_p)
                        arr_clip = np.clip(arr, -lim, lim)
                    else:
                        arr_clip = arr
                    out.update(_stats(arr_clip, f"surface_{name}", weights=Av))
                    out[f"surface_{name}_abs_p99_clip"] = float(np.percentile(np.abs(arr_clip[np.isfinite(arr_clip)]), 99)) if np.any(np.isfinite(arr_clip)) else np.nan
                out["surface_curvature_vertex_count"] = int(len(V))
                if DESCRIPTOR_CONFIG.get("save_curvature_raw_csv", False) and out_dir is not None:
                    cid = _candidate_id_from_row(row or {})
                    cdir = Path(out_dir) / "curvature_raw"
                    cdir.mkdir(parents=True, exist_ok=True)
                    pd.DataFrame({"A_v": Av, "H_ddg": H, "K_ddg": K}).to_csv(cdir / f"{cid}_curvature_raw.csv", index=False)
            else:
                out["surface_curvature_status"] = "skipped_too_many_vertices"
                out["surface_curvature_vertex_count"] = int(len(V))
    except Exception as e:
        out["surface_descriptor_status"] = "failed"
        out["surface_descriptor_error"] = repr(e)
    return out


In [ ]:

# ============================================================
# Cell 8D. Slice-pair image descriptors
# ============================================================
# Implements Nth/(N+1)th merge as requested:
#   overlap -> black/0, N-only -> white/255, N+1-only -> gray/122.
# Internally, descriptors use red/blue/purple-equivalent masks:
#   red=N-only, blue=N+1-only, purple=overlap.
# It exports Mass orientation, Curvature, Thickness, Angle, Perimeter-to-area
# using IP/LIP/LTP aggregation concepts from the reference PPT and scripts.


def _resize_bool_nearest(arr, target_shape):
    arr = np.asarray(arr, dtype=bool)
    target_shape = tuple(int(x) for x in target_shape)
    if arr.shape == target_shape:
        return arr
    if ndi_zoom is None:
        # Crude fallback using index mapping.
        grids = [np.linspace(0, arr.shape[i] - 1, target_shape[i]).round().astype(int) for i in range(arr.ndim)]
        if arr.ndim == 3:
            return arr[np.ix_(grids[0], grids[1], grids[2])]
        return arr[np.ix_(grids[0], grids[1])]
    factors = [target_shape[i] / arr.shape[i] for i in range(arr.ndim)]
    return ndi_zoom(arr.astype(np.uint8), factors, order=0) > 0


def _voxelize_mesh_to_mask(V, F, grid_n=96):
    """Voxelize STL mesh for slice descriptors. Uses trimesh voxelization when available."""
    grid_n = int(max(16, grid_n))
    bbox_min = np.min(V, axis=0)
    bbox_max = np.max(V, axis=0)
    ext = np.maximum(bbox_max - bbox_min, 1e-12)
    if TRIMESH_AVAILABLE:
        try:
            tm = trimesh.Trimesh(vertices=V, faces=F, process=True)
            pitch = float(np.max(ext) / grid_n) * float(DESCRIPTOR_CONFIG.get("voxelize_pitch_factor", 1.0))
            vg = tm.voxelized(pitch)
            try:
                vg = vg.fill()
            except Exception:
                pass
            mat = np.asarray(vg.matrix, dtype=bool)
            if mat.ndim == 3 and np.count_nonzero(mat) > 0:
                return _resize_bool_nearest(mat, (grid_n, grid_n, grid_n))
        except Exception:
            pass
    # Fallback: use surface vertices only and thicken. This is less exact but prevents empty descriptor tables.
    Pn = _normalize_points_by_bbox(V, bbox_min, bbox_max) + 0.5
    idx = np.clip(np.floor(Pn * (grid_n - 1)).astype(int), 0, grid_n - 1)
    mask = np.zeros((grid_n, grid_n, grid_n), dtype=bool)
    mask[idx[:, 0], idx[:, 1], idx[:, 2]] = True
    if binary_fill_holes is not None:
        for ax in range(3):
            mask = np.swapaxes(mask, 0, ax)
            for i in range(mask.shape[0]):
                mask[i] = binary_fill_holes(mask[i])
            mask = np.swapaxes(mask, 0, ax)
    return mask


def _orthonormal_basis_from_normal(n):
    n = np.asarray(n, dtype=float)
    n = n / (np.linalg.norm(n) + 1e-16)
    ref = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(ref, n)) > 0.9:
        ref = np.array([0.0, 1.0, 0.0])
    u = np.cross(n, ref)
    u = u / (np.linalg.norm(u) + 1e-16)
    v = np.cross(n, u)
    v = v / (np.linalg.norm(v) + 1e-16)
    return u, v, n


def _make_oriented_slices_from_mask(mask, axis_name="z", slice_count=100, img_n=160):
    """Project solid voxel centers into an oriented slicing coordinate system."""
    coords = np.argwhere(np.asarray(mask, dtype=bool))
    if len(coords) == 0:
        return []
    # Normalize voxel centers to [-0.5, 0.5].
    shape = np.array(mask.shape, dtype=float)
    P = (coords + 0.5) / shape - 0.5
    u, v, n = _orthonormal_basis_from_normal(_AXIS_VEC[axis_name])
    t = P @ n
    a = P @ u
    b = P @ v
    t_edges = np.linspace(t.min() - 1e-9, t.max() + 1e-9, int(slice_count) + 1)
    a_edges = np.linspace(a.min() - 1e-9, a.max() + 1e-9, int(img_n) + 1)
    b_edges = np.linspace(b.min() - 1e-9, b.max() + 1e-9, int(img_n) + 1)
    sidx = np.clip(np.searchsorted(t_edges, t, side="right") - 1, 0, int(slice_count) - 1)
    ia = np.clip(np.searchsorted(a_edges, a, side="right") - 1, 0, int(img_n) - 1)
    ib = np.clip(np.searchsorted(b_edges, b, side="right") - 1, 0, int(img_n) - 1)
    slices = [np.zeros((int(img_n), int(img_n)), dtype=bool) for _ in range(int(slice_count))]
    for s, x, y in zip(sidx, ia, ib):
        slices[int(s)][int(y), int(x)] = True
    # Fill small holes in each slice when scipy is available.
    if binary_fill_holes is not None:
        slices = [binary_fill_holes(sl).astype(bool) if np.any(sl) else sl for sl in slices]
    return slices


def _connected_components_2d(mask):
    mask = np.asarray(mask, dtype=bool)
    if CV2_AVAILABLE:
        num, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
        return labels, num - 1
    if ndi_label is not None:
        labels, num = ndi_label(mask)
        return labels, int(num)
    return mask.astype(int), int(np.any(mask))


def _perimeter_pixels(mask):
    mask = np.asarray(mask, dtype=bool)
    if not np.any(mask):
        return 0.0
    if CV2_AVAILABLE:
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        return float(sum(cv2.arcLength(c, True) for c in contours))
    # Fallback: count exposed 4-neighbor edges.
    padded = np.pad(mask, 1)
    c = padded[1:-1, 1:-1]
    return float(np.sum(c & ~padded[:-2, 1:-1]) + np.sum(c & ~padded[2:, 1:-1]) + np.sum(c & ~padded[1:-1, :-2]) + np.sum(c & ~padded[1:-1, 2:]))


def _contact_edge_length_pixels(mask_a, mask_b):
    mask_a = np.asarray(mask_a, dtype=bool)
    mask_b = np.asarray(mask_b, dtype=bool)
    if not np.any(mask_a) or not np.any(mask_b):
        return 0.0
    if CV2_AVAILABLE:
        kernel = np.ones((3, 3), np.uint8)
        dil = cv2.dilate(mask_a.astype(np.uint8), kernel, iterations=1).astype(bool)
        return float(np.count_nonzero(dil & mask_b))
    # Fallback dilation by shifts.
    pad = np.pad(mask_a, 1)
    dil = np.zeros_like(mask_a)
    for dy in [-1, 0, 1]:
        for dx in [-1, 0, 1]:
            dil |= pad[1+dy:1+dy+mask_a.shape[0], 1+dx:1+dx+mask_a.shape[1]]
    return float(np.count_nonzero(dil & mask_b))


def _slice_pair_descriptors(binary_a, binary_b, area_per_pixel=1.0, length_per_pixel=1.0, layer_height=1.0, min_pixels=2):
    red = np.asarray(binary_a, dtype=bool) & ~np.asarray(binary_b, dtype=bool)
    blue = np.asarray(binary_b, dtype=bool) & ~np.asarray(binary_a, dtype=bool)
    purple = np.asarray(binary_a, dtype=bool) & np.asarray(binary_b, dtype=bool)
    combined = red | blue | purple
    labels, nlab = _connected_components_2d(combined)

    comp_values = {"mass_orientation": [], "curvature": [], "thickness": [], "angle": [], "perimeter_to_area": []}
    comp_weights = []
    layer_total = {"mass_orientation": np.nan, "curvature": np.nan, "thickness": np.nan, "angle": np.nan, "perimeter_to_area": np.nan}

    total_red = float(np.count_nonzero(red)) * area_per_pixel
    total_blue = float(np.count_nonzero(blue)) * area_per_pixel
    total_purple = float(np.count_nonzero(purple)) * area_per_pixel
    denom_total = 0.5 * total_red + 0.5 * total_blue + total_purple
    if denom_total > 0:
        layer_total["mass_orientation"] = total_purple / denom_total
        layer_total["thickness"] = math.sqrt(max(total_red + total_purple, 0.0))
        pta_total = _perimeter_pixels(red | purple) * length_per_pixel / ((total_red + total_purple) + 1e-16)
        layer_total["perimeter_to_area"] = pta_total
        # LTP curvature/angle from total areas and total contacts.
        arc_r = _contact_edge_length_pixels(red, purple) * length_per_pixel
        arc_b = _contact_edge_length_pixels(blue, purple) * length_per_pixel
        term_r = total_red / arc_r if arc_r > 1e-12 and total_red > 0 else 0.0
        term_b = total_blue / arc_b if arc_b > 1e-12 and total_blue > 0 else 0.0
        denom = term_r + term_b
        if denom > 0:
            layer_total["angle"] = math.degrees(math.atan((2 * layer_height) / denom))
            layer_total["curvature"] = denom / (2 * layer_height)
        else:
            layer_total["angle"] = 90.0
            layer_total["curvature"] = 0.0

    for lab in range(1, nlab + 1):
        lm = labels == lab
        if np.count_nonzero(lm) < int(min_pixels):
            continue
        r = red & lm
        b = blue & lm
        p = purple & lm
        ar = float(np.count_nonzero(r)) * area_per_pixel
        ab = float(np.count_nonzero(b)) * area_per_pixel
        ap = float(np.count_nonzero(p)) * area_per_pixel
        weight = ((ar + ab + 2 * ap) * layer_height) / 2.0
        if weight <= 0:
            continue
        comp_weights.append(weight)
        mass_ori = ap / (0.5 * ar + 0.5 * ab + ap + 1e-16)
        thickness = math.sqrt(max(ar + ap, 0.0))
        arc1 = _contact_edge_length_pixels(r, p) * length_per_pixel
        arc2 = _contact_edge_length_pixels(b, p) * length_per_pixel
        term_r = ar / arc1 if arc1 > 1e-12 and ar > 0 else 0.0
        term_b = ab / arc2 if arc2 > 1e-12 and ab > 0 else 0.0
        denom = term_r + term_b
        if denom > 0:
            angle = math.degrees(math.atan((2 * layer_height) / denom))
            curvature = denom / (2 * layer_height)
        else:
            angle = 90.0
            curvature = 0.0
        area_ra = ar + ap
        pta = (_perimeter_pixels(r | p) * length_per_pixel / (area_ra + 1e-16)) if area_ra > 0 else np.nan
        comp_values["mass_orientation"].append(mass_ori)
        comp_values["thickness"].append(thickness)
        comp_values["angle"].append(angle)
        comp_values["curvature"].append(curvature)
        comp_values["perimeter_to_area"].append(pta)
    return comp_values, np.asarray(comp_weights, dtype=float), layer_total, (red, blue, purple)


def extract_slice_image_descriptors(V, F, row=None, out_dir=None):
    if not DESCRIPTOR_CONFIG.get("enable_slice_descriptors", True):
        return {"slice_descriptor_status": "disabled"}
    out = {"slice_descriptor_status": "ok"}
    try:
        grid_n = int(DESCRIPTOR_CONFIG.get("voxel_grid_n", 96))
        mask3d = _voxelize_mesh_to_mask(V, F, grid_n=grid_n)
        solid_voxels = int(np.count_nonzero(mask3d))
        out["slice_voxel_solid_count"] = solid_voxels
        out["slice_voxel_relative_density"] = float(np.mean(mask3d)) if mask3d.size else np.nan
        if solid_voxels == 0:
            out["slice_descriptor_status"] = "failed_empty_voxel_mask"
            return out
        size_mm = _safe_float(_get_parameter_dict(row or {}).get("size_mm", np.nan), default=np.nan)
        if not np.isfinite(size_mm):
            bbox = np.max(V, axis=0) - np.min(V, axis=0)
            size_mm = float(np.max(bbox))
        slice_count = int(DESCRIPTOR_CONFIG.get("slice_count", 100))
        img_n = int(DESCRIPTOR_CONFIG.get("slice_image_size", 160))
        layer_h = DESCRIPTOR_CONFIG.get("layer_height_mm", None)
        layer_h = float(layer_h) if layer_h is not None else float(size_mm / max(1, slice_count - 1))
        area_per_pixel = (size_mm * size_mm) / float(img_n * img_n)
        length_per_pixel = size_mm / float(img_n)
        min_pixels = int(DESCRIPTOR_CONFIG.get("min_component_pixels", 2))

        save_images = bool(DESCRIPTOR_CONFIG.get("save_slice_merge_images", False)) and out_dir is not None
        cid = _candidate_id_from_row(row or {})
        img_root = Path(out_dir) / "slice_pair_merge_images" / cid if save_images else None
        if img_root is not None:
            img_root.mkdir(parents=True, exist_ok=True)

        for axis_name in DESCRIPTOR_CONFIG.get("slice_axes", ["z"]):
            slices = _make_oriented_slices_from_mask(mask3d, axis_name=axis_name, slice_count=slice_count, img_n=img_n)
            all_ip = defaultdict(list)
            all_w = defaultdict(list)
            lip_values = defaultdict(list)
            ltp_values = defaultdict(list)
            valid_pairs = 0
            step = int(DESCRIPTOR_CONFIG.get("slice_pair_step", 1))
            max_save = int(DESCRIPTOR_CONFIG.get("slice_image_max_save_per_axis", 12))
            for i in range(0, len(slices) - step, step):
                comp, weights, ltp, masks = _slice_pair_descriptors(
                    slices[i], slices[i + step], area_per_pixel=area_per_pixel,
                    length_per_pixel=length_per_pixel, layer_height=layer_h, min_pixels=min_pixels,
                )
                if len(weights) == 0 and not any(np.isfinite(list(ltp.values()))):
                    continue
                valid_pairs += 1
                for key, vals in comp.items():
                    vals = np.asarray(vals, dtype=float)
                    if len(vals):
                        all_ip[key].extend(vals.tolist())
                        all_w[key].extend(weights[:len(vals)].tolist())
                        lip_values[key].append(float(np.nanmean(vals)))
                    if np.isfinite(ltp.get(key, np.nan)):
                        ltp_values[key].append(float(ltp[key]))
                if save_images and i < max_save:
                    red, blue, purple = masks
                    # Requested grayscale merge: overlap=0, N-only=255, N+1-only=122.
                    img = np.zeros_like(red, dtype=np.uint8)
                    img[red] = 255
                    img[blue] = 122
                    img[purple] = 0
                    if CV2_AVAILABLE:
                        cv2.imwrite(str(img_root / f"{axis_name}_pair_{i+1:03d}_{i+step+1:03d}.png"), img)
            out[f"slice_{axis_name}_valid_pair_count"] = int(valid_pairs)
            for key in ["mass_orientation", "curvature", "thickness", "angle", "perimeter_to_area"]:
                out.update(_stats(all_ip[key], f"slice_{axis_name}_{key}_IP", weights=all_w[key] if len(all_w[key]) else None))
                out.update(_stats(lip_values[key], f"slice_{axis_name}_{key}_LIP"))
                out.update(_stats(ltp_values[key], f"slice_{axis_name}_{key}_LTP"))
    except Exception as e:
        out["slice_descriptor_status"] = "failed"
        out["slice_descriptor_error"] = repr(e)
    return out


In [ ]:

# ============================================================
# Cell 8E. Lattice node/strut descriptors
# ============================================================
# Implements Extract structure variable.py-style node/strut statistics directly
# from the lattice generator parameter_json. If parameter_json is unavailable,
# it falls back to NaN rather than silently reporting wrong values.


def _key3(p, nd=None):
    nd = int(DESCRIPTOR_CONFIG.get("lattice_coord_key_decimals", 6)) if nd is None else int(nd)
    p = np.asarray(p, dtype=float)
    return tuple(np.round(p, nd).tolist())


def _axis_angles(p1, p2):
    d = np.asarray(p2, dtype=float) - np.asarray(p1, dtype=float)
    L = float(np.linalg.norm(d))
    if L <= 1e-16:
        return L, np.nan, np.nan, np.nan
    ax = math.degrees(math.asin(min(1.0, abs(d[0]) / L)))
    ay = math.degrees(math.asin(min(1.0, abs(d[1]) / L)))
    az = math.degrees(math.asin(min(1.0, abs(d[2]) / L)))
    return L, ax, ay, az


def _weighted_stats_pair(values, weights):
    return _weighted_mean_std(values, weights)


def _extract_lattice_graph_from_params(p):
    """Reconstruct lattice graph using generator functions if available."""
    if str(p.get("generator_type", "")) != "lattice":
        return None, None, "not_lattice"
    if "generate_lattice_nodes" not in globals() or "select_lattice_edges" not in globals():
        return None, None, "missing_generator_functions"
    try:
        nodes = generate_lattice_nodes(p, seed=int(p.get("seed", 0)))
        edges = select_lattice_edges(nodes, p)
        return np.asarray(nodes, dtype=float), list(edges), "ok"
    except Exception as e:
        return None, None, f"failed: {repr(e)}"


def extract_lattice_descriptors(row):
    if not DESCRIPTOR_CONFIG.get("enable_lattice_descriptors", True):
        return {"lattice_descriptor_status": "disabled"}
    p = _get_parameter_dict(row or {})
    out = {"lattice_descriptor_status": "not_lattice"}
    if str(p.get("generator_type", _generator_type_from_row(row or {}))) != "lattice":
        return out

    nodes, edges, status = _extract_lattice_graph_from_params(p)
    out["lattice_descriptor_status"] = status
    if status != "ok" or nodes is None or edges is None:
        return out

    try:
        radius = _safe_float(p.get("strut_radius_mm", np.nan), default=np.nan)
        diameter = _safe_float(p.get("strut_diameter_mm", np.nan), default=np.nan)
        if not np.isfinite(diameter) and np.isfinite(radius):
            diameter = 2 * radius

        struts = []
        for i, j in edges:
            p1, p2 = nodes[int(i)], nodes[int(j)]
            L, ax, ay, az = _axis_angles(p1, p2)
            if L <= 1e-16:
                continue
            lod = L / diameter if np.isfinite(diameter) and diameter > 0 else np.nan
            struts.append({"i": int(i), "j": int(j), "length": L, "angle_x": ax, "angle_y": ay, "angle_z": az, "l_over_d": lod})

        node_to = defaultdict(list)
        for s in struts:
            node_to[s["i"]].append(s)
            node_to[s["j"]].append(s)
        degrees = np.array([len(node_to[i]) for i in range(len(nodes))], dtype=float)
        lengths = np.array([s["length"] for s in struts], dtype=float)
        lods = np.array([s["l_over_d"] for s in struts], dtype=float)

        out.update({
            "lattice_node_count": int(len(nodes)),
            "lattice_strut_count": int(len(struts)),
            "lattice_global_Strut_No_at_Nodes_AVG": float(np.mean(degrees)) if len(degrees) else np.nan,
            "lattice_global_Strut_No_at_Nodes_STDEV": float(np.std(degrees, ddof=1)) if len(degrees) > 1 else 0.0,
            "lattice_global_Length_AVG": float(np.mean(lengths)) if len(lengths) else np.nan,
            "lattice_global_Length_STDEV": float(np.std(lengths, ddof=1)) if len(lengths) > 1 else 0.0,
            "lattice_global_l_over_d_AVG": float(np.nanmean(lods)) if np.any(np.isfinite(lods)) else np.nan,
            "lattice_global_l_over_d_STDEV": float(np.nanstd(lods, ddof=1)) if np.sum(np.isfinite(lods)) > 1 else 0.0,
        })

        for angle_key in ["angle_x", "angle_y", "angle_z"]:
            vals = np.array([s[angle_key] for s in struts], dtype=float)
            mu, sig = _weighted_stats_pair(vals, lengths)
            out[f"lattice_global_Angle_{angle_key[-1].upper()}_weighted_with_length_AVG"] = mu
            out[f"lattice_global_Angle_{angle_key[-1].upper()}_weighted_with_length_STDEV"] = sig

        node_length_avg = []
        node_lod_avg = []
        node_angle_avg = {"x": [], "y": [], "z": []}
        for ni in range(len(nodes)):
            ss = node_to.get(ni, [])
            if not ss:
                continue
            Ls = np.array([s["length"] for s in ss], dtype=float)
            node_length_avg.append(float(np.mean(Ls)))
            lds = np.array([s["l_over_d"] for s in ss], dtype=float)
            if np.any(np.isfinite(lds)):
                node_lod_avg.append(float(np.nanmean(lds)))
            for ax in ["x", "y", "z"]:
                vals = np.array([s[f"angle_{ax}"] for s in ss], dtype=float)
                mu, _ = _weighted_stats_pair(vals, Ls)
                if np.isfinite(mu):
                    node_angle_avg[ax].append(mu)

        out.update({
            "lattice_Node_Length_AVG": float(np.mean(node_length_avg)) if len(node_length_avg) else np.nan,
            "lattice_Node_Length_STDEV": float(np.std(node_length_avg, ddof=1)) if len(node_length_avg) > 1 else 0.0,
            "lattice_Node_l_over_d_AVG": float(np.mean(node_lod_avg)) if len(node_lod_avg) else np.nan,
            "lattice_Node_l_over_d_STDEV": float(np.std(node_lod_avg, ddof=1)) if len(node_lod_avg) > 1 else 0.0,
        })
        for ax in ["x", "y", "z"]:
            arr = np.asarray(node_angle_avg[ax], dtype=float)
            out[f"lattice_Node_Angle_{ax.upper()}_weighted_with_length_AVG"] = float(np.mean(arr)) if len(arr) else np.nan
            out[f"lattice_Node_Angle_{ax.upper()}_weighted_with_length_STDEV"] = float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0

        # Additional graph descriptors useful for mechanical-property modeling.
        if len(nodes) >= 3:
            Pn = _normalize_points_by_bbox(nodes)
            C = np.cov(Pn.T)
            eig = np.sort(np.linalg.eigvalsh(C))[::-1]
            out["lattice_node_spatial_anisotropy"] = float(eig[0] / (eig[-1] + 1e-16))
        out["lattice_degree_min"] = float(np.min(degrees)) if len(degrees) else np.nan
        out["lattice_degree_max"] = float(np.max(degrees)) if len(degrees) else np.nan
        out["lattice_degree_p95"] = float(np.percentile(degrees, 95)) if len(degrees) else np.nan
        out["lattice_length_p95_over_p05"] = float(np.percentile(lengths, 95) / (np.percentile(lengths, 5) + 1e-16)) if len(lengths) else np.nan
    except Exception as e:
        out["lattice_descriptor_status"] = "failed"
        out["lattice_descriptor_error"] = repr(e)
    return out


In [ ]:

# ============================================================
# Cell 8F. Integrated descriptor runner and export
# ============================================================


# ============================================================
# Cell 8F-0. Additional voxel/topology descriptors
# ============================================================

def _count_boundary_and_nonmanifold_edges(F):
    if len(F) == 0:
        return 0, 0, 0
    edges = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]])
    edges = np.sort(edges, axis=1)
    uniq, cnt = np.unique(edges, axis=0, return_counts=True)
    return int(len(uniq)), int(np.sum(cnt == 1)), int(np.sum(cnt > 2))


def _connected_component_stats_3d(mask):
    mask = np.asarray(mask, dtype=bool)
    if ndi_label is None or not np.any(mask):
        return {"component_count": np.nan, "largest_component_fraction": np.nan}
    lab, n = ndi_label(mask)
    if n == 0:
        return {"component_count": 0, "largest_component_fraction": 0.0}
    sizes = np.bincount(lab.ravel())[1:]
    return {"component_count": int(n), "largest_component_fraction": float(np.max(sizes) / (np.sum(sizes) + 1e-16))}


def _run_lengths_1d_bool(arr):
    arr = np.asarray(arr, dtype=bool)
    if arr.size == 0:
        return np.array([], dtype=int)
    # lengths of True-runs only
    x = arr.astype(np.int8)
    padded = np.r_[0, x, 0]
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return (ends - starts).astype(int)


def _sample_run_lengths(mask, axis=0, max_lines=4000, rng=None):
    rng = np.random.default_rng(1234) if rng is None else rng
    M = np.asarray(mask, dtype=bool)
    M = np.moveaxis(M, axis, 0)
    n0, n1, n2 = M.shape
    pairs = np.array(np.meshgrid(np.arange(n1), np.arange(n2), indexing="ij")).reshape(2, -1).T
    if len(pairs) > max_lines:
        pairs = pairs[rng.choice(len(pairs), size=int(max_lines), replace=False)]
    vals = []
    for i, j in pairs:
        rl = _run_lengths_1d_bool(M[:, i, j])
        if len(rl):
            vals.extend(rl.tolist())
    return np.asarray(vals, dtype=float)


def extract_voxel_topology_descriptors(V, F, row=None, rng=None):
    if not DESCRIPTOR_CONFIG.get("enable_voxel_topology_descriptors", True):
        return {"voxel_topology_status": "disabled"}
    rng = np.random.default_rng(1234) if rng is None else rng
    out = {"voxel_topology_status": "ok"}
    try:
        grid_n = int(DESCRIPTOR_CONFIG.get("voxel_topology_grid_n", DESCRIPTOR_CONFIG.get("voxel_grid_n", 96)))
        mask = _voxelize_mesh_to_mask(V, F, grid_n=grid_n)
        mask = np.asarray(mask, dtype=bool)
        out["voxel_grid_n_used"] = int(grid_n)
        out["voxel_solid_fraction"] = float(np.mean(mask)) if mask.size else np.nan
        out["voxel_solid_count"] = int(np.count_nonzero(mask))
        out["voxel_void_fraction"] = float(1.0 - np.mean(mask)) if mask.size else np.nan
        out.update({f"voxel_solid_{k}": v for k, v in _connected_component_stats_3d(mask).items()})
        out.update({f"voxel_void_{k}": v for k, v in _connected_component_stats_3d(~mask).items()})

        # Cross-sectional area fraction along principal axes.
        for ax, nm in enumerate(["x", "y", "z"]):
            occ = np.mean(mask, axis=tuple(i for i in range(3) if i != ax))
            out.update(_stats(occ, f"voxel_section_area_fraction_{nm}"))
            out[f"voxel_section_nonempty_fraction_{nm}"] = float(np.mean(occ > 0)) if len(occ) else np.nan
            rl = _sample_run_lengths(mask, axis=ax, max_lines=4000, rng=rng)
            out.update(_stats(rl / max(1, grid_n), f"voxel_solid_chord_fraction_{nm}"))

        # Local thickness proxy = 2 * distance to nearest void, normalized by grid size.
        if distance_transform_edt is not None and np.any(mask):
            dt = distance_transform_edt(mask)
            core = dt[mask]
            out.update(_stats((2.0 * core) / max(1, grid_n), "voxel_local_thickness_fraction"))
            out["voxel_core_fraction_dt_ge_2"] = float(np.mean(core >= 2.0)) if len(core) else np.nan
            out["voxel_core_fraction_dt_ge_3"] = float(np.mean(core >= 3.0)) if len(core) else np.nan
        else:
            out.update(_stats([], "voxel_local_thickness_fraction"))

        # Interface complexity in voxel space: exposed faces per solid voxel.
        exposed = 0
        for ax in range(3):
            diff = np.diff(mask.astype(np.int8), axis=ax)
            exposed += int(np.count_nonzero(diff != 0))
            # outer boundaries
            sl0 = [slice(None)] * 3; sl0[ax] = 0
            sl1 = [slice(None)] * 3; sl1[ax] = -1
            exposed += int(np.count_nonzero(mask[tuple(sl0)]))
            exposed += int(np.count_nonzero(mask[tuple(sl1)]))
        out["voxel_exposed_face_count"] = int(exposed)
        out["voxel_exposed_faces_per_solid_voxel"] = float(exposed / (np.count_nonzero(mask) + 1e-16))
    except Exception as e:
        out["voxel_topology_status"] = "failed"
        out["voxel_topology_error"] = repr(e)
    return out



def _cov_anisotropy_from_points(P):
    P = np.asarray(P, dtype=float)
    if P.ndim != 2 or P.shape[0] < 3:
        return {
            "eig1": np.nan, "eig2": np.nan, "eig3": np.nan,
            "anisotropy_ratio": np.nan,
            "orientation_axis_x": np.nan, "orientation_axis_y": np.nan, "orientation_axis_z": np.nan,
        }
    P0 = P - np.mean(P, axis=0, keepdims=True)
    C = np.cov(P0.T)
    vals, vecs = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    major = vecs[:, 0]
    return {
        "eig1": float(vals[0]), "eig2": float(vals[1]), "eig3": float(vals[2]),
        "anisotropy_ratio": float(vals[0] / (vals[2] + 1e-16)),
        "orientation_axis_x": float(abs(major[0])),
        "orientation_axis_y": float(abs(major[1])),
        "orientation_axis_z": float(abs(major[2])),
    }


def _block_density_descriptor(mask, block_sizes):
    out = {}
    M = np.asarray(mask, dtype=bool)
    if M.ndim != 3 or M.size == 0:
        return out
    for bs in block_sizes or []:
        try:
            bs = int(bs)
            if bs <= 0:
                continue
            nx0 = M.shape[0] // bs
            ny0 = M.shape[1] // bs
            nz0 = M.shape[2] // bs
            if min(nx0, ny0, nz0) < 1:
                continue
            core = M[:nx0*bs, :ny0*bs, :nz0*bs]
            blocks = core.reshape(nx0, bs, ny0, bs, nz0, bs).mean(axis=(1,3,5)).ravel()
            if blocks.size == 0:
                continue
            out[f"local_density_bs{bs}_avg"] = float(np.mean(blocks))
            out[f"local_density_bs{bs}_stdev"] = float(np.std(blocks))
            out[f"local_density_bs{bs}_heterogeneity_cv"] = float(np.std(blocks) / (np.mean(blocks) + 1e-16))
            out[f"local_density_bs{bs}_p90_p10_gap"] = float(np.percentile(blocks, 90) - np.percentile(blocks, 10))
        except Exception:
            continue
    return out


def _skeletonize_mask_3d(mask):
    M = np.asarray(mask, dtype=bool)
    if not np.any(M) or not SKIMAGE_SKELETON_AVAILABLE:
        return np.zeros_like(M, dtype=bool)
    try:
        if sk_skeletonize_3d is not None:
            sk = sk_skeletonize_3d(M)
        else:
            sk = np.zeros_like(M, dtype=bool)
            for k in range(M.shape[2]):
                sk[:, :, k] = sk_skeletonize(M[:, :, k])
        return np.asarray(sk, dtype=bool)
    except Exception:
        return np.zeros_like(M, dtype=bool)


def _graph_from_voxel_mask(mask):
    M = np.asarray(mask, dtype=bool)
    if not np.any(M) or not NETWORKX_AVAILABLE:
        return None, np.empty((0, 3), dtype=int)
    coords = np.argwhere(M)
    if len(coords) == 0:
        return None, coords
    idx = {tuple(c): i for i, c in enumerate(coords)}
    G = nx.Graph()
    G.add_nodes_from(range(len(coords)))
    nbrs = [(dx,dy,dz) for dx in (-1,0,1) for dy in (-1,0,1) for dz in (-1,0,1) if not (dx==dy==dz==0)]
    for i, c in enumerate(coords):
        x,y,z = c
        for dx,dy,dz in nbrs:
            nb = (x+dx, y+dy, z+dz)
            j = idx.get(nb, None)
            if j is not None and j > i:
                w = float((dx*dx + dy*dy + dz*dz) ** 0.5)
                G.add_edge(i, j, weight=w)
    return G, coords


def _boundary_node_ids(coords, shape, axis, margin=1):
    if len(coords) == 0:
        return np.array([], dtype=int), np.array([], dtype=int)
    c = np.asarray(coords, dtype=int)
    lo = np.where(c[:, axis] <= int(margin))[0]
    hi = np.where(c[:, axis] >= int(shape[axis] - 1 - margin))[0]
    return lo, hi


def _spanning_path_metrics_graph(G, coords, shape, axis, margin=1):
    out = {}
    axname = ["x","y","z"][axis]
    if G is None or len(coords) == 0:
        out[f"{axname}_path_continuity"] = np.nan
        out[f"{axname}_tortuosity"] = np.nan
        out[f"{axname}_conduction_path_continuity"] = np.nan
        out[f"{axname}_shortest_path_length_vox"] = np.nan
        return out
    src, dst = _boundary_node_ids(coords, shape, axis, margin=margin)
    if len(src) == 0 or len(dst) == 0:
        out[f"{axname}_path_continuity"] = 0.0
        out[f"{axname}_tortuosity"] = np.nan
        out[f"{axname}_conduction_path_continuity"] = 0.0
        out[f"{axname}_shortest_path_length_vox"] = np.nan
        return out

    src_set = set(src.tolist())
    dst_set = set(dst.tolist())
    spanning_nodes = set()
    for comp in nx.connected_components(G):
        cs = set(comp)
        if cs & src_set and cs & dst_set:
            spanning_nodes |= cs

    if not spanning_nodes:
        out[f"{axname}_path_continuity"] = 0.0
        out[f"{axname}_tortuosity"] = np.nan
        out[f"{axname}_conduction_path_continuity"] = 0.0
        out[f"{axname}_shortest_path_length_vox"] = np.nan
        return out

    SG = G.subgraph(spanning_nodes)
    best = np.inf
    for s in src:
        if s not in SG:
            continue
        dist = nx.single_source_dijkstra_path_length(SG, int(s), weight="weight")
        for t in dst:
            d = dist.get(int(t), np.inf)
            if d < best:
                best = d
    direct = float(max(1, shape[axis] - 1 - 2*margin))
    pc = float(direct / (best + 1e-16)) if np.isfinite(best) else 0.0
    tau = float(best / (direct + 1e-16)) if np.isfinite(best) else np.nan
    src_frac = float(np.mean([n in spanning_nodes for n in src])) if len(src) else 0.0
    dst_frac = float(np.mean([n in spanning_nodes for n in dst])) if len(dst) else 0.0
    cpc = float(0.5 * (src_frac + dst_frac) * pc)
    out[f"{axname}_path_continuity"] = pc
    out[f"{axname}_tortuosity"] = tau
    out[f"{axname}_conduction_path_continuity"] = cpc
    out[f"{axname}_shortest_path_length_vox"] = float(best) if np.isfinite(best) else np.nan
    return out


def _graph_modal_proxies(G, max_modes=24, n_bands=8):
    out = {}
    if G is None or G.number_of_nodes() < 3:
        out["graph_modal_density_proxy"] = np.nan
        for b in range(1, int(n_bands) + 1):
            out[f"graph_modal_band_{b}_count_proxy"] = np.nan
        return out
    try:
        import scipy.sparse as sp
        import scipy.sparse.linalg as spla
        n = G.number_of_nodes()
        rows = []
        cols = []
        vals = []
        deg = np.zeros(n, dtype=float)
        for u, v, d in G.edges(data=True):
            wlen = float(d.get("weight", 1.0))
            k = 1.0 / max(wlen, 1e-12)
            rows += [u, v]
            cols += [v, u]
            vals += [-k, -k]
            deg[u] += k
            deg[v] += k
        rows += list(range(n))
        cols += list(range(n))
        vals += deg.tolist()
        L = sp.csr_matrix((vals, (rows, cols)), shape=(n, n))
        k = int(min(max_modes, max(1, n - 1)))
        evals = spla.eigsh(L, k=k, which="SM", return_eigenvectors=False)
        evals = np.sort(np.real(evals))
        pos = evals[evals > 1e-9]
        if len(pos) == 0:
            out["graph_modal_density_proxy"] = 0.0
            for b in range(1, int(n_bands) + 1):
                out[f"graph_modal_band_{b}_count_proxy"] = 0
            return out
        span = float(pos.max() - pos.min() + 1e-16)
        out["graph_modal_density_proxy"] = float(len(pos) / span)
        bins = np.linspace(float(pos.min()), float(pos.max()) + 1e-12, int(n_bands) + 1)
        hist, _ = np.histogram(pos, bins=bins)
        for b, count in enumerate(hist, start=1):
            out[f"graph_modal_band_{b}_count_proxy"] = int(count)
    except Exception:
        out["graph_modal_density_proxy"] = np.nan
        for b in range(1, int(n_bands) + 1):
            out[f"graph_modal_band_{b}_count_proxy"] = np.nan
    return out


def _graph_general_descriptors(G, coords=None, shape=None, prefix="graph"):
    out = {}
    if G is None or G.number_of_nodes() == 0:
        out[f"{prefix}_node_count"] = 0
        out[f"{prefix}_edge_count"] = 0
        return out
    deg = np.array([d for _, d in G.degree()], dtype=float)
    out[f"{prefix}_node_count"] = int(G.number_of_nodes())
    out[f"{prefix}_edge_count"] = int(G.number_of_edges())
    out[f"{prefix}_average_degree"] = float(np.mean(deg))
    out[f"{prefix}_degree_stdev"] = float(np.std(deg))
    out[f"{prefix}_junction_node_count"] = int(np.sum(deg >= 3))
    out[f"{prefix}_terminal_node_count"] = int(np.sum(deg == 1))
    out[f"{prefix}_junction_fraction"] = float(np.mean(deg >= 3))
    if coords is not None and len(coords) == G.number_of_nodes():
        coords = np.asarray(coords, dtype=float)
        if shape is None:
            shape = np.max(coords, axis=0) - np.min(coords, axis=0) + 1.0
        bbox_vol = float(np.prod(shape))
        out[f"{prefix}_junction_density_per_bbox"] = float(np.sum(deg >= 3) / (bbox_vol + 1e-16))
        an = _cov_anisotropy_from_points(coords)
        out[f"{prefix}_orientation_anisotropy"] = an["anisotropy_ratio"]
        out[f"{prefix}_orientation_axis_x"] = an["orientation_axis_x"]
        out[f"{prefix}_orientation_axis_y"] = an["orientation_axis_y"]
        out[f"{prefix}_orientation_axis_z"] = an["orientation_axis_z"]
    edge_lengths = np.array([float(d.get("weight", 1.0)) for _,_,d in G.edges(data=True)], dtype=float)
    if len(edge_lengths):
        out[f"{prefix}_characteristic_length"] = float(np.mean(edge_lengths))
        out[f"{prefix}_characteristic_length_weighted"] = float(np.sum(edge_lengths**2) / (np.sum(edge_lengths) + 1e-16))
    else:
        out[f"{prefix}_characteristic_length"] = np.nan
        out[f"{prefix}_characteristic_length_weighted"] = np.nan
    try:
        out[f"{prefix}_component_count"] = int(nx.number_connected_components(G))
        cc = [len(c) for c in nx.connected_components(G)]
        out[f"{prefix}_largest_component_fraction"] = float(max(cc) / (sum(cc) + 1e-16))
    except Exception:
        pass
    try:
        if G.number_of_nodes() <= int(DESCRIPTOR_CONFIG.get("graph_algebraic_connectivity_max_nodes", 400)):
            out[f"{prefix}_algebraic_connectivity"] = float(nx.algebraic_connectivity(G, weight="weight"))
        else:
            out[f"{prefix}_algebraic_connectivity"] = np.nan
    except Exception:
        out[f"{prefix}_algebraic_connectivity"] = np.nan
    try:
        out[f"{prefix}_average_clustering"] = float(nx.average_clustering(G))
    except Exception:
        out[f"{prefix}_average_clustering"] = np.nan
    try:
        if G.number_of_nodes() <= 600:
            out[f"{prefix}_global_efficiency"] = float(nx.global_efficiency(G))
        else:
            out[f"{prefix}_global_efficiency"] = np.nan
    except Exception:
        out[f"{prefix}_global_efficiency"] = np.nan
    try:
        if G.number_of_nodes() <= 400:
            bc = nx.betweenness_centrality(G, weight="weight", normalized=True)
            vals = np.array(list(bc.values()), dtype=float)
            out[f"{prefix}_weighted_betweenness_avg"] = float(np.mean(vals))
            out[f"{prefix}_weighted_betweenness_max"] = float(np.max(vals))
        else:
            out[f"{prefix}_weighted_betweenness_avg"] = np.nan
            out[f"{prefix}_weighted_betweenness_max"] = np.nan
    except Exception:
        out[f"{prefix}_weighted_betweenness_avg"] = np.nan
        out[f"{prefix}_weighted_betweenness_max"] = np.nan
    if DESCRIPTOR_CONFIG.get("enable_modal_proxy_descriptors", True):
        out.update(_graph_modal_proxies(
            G,
            max_modes=int(DESCRIPTOR_CONFIG.get("modal_proxy_max_modes", 24)),
            n_bands=int(DESCRIPTOR_CONFIG.get("modal_proxy_band_count", 8)),
        ))
    return out


def extract_canonical_surface_descriptors(V, F):
    out = {}
    try:
        bbox_min = np.min(V, axis=0)
        bbox_max = np.max(V, axis=0)
        ext = np.maximum(bbox_max - bbox_min, 1e-12)
        bbox_vol = float(np.prod(ext))
        area = _surface_area(V, F)
        vol = abs(_mesh_volume_signed(V, F))
        out["relative_density"] = float(vol / (bbox_vol + 1e-16))
        out["surface_area_to_volume_ratio"] = float(area / (vol + 1e-16))
        E = _mesh_edges(F)
        if len(E):
            edge_lengths = np.linalg.norm(V[E[:, 0]] - V[E[:, 1]], axis=1)
            out["characteristic_length"] = float(np.mean(edge_lengths))
        an = _cov_anisotropy_from_points(V)
        out["orientation_anisotropy"] = an["anisotropy_ratio"]
        out["orientation_axis_x"] = an["orientation_axis_x"]
        out["orientation_axis_y"] = an["orientation_axis_y"]
        out["orientation_axis_z"] = an["orientation_axis_z"]
    except Exception as e:
        out["canonical_surface_error"] = repr(e)
    return out


def extract_voxel_graph_descriptors(V, F, row=None):
    out = {"advanced_graph_status": "ok"}
    try:
        grid_n = int(DESCRIPTOR_CONFIG.get("advanced_graph_grid_n", DESCRIPTOR_CONFIG.get("voxel_topology_grid_n", 96)))
        mask = _voxelize_mesh_to_mask(V, F, grid_n=grid_n)
        mask = np.asarray(mask, dtype=bool)
        out["advanced_graph_grid_n_used"] = int(grid_n)
        if np.any(mask):
            solid_coords = np.argwhere(mask)
            an = _cov_anisotropy_from_points(solid_coords)
            out["voxel_orientation_anisotropy"] = an["anisotropy_ratio"]
            out["voxel_orientation_axis_x"] = an["orientation_axis_x"]
            out["voxel_orientation_axis_y"] = an["orientation_axis_y"]
            out["voxel_orientation_axis_z"] = an["orientation_axis_z"]
            out.update(_block_density_descriptor(mask, DESCRIPTOR_CONFIG.get("local_density_block_sizes", [4, 8])))
        sk = _skeletonize_mask_3d(mask)
        G, coords = _graph_from_voxel_mask(sk)
        out.update(_graph_general_descriptors(G, coords=coords, shape=mask.shape, prefix="graph"))
        margin = int(DESCRIPTOR_CONFIG.get("graph_boundary_margin_vox", 1))
        path_axes = DESCRIPTOR_CONFIG.get("graph_path_axes", ["x", "y", "z"])
        if isinstance(path_axes, str):
            path_axes = [path_axes]
        axis_map = {"x": 0, "y": 1, "z": 2}
        for nm in path_axes:
            ax = axis_map.get(str(nm).lower())
            if ax is None:
                continue
            pm = _spanning_path_metrics_graph(G, coords, mask.shape, axis=ax, margin=margin)
            for k, v in pm.items():
                out[f"graph_{k}"] = v
        out["junction_density"] = out.get("graph_junction_density_per_bbox", np.nan)
        out["graph_connectivity"] = out.get("graph_algebraic_connectivity", np.nan)
        out["graph_shortest_path_weighted_betweenness_avg"] = out.get("graph_weighted_betweenness_avg", np.nan)
        out["graph_shortest_path_weighted_betweenness_max"] = out.get("graph_weighted_betweenness_max", np.nan)
        out["modal_density_proxy"] = out.get("graph_modal_density_proxy", np.nan)
    except Exception as e:
        out["advanced_graph_status"] = "failed"
        out["advanced_graph_error"] = repr(e)
    return out


def extract_lattice_graph_advanced_descriptors(row):
    out = {"lattice_graph_status": "not_lattice"}
    p = _get_parameter_dict(row or {})
    if str(p.get("generator_type", _generator_type_from_row(row or {}))) != "lattice":
        return out
    nodes, edges, status = _extract_lattice_graph_from_params(p)
    out["lattice_graph_status"] = status
    if status != "ok" or nodes is None or edges is None or not NETWORKX_AVAILABLE:
        return out
    try:
        nodes = np.asarray(nodes, dtype=float)
        G = nx.Graph()
        G.add_nodes_from(range(len(nodes)))
        for i, j in edges:
            L = float(np.linalg.norm(nodes[int(i)] - nodes[int(j)]))
            if L > 1e-12:
                G.add_edge(int(i), int(j), weight=L)
        bbox_min = np.min(nodes, axis=0)
        bbox_max = np.max(nodes, axis=0)
        shape = np.maximum(bbox_max - bbox_min, 1e-12)
        out.update(_graph_general_descriptors(G, coords=nodes, shape=shape, prefix="lattice_graph"))
        margin_frac = 0.05
        for ax, nm in enumerate(["x", "y", "z"]):
            vals = nodes[:, ax]
            lo = np.where(vals <= (vals.min() + margin_frac * (vals.max() - vals.min() + 1e-16)))[0]
            hi = np.where(vals >= (vals.max() - margin_frac * (vals.max() - vals.min() + 1e-16)))[0]
            best = np.inf
            lo_set = set(lo.tolist())
            hi_set = set(hi.tolist())
            spanning_nodes = set()
            for comp in nx.connected_components(G):
                cs = set(comp)
                if cs & lo_set and cs & hi_set:
                    spanning_nodes |= cs
            if not spanning_nodes:
                out[f"lattice_graph_{nm}_path_continuity"] = 0.0
                out[f"lattice_graph_{nm}_tortuosity"] = np.nan
                out[f"lattice_graph_{nm}_conduction_path_continuity"] = 0.0
                continue
            SG = G.subgraph(spanning_nodes)
            for s in lo:
                if s not in SG:
                    continue
                dist = nx.single_source_dijkstra_path_length(SG, int(s), weight="weight")
                for t in hi:
                    d = dist.get(int(t), np.inf)
                    if d < best:
                        best = d
            direct = float(vals.max() - vals.min())
            out[f"lattice_graph_{nm}_path_continuity"] = float(direct / (best + 1e-16)) if np.isfinite(best) else 0.0
            out[f"lattice_graph_{nm}_tortuosity"] = float(best / (direct + 1e-16)) if np.isfinite(best) else np.nan
            lo_frac = float(np.mean([n in spanning_nodes for n in lo])) if len(lo) else 0.0
            hi_frac = float(np.mean([n in spanning_nodes for n in hi])) if len(hi) else 0.0
            out[f"lattice_graph_{nm}_conduction_path_continuity"] = float(0.5 * (lo_frac + hi_frac) * out[f"lattice_graph_{nm}_path_continuity"])
        out["junction_density"] = out.get("lattice_graph_junction_density_per_bbox", np.nan)
        out["graph_connectivity"] = out.get("lattice_graph_algebraic_connectivity", np.nan)
        out["weighted_betweenness_avg"] = out.get("lattice_graph_weighted_betweenness_avg", np.nan)
        out["weighted_betweenness_max"] = out.get("lattice_graph_weighted_betweenness_max", np.nan)
        axis_weights = DESCRIPTOR_CONFIG.get("stiffness_proxy_axis_weights", {"x":1.0,"y":1.0,"z":1.0})
        stiff = {}
        for ax, nm in enumerate(["x","y","z"]):
            contrib = []
            for u,v,d in G.edges(data=True):
                vec = nodes[v] - nodes[u]
                L = float(np.linalg.norm(vec))
                if L <= 1e-12:
                    continue
                align = abs(vec[ax]) / L
                contrib.append((align ** 2) / L)
            stiff[nm] = float(np.sum(contrib)) if len(contrib) else np.nan
            out[f"lattice_graph_stiffness_proxy_{nm}"] = stiff[nm]
        out["stiffness_proxy"] = float(
            axis_weights.get("x",1.0)*(stiff.get("x",0.0) if np.isfinite(stiff.get("x",np.nan)) else 0.0) +
            axis_weights.get("y",1.0)*(stiff.get("y",0.0) if np.isfinite(stiff.get("y",np.nan)) else 0.0) +
            axis_weights.get("z",1.0)*(stiff.get("z",0.0) if np.isfinite(stiff.get("z",np.nan)) else 0.0)
        )
        out["modal_density_proxy"] = out.get("lattice_graph_modal_density_proxy", np.nan)
    except Exception as e:
        out["lattice_graph_status"] = "failed"
        out["lattice_graph_error"] = repr(e)
    return out


def extract_extra_surface_topology_descriptors(V, F):
    out = {}
    try:
        E, boundary_e, nonmanifold_e = _count_boundary_and_nonmanifold_edges(F)
        out["mesh_unique_edge_count"] = int(E)
        out["mesh_boundary_edge_count"] = int(boundary_e)
        out["mesh_nonmanifold_edge_count"] = int(nonmanifold_e)
        out["mesh_is_closed_by_edges"] = bool(boundary_e == 0)
        out["mesh_euler_characteristic"] = int(len(V) - E + len(F)) if E else np.nan
        if boundary_e == 0:
            out["mesh_genus_estimate_if_closed"] = float((2 - out["mesh_euler_characteristic"]) / 2.0)
        else:
            out["mesh_genus_estimate_if_closed"] = np.nan

        tri, area, normals, centers = _face_geometry(V, F)
        total_area = np.sum(area) + 1e-16
        z_axis = np.array([0.0, 0.0, 1.0])
        cosz = np.abs(normals @ z_axis)
        build_angle = np.degrees(np.arccos(np.clip(cosz, -1.0, 1.0)))
        thr = float(DESCRIPTOR_CONFIG.get("overhang_angle_threshold_deg", 45.0))
        out["surface_area_fraction_overhang_like"] = float(np.sum(area[build_angle > thr]) / total_area) if len(area) else np.nan
        out["surface_area_fraction_horizontal_like"] = float(np.sum(area[cosz > np.cos(np.radians(15))]) / total_area) if len(area) else np.nan
        out["surface_area_fraction_vertical_like"] = float(np.sum(area[cosz < np.sin(np.radians(15))]) / total_area) if len(area) else np.nan
    except Exception as e:
        out["extra_surface_topology_error"] = repr(e)
    return out


def extract_descriptors_for_candidate(row):
    row = dict(row)
    cid = _candidate_id_from_row(row)
    gtype = _generator_type_from_row(row)
    out = {
        "candidate_id": cid,
        "generator_type": gtype,
        "descriptor_schema_version": DESCRIPTOR_SCHEMA_VERSION,
        "descriptor_status": "ok",
        "descriptor_error": "",
    }
    out_dir = _descriptor_output_dir()

    try:
        stl_path = _resolve_candidate_stl_path(row)
        out["descriptor_stl_path"] = str(stl_path)
        V, F = load_stl_mesh_arrays(stl_path)
        if len(V) == 0 or len(F) == 0:
            raise RuntimeError("Empty STL mesh")
        out["mesh_vertex_count"] = int(len(V))
        out["mesh_face_count"] = int(len(F))
        out["mesh_load_status"] = "ok"

        # 1. Point-based descriptors
        out.update(extract_point_descriptors(V, F, row=row, out_dir=out_dir))

        # 2. Surface descriptors
        out.update(extract_surface_descriptors(V, F, row=row, out_dir=out_dir))

        # 3. Slice-image descriptors
        out.update(extract_slice_image_descriptors(V, F, row=row, out_dir=out_dir))

        # 4. Lattice descriptors
        out.update(extract_lattice_descriptors(row))

        # 5. Extra geometry, topology, and voxel descriptors
        if DESCRIPTOR_CONFIG.get("enable_additional_descriptors", True):
            bbox_min = np.min(V, axis=0)
            bbox_max = np.max(V, axis=0)
            ext = np.maximum(bbox_max - bbox_min, 1e-12)
            Pn = _normalize_points_by_bbox(V, bbox_min, bbox_max)
            if len(Pn) >= 3:
                C = np.cov(Pn.T)
                eig = np.sort(np.linalg.eigvalsh(C))[::-1]
                out["geom_vertex_cov_eig1"] = float(eig[0])
                out["geom_vertex_cov_eig2"] = float(eig[1])
                out["geom_vertex_cov_eig3"] = float(eig[2])
                out["geom_vertex_anisotropy"] = float(eig[0] / (eig[2] + 1e-16))
            out["geom_bbox_aspect_xy"] = float(ext[0] / ext[1])
            out["geom_bbox_aspect_xz"] = float(ext[0] / ext[2])
            out["geom_bbox_aspect_yz"] = float(ext[1] / ext[2])
            out["geom_bbox_volume"] = float(np.prod(ext))
            out["geom_vertices_per_bbox_volume"] = float(len(V) / (np.prod(ext) + 1e-16))
            out.update(extract_extra_surface_topology_descriptors(V, F))
            out.update(extract_voxel_topology_descriptors(V, F, row=row))
            out.update(extract_canonical_surface_descriptors(V, F))
            out.update(extract_voxel_graph_descriptors(V, F, row=row))
            out.update(extract_lattice_graph_advanced_descriptors(row))

            # Canonical aliases / harmonized names requested for downstream ML
            if "voxel_local_thickness_fraction_avg" in out:
                out["mean_thickness"] = out.get("voxel_local_thickness_fraction_avg", np.nan)
            elif "surface_edge_length_avg" in out:
                out["mean_thickness"] = out.get("surface_edge_length_avg", np.nan)

            if "relative_density" not in out and "relative_density_from_mesh_volume" in out:
                out["relative_density"] = out.get("relative_density_from_mesh_volume", np.nan)
            if "characteristic_length" not in out:
                out["characteristic_length"] = (
                    out.get("graph_characteristic_length", np.nan)
                    if np.isfinite(out.get("graph_characteristic_length", np.nan))
                    else out.get("lattice_graph_characteristic_length", out.get("surface_edge_length_avg", np.nan))
                )
            if "orientation_anisotropy" not in out:
                out["orientation_anisotropy"] = (
                    out.get("voxel_orientation_anisotropy", np.nan)
                    if np.isfinite(out.get("voxel_orientation_anisotropy", np.nan))
                    else out.get("geom_vertex_anisotropy", np.nan)
                )
            if "connectivity" not in out:
                out["connectivity"] = (
                    out.get("graph_connectivity", np.nan)
                    if np.isfinite(out.get("graph_connectivity", np.nan))
                    else out.get("voxel_solid_largest_component_fraction", np.nan)
                )
            if "local_density_heterogeneity" not in out:
                out["local_density_heterogeneity"] = (
                    out.get("local_density_bs4_heterogeneity_cv", np.nan)
                    if np.isfinite(out.get("local_density_bs4_heterogeneity_cv", np.nan))
                    else out.get("local_density_bs8_heterogeneity_cv", np.nan)
                )

            # Generic axis aliases from voxel graph first, lattice graph second
            for ax in ["x", "y", "z"]:
                if f"{ax}_path_continuity" not in out:
                    out[f"{ax}_path_continuity"] = out.get(f"graph_{ax}_path_continuity", out.get(f"lattice_graph_{ax}_path_continuity", np.nan))
                if f"{ax}_tortuosity" not in out:
                    out[f"{ax}_tortuosity"] = out.get(f"graph_{ax}_tortuosity", out.get(f"lattice_graph_{ax}_tortuosity", np.nan))
                if f"{ax}_conduction_path_continuity" not in out:
                    out[f"{ax}_conduction_path_continuity"] = out.get(f"graph_{ax}_conduction_path_continuity", out.get(f"lattice_graph_{ax}_conduction_path_continuity", np.nan))

            # Generic stiffness proxy if only voxel graph exists
            if "stiffness_proxy" not in out or not np.isfinite(out.get("stiffness_proxy", np.nan)):
                weights = DESCRIPTOR_CONFIG.get("stiffness_proxy_axis_weights", {"x":1.0,"y":1.0,"z":1.0})
                vals = [out.get(f"{a}_conduction_path_continuity", np.nan) for a in ["x","y","z"]]
                vals = [0.0 if not np.isfinite(v) else float(v) for v in vals]
                out["stiffness_proxy"] = (
                    float(weights.get("x",1.0))*vals[0] +
                    float(weights.get("y",1.0))*vals[1] +
                    float(weights.get("z",1.0))*vals[2]
                )
    except Exception as e:
        out["descriptor_status"] = "failed"
        out["descriptor_error"] = repr(e)
    return out


def diagnose_generation_and_descriptor_inputs(first_n=5):
    df = _get_descriptor_input_table()
    print("=" * 80)
    print("Descriptor input diagnosis")
    print(f"rows={len(df)}, columns={len(df.columns)}")
    print(f"columns sample={list(df.columns[:12])}")
    print(f"OUTPUT_ROOT={Path(globals().get('OUTPUT_ROOT', Path.cwd())).resolve()}")
    print(f"descriptor_output={_descriptor_output_dir().resolve()}")
    if "status" in df.columns:
        print(df["status"].value_counts(dropna=False).head(10))
    if "generator_type" in df.columns:
        print(df["generator_type"].value_counts(dropna=False).head(10))
    sample = df.head(first_n).to_dict("records")
    for r in sample:
        p = _resolve_candidate_stl_path(r)
        print(f"{_candidate_id_from_row(r)} | {_generator_type_from_row(r)} | STL exists={p.exists()} | {p}")
    print("=" * 80)
    return df


def run_descriptor_extraction(n_workers=None, first_n=None, only_successful=True):
    df = _get_descriptor_input_table()
    if only_successful and "status" in df.columns:
        df = df[df["status"].astype(str).isin(["ok", "skipped_existing"])]
    if first_n is not None:
        df = df.head(int(first_n))
    rows = df.to_dict("records")
    n_workers = int(n_workers or DESCRIPTOR_CONFIG.get("cpu_workers", 1))
    n_workers = max(1, n_workers)
    out_dir = _descriptor_output_dir()
    print(f"Running structural descriptors: n={len(rows)}, workers={n_workers}, output={out_dir}")

    results = []
    if n_workers == 1 or DESCRIPTOR_CONFIG.get("parallel_backend", "thread") == "serial":
        for k, row in enumerate(rows, 1):
            res = extract_descriptors_for_candidate(row)
            results.append(res)
            if k == 1 or k % 10 == 0 or k == len(rows):
                print(f"[{k}/{len(rows)}] {res.get('candidate_id')} -> {res.get('descriptor_status')}")
    else:
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futs = {ex.submit(extract_descriptors_for_candidate, row): row for row in rows}
            for k, fut in enumerate(as_completed(futs), 1):
                res = fut.result()
                results.append(res)
                if k == 1 or k % 10 == 0 or k == len(rows):
                    print(f"[{k}/{len(rows)}] {res.get('candidate_id')} -> {res.get('descriptor_status')}")

    desc_df = pd.DataFrame(results)
    # Keep status/error columns near front.
    front = [c for c in ["candidate_id", "generator_type", "descriptor_status", "descriptor_error", "descriptor_schema_version", "descriptor_stl_path"] if c in desc_df.columns]
    desc_df = desc_df[front + [c for c in desc_df.columns if c not in front]]

    if DESCRIPTOR_CONFIG.get("write_csv", True):
        desc_df.to_csv(out_dir / "structural_descriptors_full.csv", index=False, encoding="utf-8-sig")
    if DESCRIPTOR_CONFIG.get("write_excel", True):
        try:
            desc_df.to_excel(out_dir / "structural_descriptors_full.xlsx", index=False)
        except Exception as e:
            print(f"Excel export skipped: {e}")
    if DESCRIPTOR_CONFIG.get("write_merged_table", True):
        merged = df.merge(desc_df, on=["candidate_id", "generator_type"], how="left", suffixes=("", "_descriptor"))
        merged.to_csv(out_dir / "generated_table_with_structural_descriptors.csv", index=False, encoding="utf-8-sig")
        try:
            merged.to_excel(out_dir / "generated_table_with_structural_descriptors.xlsx", index=False)
        except Exception as e:
            print(f"Merged Excel export skipped: {e}")

    # Summary of failures and non-empty descriptor families
    status_cols = [c for c in desc_df.columns if c.endswith("_descriptor_status") or c == "descriptor_status"]
    print("\nDescriptor status summary")
    for c in status_cols:
        print(c)
        print(desc_df[c].value_counts(dropna=False).head(10))
    diagnose_descriptor_dataframe(desc_df, nan_threshold=0.98)
    return desc_df



def diagnose_descriptor_dataframe(desc_df, nan_threshold=0.98):
    """Inspect which descriptor families failed or are almost completely blank."""
    print("\nDetailed descriptor completeness diagnosis")
    if desc_df is None or len(desc_df) == 0:
        print("No descriptor rows to diagnose.")
        return pd.DataFrame()
    status_cols = [c for c in desc_df.columns if c.endswith("_status") or c == "descriptor_status"]
    for c in status_cols:
        print(f"\n[{c}]")
        print(desc_df[c].value_counts(dropna=False).head(20))
    numeric_cols = desc_df.select_dtypes(include=[np.number]).columns.tolist()
    rows = []
    for c in numeric_cols:
        nan_frac = float(desc_df[c].isna().mean())
        finite_count = int(np.isfinite(desc_df[c]).sum())
        if nan_frac >= nan_threshold or finite_count == 0:
            rows.append({"column": c, "nan_fraction": nan_frac, "finite_count": finite_count})
    blank = pd.DataFrame(rows).sort_values(["nan_fraction", "column"], ascending=[False, True]) if rows else pd.DataFrame(columns=["column", "nan_fraction", "finite_count"])
    if len(blank):
        print("\nColumns that are almost empty:")
        display(blank.head(80)) if "display" in globals() else print(blank.head(80).to_string(index=False))
    else:
        print("No almost-empty numeric descriptor columns above threshold.")
    return blank



## Structural descriptor extraction usage — split and fixed

Run Cells 8–8G after candidate generation. The descriptor extraction is intentionally divided by method: point-based, surface-based, slice-image-based, and lattice node/strut-based. The final runner exports both `structural_descriptors_full` and `generated_table_with_structural_descriptors` into `04_structural_descriptors_split`.


In [ ]:
# ============================================================
# Cell 8G. Safe-run structural descriptor extraction
# ============================================================
# Why SAFE START?
# In Windows/Jupyter, heavy STL generation + descriptor extraction can keep the UI at
# "Restarting kernel" when workers, Open3D/trimesh voxelization, or memory use crash the Python process.
# This cell is intentionally OFF by default. Run a small first_n test first.

DESCRIPTOR_RUN_NOW = False
DESCRIPTOR_FIRST_N = 10      # First validation run. After success, set None for all successful candidates.
DESCRIPTOR_N_WORKERS = 1     # First validation run. After success, try 2, 4, 8 sequentially.

# Requested default: 100 Nth/(N+1)th slice pairs are controlled here.
DESCRIPTOR_CONFIG["slice_count"] = 100
DESCRIPTOR_CONFIG["cpu_workers"] = DESCRIPTOR_N_WORKERS
DESCRIPTOR_CONFIG["parallel_backend"] = "serial"
DESCRIPTOR_CONFIG["descriptor_backend"] = "cpu"
DESCRIPTOR_CONFIG["interior_sampling_method"] = "auto"
DESCRIPTOR_CONFIG["interior_require_exact"] = False
DESCRIPTOR_CONFIG["enable_voxel_topology_descriptors"] = True

print("SAFE START MODE")
print("1) First run:  desc_df = run_descriptor_extraction(n_workers=1, first_n=10)")
print("2) If OK:     desc_df = run_descriptor_extraction(n_workers=2, first_n=50)")
print("3) Final:     desc_df = run_descriptor_extraction(n_workers=4, first_n=None)")
print("Generation is also OFF by default: AUTO_RUN_GENERATION=False, RUN_GENERATION_MODE='none'.")

if DESCRIPTOR_RUN_NOW:
    diagnose_generation_and_descriptor_inputs()
    desc_df = run_descriptor_extraction(n_workers=DESCRIPTOR_N_WORKERS, first_n=DESCRIPTOR_FIRST_N)
    display(desc_df.head())


In [ ]:

# ============================================================
# Cell 9. Optional STP conversion for selected candidates only
# ============================================================

def convert_selected_stl_to_stp(selected_table_df):
    logs = []
    for _, row in selected_table_df.iterrows():
        stl_path = Path(row["stl_path"])
        if not stl_path.exists():
            logs.append({
                "candidate_id": row["candidate_id"],
                "status": "failed",
                "message": "STL not found",
                "stp_path": "",
            })
            continue
        generator_type = row["generator_type"]
        stp_path = DIR_GEOM / generator_type / "STP_selected_only" / f"{row['candidate_id']}.stp"
        ok, msg = try_export_step_from_stl(stl_path, stp_path)
        logs.append({
            "candidate_id": row["candidate_id"],
            "status": "ok" if ok else "failed",
            "message": msg,
            "stp_path": str(stp_path) if ok else "",
        })
    log_df = pd.DataFrame(logs)
    log_df.to_csv(DIR_SELECTED / "selected_stp_conversion_log.csv", index=False, encoding="utf-8-sig")
    log_df.to_excel(DIR_SELECTED / "selected_stp_conversion_log.xlsx", index=False)
    return log_df

# Example after candidate selection:
# selected_df = merged_result_df.query("status == 'ok'").head(10)
# stp_log_df = convert_selected_stl_to_stp(selected_df)
# display(stp_log_df)


## How to scale to tens of thousands of structures

This version has an explicit auto-run switch in Cell 1.

```python
RUN_GENERATION_MODE = "preview"  # only PREVIEW_N candidates
RUN_GENERATION_MODE = "all"      # all rows in candidate_df
RUN_GENERATION_MODE = "none"     # table only
AUTO_RUN_GENERATION = True
```

For large generation, increase counts in `LATTICE_GENERATION_CONFIG`, `TPMS_GENERATION_CONFIG`, and `VOXEL_GENERATION_CONFIG`, keep `EXPORT_STP=False`, and use `RUN_GENERATION_MODE="all"`.

After generation, the notebook automatically checks physical file existence and writes:

- `03_generation_logs/generation_file_check_summary.csv`
- `03_generation_logs/generation_file_inventory.csv`
- `03_generation_logs/generation_result_log_with_file_check.csv`

Recommended workflow:

```text
large batch generation → STL only
candidate_id-based descriptor extraction → merge with candidate table
surrogate prediction → Pareto/OOD filtering
selected candidates only → STP/STEP conversion
```
